In [1]:
!pip install -qq reportlab stable-baselines3 ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.5 MB/s eta 0:00:00


In [2]:
"""
Scene Solver v3 — Coherent Reasoning System
============================================
Architecture (enforced execution order):

  Stage 0 : AutoEncoder (AE)           — PRIMARY anomaly signal (batched)
  Stage 1 : TimeSformer Binary         — Semantic understanding (anomaly/normal)
  Stage 2 : TimeSformer Multi-class    — Fine-grained anomaly classification
  Stage 3 : YOLO Weapon (EXPERIMENTAL) — Hard safety trigger (conditional)
  Stage 4 : Audio Feature Extraction   — Environmental signal
  Stage 5 : Tracking / Behavior        — Motion behavior extraction
  Stage 6 : Fusion Layer               — xAI-style reasoning core
  Stage 7 : RL3033 (EXPERIMENTAL)      — Temporal reasoning agent (FINAL stage)
  Stage 8 : LLaVA (EXPERIMENTAL)       — Context explanation (multi-frame)

PATCHES APPLIED IN THIS VERSION:
  PATCH-SAMPLING : Stage 2 uses segment-wise AE-weighted temporal sampling.
  PATCH-SAMPLING2: segmentwise_ae_sampling hardened with strict validation.
  PATCH-TRACE    : Fusion reasoning trace rewritten as plain English sentences.
  PATCH-DASH     : All em-dashes replaced with commas/parentheses throughout.

  PDF FIXES (this revision):
  PDF-BADGE      : Status badge now spans full body width so text is always
                   centred. Text changed to 'ANOMALY DETECTED (CRITICAL)' /
                   'NO ANOMALY DETECTED (SAFE)'.
  PDF-LABELS     : Executive Summary field labels changed from
                   'Stage N, Name' to 'Stage N (Name)' throughout.
  PDF-OVERFLOW   : Stage 3 (Weapon Detection) label no longer overflows into
                   the value column; key_w is now dynamic per field.
  PDF-EXPBOX     : Experimental disclaimer boxes on pages 5 and 11 now use
                   _wrap_text so long text stays inside the box.

  LOGIC FIXES (this revision):
  LOGIC-NORMAL   : When Stage 1 ( Binary) classifies video as Normal:
                   - Overall status is overridden to 'SAFE' (AE alone cannot
                     declare CRITICAL).
                   - Stage 2 (TF Multi) is skipped (returns a 'Skipped' dict).
                   - Stage 3 (YOLO) already skipped — unchanged.
                   - LLaVA (Stage 8) now always runs when enable_llava=True,
                     regardless of Stage 1 result (user requirement), using
                     neutral investigative prompts rather than 'suspicious' framing.
                   - Fusion and RL3033 receive the SAFE/LOW context correctly.

  PDF READABILITY FIXES (latest revision):
  PDF-PIE-LABELS : Stage 2 pie chart no longer renders overlapping % labels.
                   Users are directed to rely on the numbered legend beside the chart.
  PDF-AE-NORMAL  : AE anomaly detail section is suppressed when Stage 1 = Normal,
                   replaced with a concise safe-status note.
  PDF-EXPBOX2    : Experimental warning boxes: removed leading blank line,
                   fixed text rendering to stay fully inside the box.
  PDF-RL-EXPLAIN : RL3033 observation space entries now include plain-English
                   explanations of what each technical dimension means.
  PDF-AUDIO-EXPLAIN: Audio feature descriptions expanded with plain-English
                   explanations of what each acoustic feature measures.

  PDF IMPROVEMENTS (latest revision):
  PDF-EXPBOX3    : Experimental boxes now render as a single continuous paragraph
                   with no internal newline fragmentation. Text flows naturally.
  PDF-LLAVA-NARR : LLaVA section now synthesises all frame descriptions into one
                   coherent, human-readable scene narrative. Redundancy eliminated.
  PDF-DENSITY    : All sections enriched with interpretive context and
                   system-level insights beyond raw output restatement.
"""

# ── Standard library ──────────────────────────────────────────────────────────
import json
import os
import gc
import math
import time
import logging
import threading
import shutil
from datetime import datetime
from typing import List, Dict, Optional, Tuple, Any

# ── Third-party ───────────────────────────────────────────────────────────────
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.pdfgen import canvas as rl_canvas
from reportlab.platypus import Table, TableStyle

# ── Optional heavy imports ────────────────────────────────────────────────────
try:
    from transformers import (
        TimesformerForVideoClassification,
        LlavaForConditionalGeneration,
        AutoProcessor,
        VideoMAEImageProcessor,
    )
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("[WARNING] transformers not available — Stage 1/2/8 will be skipped.")

try:
    from stable_baselines3 import PPO
    RL_AVAILABLE = True
except ImportError:
    RL_AVAILABLE = False
    print("[WARNING] stable-baselines3 not available — Stage 7 (RL) will be skipped.")

try:
    import librosa
    LIBROSA_AVAILABLE = True
except ImportError:
    LIBROSA_AVAILABLE = False
    print("[WARNING] librosa not available — Stage 4 (Audio) will be limited.")

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except ImportError:
    YOLO_AVAILABLE = False
    print("[INFO] ultralytics not available — Stage 3 (YOLO) will be disabled.")

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("SceneSolver")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
class Config:
    DRIVE_ROOT = "/content/drive/MyDrive/solver1"
    PATHS = {
        "AE":        "/content/drive/MyDrive/solver1/AE/best_autoencoder.pth",
        "RL_AGENT":  "/content/drive/MyDrive/solver1/RL/best_model.zip",
        "TS_BINARY": "/content/drive/MyDrive/solver1/TFB/timesformer_best_binary.pth",
        "TS_MULTI":  "/content/drive/MyDrive/solver1/TFM/best_model.pth",
        "LLAVA":     "/content/drive/MyDrive/solver1/FireLLaVA_Final_Model",
        "YOLO":      "/content/drive/MyDrive/solver1/weapon_yolo.pt",
    }
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    SAFE_THRESHOLD_FLOOR    = 0.030
    SAFE_THRESHOLD_LEGACY   = 0.055
    AUDIO_ANOMALY_THRESHOLD = 0.1

    CLASS_NAMES = {
        0: "Aggression",     1: "TheftOrLarceny", 2: "Firebombing",
        3: "LawEnforcement", 4: "Shooting",        5: "Vandalism",
        6: "RoadAccidents",
    }

    WEAPON_CLASS_NAMES = {
        0: "Melee",
        1: "Firearms",
    }

    # RL3033 observation space dims (STRICT — match training)
    RL_AE_WINDOW    = 32
    RL_AE_CHANNELS  = 6
    RL_LATENT_MOT   = 1536
    RL_LATENT_STK   = 2560
    RL_SPATIAL_DIM  = 196
    RL_CONTEXT_DIM  = 5

    AE_BATCH_SIZE   = 16
    AE_FRAME_STRIDE = 5

    EMA_SHORT_ALPHA = 0.3
    EMA_LONG_ALPHA  = 0.05

    FUSION_WEAPON_BOOST       = 0.40
    FUSION_MOTION_BOOST       = 0.15
    FUSION_AUDIO_BOOST        = 0.10
    FUSION_NO_SUPPORT_PENALTY = 0.15

    ALERT_RISK_HIGH   = 0.70
    ALERT_RISK_MEDIUM = 0.40

    AE_WINDOW_PAD_RIGHT = True


# ══════════════════════════════════════════════════════════════════════════════
# UTILITIES
# ══════════════════════════════════════════════════════════════════════════════
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def compute_dynamic_threshold(scores: List[float],
                               floor: float = Config.SAFE_THRESHOLD_FLOOR) -> float:
    if not scores:
        return Config.SAFE_THRESHOLD_LEGACY
    arr = np.array(scores, dtype=np.float64)
    t   = float(np.mean(arr) + 2.0 * np.std(arr))
    return max(t, floor)


def extract_frames(video_path: str) -> Tuple[List[np.ndarray], float]:
    cap    = cv2.VideoCapture(video_path)
    frames = []
    fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames, fps


def extract_audio(video_path: str) -> Optional[str]:
    output_path = "/tmp/audio.wav"
    os.system(
        f"ffmpeg -y -loglevel quiet -i '{video_path}' "
        f"-ac 1 -ar 22050 '{output_path}' 2>/dev/null"
    )
    return output_path if os.path.exists(output_path) else None


def frame_to_timestamp(frame_idx: int, fps: float) -> str:
    """Convert a frame index to MM:SS.mmm."""
    if fps <= 0:
        fps = 25.0
    total_seconds = frame_idx / fps
    minutes = int(total_seconds // 60)
    seconds = total_seconds % 60
    return f"{minutes:02d}:{seconds:06.3f}"


def calculate_enhanced_score(img: torch.Tensor, recon: torch.Tensor) -> float:
    mse = torch.mean((img - recon) ** 2).item()
    mae = torch.mean(torch.abs(img - recon)).item()
    return 0.6 * mse + 0.4 * mae


def compute_ema(values: np.ndarray, alpha: float) -> np.ndarray:
    ema    = np.zeros_like(values)
    ema[0] = values[0]
    for i in range(1, len(values)):
        ema[i] = alpha * values[i] + (1.0 - alpha) * ema[i - 1]
    return ema


def find_anomaly_extent(scores: List[float],
                         threshold: float) -> Tuple[int, int]:
    above = [i for i, s in enumerate(scores) if s > threshold]
    if not above:
        peak = int(np.argmax(scores))
        half = max(1, len(scores) // 10)
        return max(0, peak - half), min(len(scores) - 1, peak + half)
    return above[0], above[-1]


def z_score_normalize(arr: np.ndarray) -> np.ndarray:
    mean = np.mean(arr)
    std  = np.std(arr)
    return (arr - mean) / max(std, 1e-8)


def check_audio_has_content(audio_path: str) -> bool:
    if not audio_path or not os.path.exists(audio_path):
        return False
    if not LIBROSA_AVAILABLE:
        return False
    try:
        y, sr = librosa.load(audio_path, sr=22050, duration=10)
        rms = float(np.sqrt(np.mean(y ** 2)))
        return rms > 1e-4
    except Exception:
        return False


# ══════════════════════════════════════════════════════════════════════════════
# SEGMENT-WISE AE-WEIGHTED TEMPORAL SAMPLING  (PATCH-SAMPLING2)
# ══════════════════════════════════════════════════════════════════════════════
def segmentwise_ae_sampling(
    frames: List[np.ndarray],
    ae_scores: List[float],
    sampled_indices: List[int],
    num_frames: int = 16,
) -> List[np.ndarray]:
    n_total = len(frames)
    if n_total == 0:
        blank = np.zeros((224, 224, 3), dtype=np.uint8)
        return [blank] * num_frames
    if n_total < num_frames:
        padded = list(frames)
        while len(padded) < num_frames:
            padded.append(padded[-1])
        return padded
    if not ae_scores or not sampled_indices:
        step = n_total / num_frames
        return [frames[min(int(i * step), n_total - 1)] for i in range(num_frames)]
    if len(ae_scores) != len(sampled_indices):
        raise ValueError(
            f"segmentwise_ae_sampling: len(ae_scores)={len(ae_scores)} != "
            f"len(sampled_indices)={len(sampled_indices)}."
        )
    for pos, fi in enumerate(sampled_indices):
        if not (0 <= int(fi) < n_total):
            raise ValueError(
                f"segmentwise_ae_sampling: sampled_indices[{pos}]={fi} out of "
                f"range for frames of length {n_total}."
            )
    segment_size = n_total / num_frames
    buckets: List[List[Tuple[int, float]]] = [[] for _ in range(num_frames)]
    for raw_fi, raw_score in zip(sampled_indices, ae_scores):
        fi    = int(raw_fi)
        score = float(raw_score)
        seg   = min(int(fi / segment_size), num_frames - 1)
        buckets[seg].append((fi, score))
    selected: List[np.ndarray] = []
    for seg_idx in range(num_frames):
        seg_start = int(seg_idx * segment_size)
        seg_end   = min(int((seg_idx + 1) * segment_size), n_total)
        if seg_start >= seg_end:
            best_fi = min(seg_start, n_total - 1)
        elif buckets[seg_idx]:
            best_fi = max(buckets[seg_idx], key=lambda x: x[1])[0]
        else:
            best_fi = (seg_start + seg_end) // 2
        best_fi = min(max(best_fi, 0), n_total - 1)
        selected.append(frames[best_fi])
    return selected


# ══════════════════════════════════════════════════════════════════════════════
# AUTOENCODER MODEL DEFINITION
# ══════════════════════════════════════════════════════════════════════════════
class ResNetAE(nn.Module):
    def __init__(self, latent_dim: int = 512):
        super().__init__()
        r = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.encoder  = nn.Sequential(*list(r.children())[:-2])
        self.avgpool  = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_enc   = nn.Linear(2048, latent_dim)
        self.fc_dec   = nn.Linear(latent_dim, 2048)
        self.decoder  = nn.Sequential(
            nn.ConvTranspose2d(2048, 1024, 4, 1, 0),
            nn.BatchNorm2d(1024), nn.ReLU(),
            nn.ConvTranspose2d(1024, 512, 4, 2, 1),
            nn.BatchNorm2d(512), nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 3, 4, 2, 1)
        )
        self.upsample = nn.Upsample((224, 224), mode="bilinear", align_corners=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.encoder(x)
        l = self.fc_enc(self.avgpool(f).flatten(1))
        r = self.upsample(self.decoder(self.fc_dec(l).view(-1, 2048, 1, 1)))
        return torch.sigmoid(r)


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 0 — AUTOENCODER
# ══════════════════════════════════════════════════════════════════════════════
class Stage0AE:
    def __init__(self, config: Config):
        self.config    = config
        self.model: Optional[ResNetAE] = None
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    def load_model(self):
        log.info("[Stage0-AE] Loading autoencoder...")
        self.model = ResNetAE().to(self.config.DEVICE)
        path = self.config.PATHS["AE"]
        if os.path.exists(path):
            ckpt = torch.load(path, map_location=self.config.DEVICE, weights_only=False)
            if isinstance(ckpt, dict):
                for k in ["model_state", "model_state_dict", "state_dict"]:
                    if k in ckpt:
                        state = ckpt[k]
                        break
                else:
                    raise ValueError(f"No valid model state key: {ckpt.keys()}")
            else:
                state = ckpt
            self.model.load_state_dict(state)
        else:
            log.warning(f"[Stage0-AE] Checkpoint not found: {path}")
        self.model.eval()
        return self

    def analyze(self, video_path: str) -> dict:
        if self.model is None:
            self.load_model()
        log.info("[Stage0-AE] Extracting frames...")
        frames, fps = extract_frames(video_path)
        log.info(f"[Stage0-AE] {len(frames)} frames @ {fps:.1f} FPS")
        sampled_indices = list(range(0, len(frames), self.config.AE_FRAME_STRIDE))
        scores: List[float] = []
        bs = self.config.AE_BATCH_SIZE
        log.info(f"[Stage0-AE] Scoring {len(sampled_indices)} frames (batch={bs})...")
        for batch_start in range(0, len(sampled_indices), bs):
            batch_indices = sampled_indices[batch_start: batch_start + bs]
            tensors = []
            for idx in batch_indices:
                img    = Image.fromarray(frames[idx])
                tensor = self.transform(img)
                tensors.append(tensor)
            batch = torch.stack(tensors).to(self.config.DEVICE)
            with torch.no_grad():
                recon = self.model(batch)
            for i in range(len(batch_indices)):
                score = calculate_enhanced_score(batch[i:i+1], recon[i:i+1])
                scores.append(score)
            del batch, recon
            cleanup_memory()
        if not scores:
            return {
                "status": "SAFE", "max_error": 0.0, "avg_error": 0.0,
                "peak_frame_idx": 0, "peak_score_idx": 0,
                "frames": frames, "scores": scores, "fps": fps,
                "threshold": self.config.SAFE_THRESHOLD_LEGACY,
                "anomaly_start_frame": 0, "anomaly_end_frame": 0,
                "sampled_indices": sampled_indices,
            }
        threshold      = compute_dynamic_threshold(scores)
        max_error      = float(np.max(scores))
        avg_error      = float(np.mean(scores))
        peak_score_idx = int(np.argmax(scores))
        peak_frame_idx = sampled_indices[peak_score_idx]
        anom_start_si, anom_end_si = find_anomaly_extent(scores, threshold)
        anom_start_frame = sampled_indices[anom_start_si]
        anom_end_frame   = sampled_indices[anom_end_si]
        is_anomaly = max_error > threshold or avg_error > threshold * 0.7
        status     = "CRITICAL" if is_anomaly else "SAFE"
        cleanup_memory()
        log.info(f"[Stage0-AE] {status} | max={max_error:.6f} | "
                 f"avg={avg_error:.6f} | threshold={threshold:.6f}")
        return {
            "status":              status,
            "max_error":           max_error,
            "avg_error":           avg_error,
            "peak_frame_idx":      peak_frame_idx,
            "peak_score_idx":      peak_score_idx,
            "frames":              frames,
            "scores":              scores,
            "sampled_indices":     sampled_indices,
            "fps":                 fps,
            "threshold":           threshold,
            "anomaly_start_frame": anom_start_frame,
            "anomaly_end_frame":   anom_end_frame,
        }

    def build_ae_signals(self, scores: List[float],
                          peak_score_idx: int) -> np.ndarray:
        W   = self.config.RL_AE_WINDOW
        eps = 1e-8
        if len(scores) == 0:
            return np.zeros((W, 6), dtype=np.float32)
        arr = np.array(scores, dtype=np.float64)
        n   = len(arr)
        delta = np.zeros(n); accel = np.zeros(n)
        delta[1:] = arr[1:] - arr[:-1]
        accel[2:] = delta[2:] - delta[1:-1]
        ema_s = compute_ema(arr, self.config.EMA_SHORT_ALPHA)
        ema_l = compute_ema(arr, self.config.EMA_LONG_ALPHA)
        spike = arr / (ema_l + eps)
        full_signals = np.stack([arr, delta, accel, ema_s, ema_l, spike], axis=1)
        for c in range(6):
            full_signals[:, c] = z_score_normalize(full_signals[:, c])
        half_w    = W // 2
        start_idx = max(0, peak_score_idx - half_w)
        end_idx   = min(n, start_idx + W)
        window    = full_signals[start_idx:end_idx]
        if len(window) < W:
            pad    = np.zeros((W - len(window), 6), dtype=np.float32)
            window = np.concatenate([window, pad], axis=0)
        return window[:W].astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 1 — TIMESFORMER BINARY
# ══════════════════════════════════════════════════════════════════════════════
class Stage1TSBinary:
    def __init__(self, config: Config):
        self.config = config
        self.model  = None
        self._last_logits: Optional[np.ndarray] = None
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def load_model(self):
        if not TRANSFORMERS_AVAILABLE:
            log.warning("[Stage1-TSBinary] transformers unavailable.")
            return self
        log.info("[Stage1-TSBinary] Loading binary TimeSformer...")
        self.model = TimesformerForVideoClassification.from_pretrained(
            "facebook/timesformer-base-finetuned-k400",
            num_labels=2,
            ignore_mismatched_sizes=True,
        ).to(self.config.DEVICE)
        path = self.config.PATHS["TS_BINARY"]
        if os.path.exists(path):
            raw = torch.load(path, map_location=self.config.DEVICE, weights_only=False)
            if isinstance(raw, dict) and "model" in raw:
                state = raw["model"]
            elif isinstance(raw, dict) and "model_state_dict" in raw:
                state = raw["model_state_dict"]
            else:
                state = raw
            self.model.load_state_dict(state, strict=False)
            log.info("[Stage1-TSBinary] Weights loaded.")
        else:
            log.warning(f"[Stage1-TSBinary] Checkpoint not found: {path}")
        self.model.eval()
        return self

    def analyze(self, frames: List[np.ndarray], peak_idx: int) -> dict:
        if self.model is None:
            self.load_model()
        if self.model is None or not frames:
            self._last_logits = np.zeros(2, dtype=np.float32)
            return {"classification": "Unknown", "confidence": 0.0,
                    "prediction_idx": -1, "logits": self._last_logits.tolist()}
        log.info("[Stage1-TSBinary] Binary classification (fixed-interval clip)...")
        clip = frames[max(0, peak_idx - 16): peak_idx + 16]
        while len(clip) < 32:
            clip.append(clip[-1] if clip else np.zeros((224, 224, 3), dtype=np.uint8))
        tensors = torch.stack([self.transform(f) for f in clip])
        tensors = tensors.unsqueeze(0).to(self.config.DEVICE)
        with torch.no_grad():
            outputs = self.model(pixel_values=tensors)
            logits  = outputs.logits
            pred    = torch.argmax(logits, 1).item()
            conf    = torch.softmax(logits, 1).max().item()
            self._last_logits = logits.cpu().numpy().flatten().astype(np.float32)
        classification = "Normal" if pred == 0 else "Suspicious"
        cleanup_memory()
        log.info(f"[Stage1-TSBinary] {classification} ({conf:.2%})")
        return {"classification": classification, "confidence": conf,
                "prediction_idx": pred, "logits": self._last_logits.tolist()}

    def get_logits(self) -> np.ndarray:
        if self._last_logits is None:
            return np.zeros(2, dtype=np.float32)
        return self._last_logits


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2 — TIMESFORMER MULTI-CLASS  (PATCH-SAMPLING)
# ══════════════════════════════════════════════════════════════════════════════
class _TSMultiWrapper(nn.Module):
    """Exact replica of TimeSformerAnomalyClassifier from training."""
    def __init__(self, n_classes: int = 7):
        super().__init__()
        self.timesformer = TimesformerForVideoClassification.from_pretrained(
            "facebook/timesformer-base-finetuned-k400",
            num_labels=n_classes,
            ignore_mismatched_sizes=True,
        )
        h = self.timesformer.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(h, h // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(h // 2, n_classes),
        )
        self.timesformer.classifier = self.classifier

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        return self.timesformer(pixel_values=pixel_values).logits


class Stage2TSMulti:
    def __init__(self, config: Config):
        self.config    = config
        self.model     = None
        self.processor = None
        self._last_logits: Optional[np.ndarray] = None

    def load_model(self):
        if not TRANSFORMERS_AVAILABLE:
            return self
        log.info("[Stage2-TSMulti] Loading multi-class TimeSformer...")
        path = self.config.PATHS["TS_MULTI"]
        if not os.path.exists(path):
            log.warning(f"[Stage2-TSMulti] Not found: {path}")
            return self
        try:
            raw = torch.load(path, map_location=self.config.DEVICE, weights_only=False)
            if isinstance(raw, dict):
                state_dict = (raw.get("model_state_dict") or
                              raw.get("model") or
                              raw.get("state_dict") or raw)
            else:
                state_dict = raw

            model = _TSMultiWrapper(n_classes=7)
            # Strip duplicate "timesformer.classifier.*" keys
            cleaned = {k: v for k, v in state_dict.items()
                       if not k.startswith("timesformer.classifier.")}
            n_removed = len(state_dict) - len(cleaned)
            if n_removed:
                log.info(f"[Stage2-TSMulti] Stripped {n_removed} duplicate keys.")
            missing, unexpected = model.load_state_dict(cleaned, strict=False)
            if missing:
                log.warning(f"[Stage2-TSMulti] Missing keys ({len(missing)}): {missing[:3]}")
            if unexpected:
                log.warning(f"[Stage2-TSMulti] Unexpected keys ({len(unexpected)}): {unexpected[:3]}")
            self.model = model.to(self.config.DEVICE).eval()
            self.processor = VideoMAEImageProcessor.from_pretrained(
                "facebook/timesformer-base-finetuned-k400")
            log.info("[Stage2-TSMulti] Loaded successfully.")
        except Exception as e:
            log.error(f"[Stage2-TSMulti] Load failed: {e}")
            import traceback; traceback.print_exc()
            self.model = None
        return self

    def analyze(self,
                frames: List[np.ndarray],
                peak_idx: int,
                ae_scores: Optional[List[float]] = None,
                sampled_indices: Optional[List[int]] = None) -> dict:
        if self.model is None or self.processor is None:
            self.load_model()
        if self.model is None:
            self._last_logits = np.zeros(7, dtype=np.float32)
            return {"class": "Unknown", "confidence": 0.0, "top3": [],
                    "all_probabilities": {}, "logits": self._last_logits.tolist(),
                    "sampling_method": "unavailable"}
        log.info("[Stage2-TSMulti] Multi-class classification (segment-wise AE sampling)...")
        if ae_scores is not None and sampled_indices is not None and len(ae_scores) > 0:
            clip = segmentwise_ae_sampling(
                frames=frames, ae_scores=ae_scores,
                sampled_indices=sampled_indices, num_frames=16)
            sampling_method = "segmentwise_ae"
        else:
            start = max(0, peak_idx - 8)
            clip  = frames[start: min(len(frames), peak_idx + 8)]
            while len(clip) < 16:
                clip.append(clip[-1] if clip else np.zeros((224, 224, 3), dtype=np.uint8))
            clip = clip[:16]
            sampling_method = "fixed_window_fallback"
        pil = [Image.fromarray(f) if isinstance(f, np.ndarray) else f for f in clip]
        try:
            inputs = self.processor(pil, return_tensors="pt")
            pv     = inputs["pixel_values"].to(self.config.DEVICE)
            with torch.no_grad():
                logits = self.model(pixel_values=pv)
                probs  = torch.softmax(logits, dim=-1)[0]
                self._last_logits = logits.cpu().numpy().flatten()[:7].astype(np.float32)
            top3_p, top3_i = torch.topk(probs, min(3, len(probs)))
            cls   = self.config.CLASS_NAMES.get(top3_i[0].item(), "Unknown")
            top3  = [(self.config.CLASS_NAMES.get(i.item(), "Unknown"), p.item())
                     for i, p in zip(top3_i, top3_p)]
            all_p = {self.config.CLASS_NAMES.get(i, "Unknown"): probs[i].item()
                     for i in range(len(probs))}
            cleanup_memory()
            log.info(f"[Stage2-TSMulti] {cls} ({top3_p[0].item():.2%}) via {sampling_method}")
            return {"class": cls, "confidence": top3_p[0].item(),
                    "top3": top3, "all_probabilities": all_p,
                    "logits": self._last_logits.tolist(),
                    "sampling_method": sampling_method}
        except Exception as e:
            log.error(f"[Stage2-TSMulti] Inference failed: {e}")
            self._last_logits = np.zeros(7, dtype=np.float32)
            return {"class": "Error", "confidence": 0.0, "top3": [],
                    "all_probabilities": {}, "logits": self._last_logits.tolist(),
                    "sampling_method": sampling_method}

    def get_logits(self) -> np.ndarray:
        if self._last_logits is None:
            return np.zeros(7, dtype=np.float32)
        return self._last_logits


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 3 — YOLO WEAPON DETECTOR (EXPERIMENTAL)
# ══════════════════════════════════════════════════════════════════════════════
class Stage3YOLOWeapon:
    def __init__(self, config: Config):
        self.config = config
        self.model  = None

    def load_model(self):
        if not YOLO_AVAILABLE:
            return self
        path = self.config.PATHS["YOLO"]
        try:
            load_path = path if os.path.exists(path) else "yolov8n.pt"
            self.model = YOLO(load_path)
            log.info(f"[Stage3-YOLO] Weapon model loaded from: {load_path}")
        except Exception as e:
            log.warning(f"[Stage3-YOLO] Load failed: {e}")
            self.model = None
        return self

    def analyze(self,
                frames: List[np.ndarray],
                anomaly_start_frame: int,
                anomaly_end_frame: int,
                max_frames: int = 32) -> dict:
        if self.model is None:
            self.load_model()
        null_result = {
            "weapon_flag":          False,
            "weapon_count":         0,
            "dominant_class":       "None",
            "best_detection":       {},
            "all_detections":       [],
            "top_frames_annotated": [],
            "spatial_heatmap":      np.zeros(196, dtype=np.float32),
            "summary":              "YOLO weapon detection unavailable",
        }
        if self.model is None or not frames:
            return null_result
        a_start = max(0, anomaly_start_frame)
        a_end   = min(len(frames) - 1, anomaly_end_frame)
        n_avail = a_end - a_start + 1
        if n_avail <= 0:
            return null_result
        step         = max(1, n_avail // max_frames)
        scan_indices = list(range(a_start, a_end + 1, step))[:max_frames]
        log.info(f"[Stage3-YOLO] Scanning {len(scan_indices)} frames in [{a_start},{a_end}]...")
        all_detections  = []
        best_detection  = {}
        best_conf       = 0.0
        class_counts: Dict[int, int] = {0: 0, 1: 0}
        spatial_grid    = np.zeros(196, dtype=np.float32)
        frame_conf_list = []
        for fi in scan_indices:
            try:
                results_yolo = self.model(frames[fi], verbose=False)[0]
                frame_has_det  = False
                frame_max_conf = 0.0
                annotated_frame = frames[fi].copy()
                for box in results_yolo.boxes:
                    cls_idx = int(box.cls[0])
                    if cls_idx not in self.config.WEAPON_CLASS_NAMES:
                        continue
                    conf = float(box.conf[0])
                    x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
                    det = {
                        "class_idx":  cls_idx,
                        "class_name": self.config.WEAPON_CLASS_NAMES[cls_idx],
                        "confidence": conf,
                        "frame_idx":  fi,
                    }
                    all_detections.append(det)
                    class_counts[cls_idx] = class_counts.get(cls_idx, 0) + 1
                    if conf > best_conf:
                        best_conf      = conf
                        best_detection = det
                    h, w = frames[fi].shape[:2]
                    cx   = (x1 + x2) / 2 / max(w, 1)
                    cy   = (y1 + y2) / 2 / max(h, 1)
                    gx   = min(int(cx * 14), 13)
                    gy   = min(int(cy * 14), 13)
                    spatial_grid[gy * 14 + gx] += conf
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (255, 50, 50), 3)
                    label = f"{self.config.WEAPON_CLASS_NAMES[cls_idx]} {conf:.0%}"
                    cv2.putText(annotated_frame, label, (x1, max(y1 - 10, 15)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 50, 50), 2)
                    frame_has_det  = True
                    frame_max_conf = max(frame_max_conf, conf)
                if frame_has_det:
                    frame_conf_list.append((fi, frame_max_conf, annotated_frame))
            except Exception as e:
                log.warning(f"[Stage3-YOLO] Frame {fi} detection failed: {e}")
        frame_conf_list.sort(key=lambda x: x[1], reverse=True)
        top_frames_annotated = [(fi, ann) for fi, _, ann in frame_conf_list[:2]]
        weapon_count   = len(all_detections)
        weapon_flag    = weapon_count > 0
        dom_cls_idx    = max(class_counts, key=class_counts.get)
        dominant_class = (self.config.WEAPON_CLASS_NAMES[dom_cls_idx]
                          if weapon_flag else "None")
        s_max = spatial_grid.max()
        if s_max > 0:
            spatial_grid /= s_max
        summary = (
            f"Weapons detected, dominant: {dominant_class}, best confidence: {best_conf:.2%}"
            if weapon_flag else "No weapons detected"
        )
        log.info(f"[Stage3-YOLO] {summary}")
        return {
            "weapon_flag":          weapon_flag,
            "weapon_count":         weapon_count,
            "dominant_class":       dominant_class,
            "best_conf":            best_conf,
            "best_detection":       best_detection,
            "all_detections":       all_detections,
            "top_frames_annotated": top_frames_annotated,
            "spatial_heatmap":      spatial_grid,
            "summary":              summary,
        }


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 4 — AUDIO FEATURE EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════
class Stage4Audio:
    def __init__(self, config: Config):
        self.config = config
        self._NORM_RANGES = {
            "zcr":               (0.0, 0.5),
            "mfcc_variance":     (0.0, 200.0),
            "spectral_flux":     (0.0, 50.0),
            "rms_energy":        (0.0, 0.5),
            "spectral_centroid": (0.0, 8000.0),
        }

    def _global_normalize(self, value: float, key: str) -> float:
        lo, hi = self._NORM_RANGES.get(key, (0.0, 1.0))
        return float(np.clip((value - lo) / max(hi - lo, 1e-8), 0.0, 1.0))

    def analyze(self, video_path: str) -> dict:
        log.info("[Stage4-Audio] Extracting audio features...")
        audio_path = extract_audio(video_path)
        null_result = {
            "status":             "NO_AUDIO",
            "audio_present":      False,
            "zcr":                0.0,
            "mfcc_variance":      0.0,
            "spectral_flux":      0.0,
            "rms_energy":         0.0,
            "spectral_centroid":  0.0,
            "audio_score":        0.0,
            "audio_features_vec": np.zeros(5, dtype=np.float32),
        }
        if not audio_path:
            return null_result
        has_content = check_audio_has_content(audio_path)
        if not has_content:
            null_result["status"] = "SILENT"
            null_result["audio_present"] = False
            log.info("[Stage4-Audio] Audio track is silent or absent.")
            return null_result
        if not LIBROSA_AVAILABLE:
            null_result["status"] = "LIBROSA_UNAVAILABLE"
            return null_result
        try:
            y, sr = librosa.load(audio_path, sr=22050, duration=30)
            zcr_raw   = float(np.mean(librosa.feature.zero_crossing_rate(y)[0]))
            mfcc_raw  = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            mfcc_var  = float(np.mean(np.var(mfcc_raw, axis=1)))
            S         = np.abs(librosa.stft(y))
            flux      = np.sqrt(np.mean(np.diff(S, axis=1) ** 2, axis=0))
            flux_mean = float(np.mean(flux))
            rms       = float(np.mean(librosa.feature.rms(y=y)[0]))
            sc        = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)[0]))
            zcr_n  = self._global_normalize(zcr_raw,   "zcr")
            mfcc_n = self._global_normalize(mfcc_var,  "mfcc_variance")
            flux_n = self._global_normalize(flux_mean, "spectral_flux")
            rms_n  = self._global_normalize(rms,       "rms_energy")
            sc_n   = self._global_normalize(sc,        "spectral_centroid")
            audio_score = float(np.clip(
                0.30 * flux_n + 0.25 * mfcc_n + 0.20 * zcr_n +
                0.15 * rms_n  + 0.10 * sc_n, 0.0, 1.0))
            audio_features_vec = np.array([zcr_n, mfcc_n, flux_n, rms_n, sc_n], dtype=np.float32)
            n_anom = sum([flux_n > 0.4, mfcc_n > 0.4, zcr_n > 0.4, rms_n > 0.4])
            status = ("ANOMALY" if n_anom >= 2 else
                      "SUSPICIOUS" if n_anom == 1 else "NORMAL")
            log.info(f"[Stage4-Audio] {status} | audio_score={audio_score:.3f}")
            return {
                "status":             status,
                "audio_present":      True,
                "zcr":                zcr_raw,
                "mfcc_variance":      mfcc_var,
                "spectral_flux":      flux_mean,
                "rms_energy":         rms,
                "spectral_centroid":  sc,
                "audio_score":        audio_score,
                "audio_features_vec": audio_features_vec,
                "normalized": {
                    "zcr": zcr_n, "mfcc": mfcc_n, "flux": flux_n,
                    "rms": rms_n, "sc":   sc_n,
                },
            }
        except Exception as e:
            log.error(f"[Stage4-Audio] Failed: {e}")
            null_result["status"] = "ERROR"
            return null_result


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 5 — TRACKING / BEHAVIOR EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════
class Stage5Tracking:
    def __init__(self, config: Config):
        self.config = config

    def analyze(self,
                frames: List[np.ndarray],
                anomaly_start_frame: int,
                n_frames: int = 32) -> dict:
        null_result = {
            "motion_intensity":  0.0,
            "behaviors":         [],
            "track_count":       0,
            "tracking_features": np.zeros(20, dtype=np.float32),
            "status":            "UNAVAILABLE",
        }
        if not frames:
            return null_result
        a_start = max(0, anomaly_start_frame)
        clip    = frames[a_start: a_start + n_frames]
        if len(clip) < 2:
            return null_result
        if YOLO_AVAILABLE:
            try:
                return self._yolo_track(clip)
            except Exception as e:
                log.warning(f"[Stage5-Tracking] YOLO tracking failed: {e}, falling back to optical flow.")
        return self._optical_flow_track(clip)

    def _yolo_track(self, clip: List[np.ndarray]) -> dict:
        try:
            tracker_model = YOLO("yolov8n.pt")
        except Exception as e:
            raise RuntimeError(f"Cannot load YOLOv8n for tracking: {e}")
        all_tracks: Dict[int, List[Tuple[float, float]]] = {}
        per_frame_counts = []
        for frame in clip:
            try:
                results = tracker_model.track(frame, persist=True, verbose=False)[0]
                if results.boxes.id is None:
                    per_frame_counts.append(0)
                    continue
                per_frame_counts.append(len(results.boxes.id))
                for tid, box in zip(
                    results.boxes.id.int().tolist(),
                    results.boxes.xyxy.tolist()
                ):
                    cx = (box[0] + box[2]) / 2
                    cy = (box[1] + box[3]) / 2
                    all_tracks.setdefault(int(tid), []).append((cx, cy))
            except Exception:
                per_frame_counts.append(0)
        behaviors, motion_intensity = self._analyze_tracks(all_tracks)
        tracking_features = self._build_tracking_features(
            all_tracks, motion_intensity, behaviors, per_frame_counts)
        log.info(f"[Stage5-Tracking] YOLO | tracks={len(all_tracks)} | "
                 f"motion={motion_intensity:.3f} | behaviors={behaviors}")
        return {
            "motion_intensity":  motion_intensity,
            "behaviors":         behaviors,
            "track_count":       len(all_tracks),
            "tracking_features": tracking_features,
            "status":            "YOLO_TRACK",
        }

    def _optical_flow_track(self, clip: List[np.ndarray]) -> dict:
        magnitudes = []
        prev_gray = cv2.cvtColor(clip[0], cv2.COLOR_RGB2GRAY)
        for frame in clip[1:]:
            gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
            try:
                flow = cv2.calcOpticalFlowFarneback(
                    prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
                magnitudes.append(float(np.mean(mag)))
            except Exception:
                pass
            prev_gray = gray
        if not magnitudes:
            return {
                "motion_intensity": 0.0, "behaviors": [],
                "track_count": 0,
                "tracking_features": np.zeros(20, dtype=np.float32),
                "status": "OPTICAL_FLOW_FAILED",
            }
        mag_arr          = np.array(magnitudes)
        mean_mag         = float(np.mean(mag_arr))
        motion_intensity = float(np.clip(mean_mag / 15.0, 0.0, 1.0))
        std_mag          = float(np.std(mag_arr))
        behaviors = []
        if mean_mag > 8.0:
            behaviors.append("FAST_MOVEMENT")
        elif mean_mag < 1.5:
            behaviors.append("STATIC")
        if std_mag > mean_mag * 0.5 and mean_mag > 3.0:
            behaviors.append("CLUSTERING")
        tracking_features = np.zeros(20, dtype=np.float32)
        tracking_features[0] = float(np.clip(mean_mag / 15.0, 0, 1))
        tracking_features[1] = float(np.clip(float(np.max(mag_arr)) / 30.0, 0, 1))
        tracking_features[2] = float(np.clip(std_mag / 10.0, 0, 1))
        tracking_features[3] = float("FAST_MOVEMENT" in behaviors)
        tracking_features[4] = float("STATIC" in behaviors)
        tracking_features[5] = float("CLUSTERING" in behaviors)
        series_len = min(14, len(magnitudes))
        norm_series = np.clip(np.array(magnitudes[:series_len]) / 15.0, 0, 1)
        tracking_features[6:6 + series_len] = norm_series.astype(np.float32)
        return {
            "motion_intensity":  motion_intensity,
            "behaviors":         behaviors,
            "track_count":       0,
            "tracking_features": tracking_features,
            "status":            "OPTICAL_FLOW",
        }

    @staticmethod
    def _analyze_tracks(tracks):
        behaviors         = []
        all_displacements = []
        tids              = list(tracks.keys())
        for tid, pts in tracks.items():
            if len(pts) < 2:
                continue
            disp_per_frame = [
                math.hypot(pts[i][0]-pts[i-1][0], pts[i][1]-pts[i-1][1])
                for i in range(1, len(pts))
            ]
            all_displacements.extend(disp_per_frame)
            avg_disp = sum(disp_per_frame) / max(len(disp_per_frame), 1)
            if avg_disp > 8.0:
                behaviors.append("FAST_MOVEMENT")
            if avg_disp < 2.0 and len(pts) >= 10:
                behaviors.append("STATIC")
        for i in range(len(tids)):
            for j in range(i + 1, len(tids)):
                a, b = tracks[tids[i]], tracks[tids[j]]
                n = min(len(a), len(b))
                if n < 6:
                    continue
                dists = [math.hypot(a[k][0]-b[k][0], a[k][1]-b[k][1]) for k in range(n)]
                dec = sum(1 for k in range(1, n) if dists[k] < dists[k-1])
                if dec / n > 0.75 and dists[0] - dists[-1] > 50:
                    behaviors.append("CLUSTERING")
                    break
        behaviors = list(dict.fromkeys(behaviors))
        if not behaviors:
            behaviors.append("STATIC")
        mean_disp        = float(np.mean(all_displacements)) if all_displacements else 0.0
        motion_intensity = float(np.clip(mean_disp / 15.0, 0.0, 1.0))
        return behaviors, motion_intensity

    @staticmethod
    def _build_tracking_features(tracks, motion_intensity, behaviors, per_frame_counts):
        feat    = np.zeros(20, dtype=np.float32)
        feat[0] = float(np.clip(motion_intensity, 0, 1))
        feat[1] = float(np.clip(len(tracks) / 10.0, 0, 1))
        feat[2] = float("FAST_MOVEMENT" in behaviors)
        feat[3] = float("STATIC" in behaviors)
        feat[4] = float("CLUSTERING" in behaviors)
        feat[5] = float(np.clip(np.mean(per_frame_counts) / 10.0, 0, 1)
                        if per_frame_counts else 0.0)
        all_v = []
        for pts in tracks.values():
            if len(pts) < 2:
                continue
            all_v.extend([
                math.hypot(pts[i][0]-pts[i-1][0], pts[i][1]-pts[i-1][1])
                for i in range(1, len(pts))
            ])
        feat[6] = float(np.clip(np.std(all_v) / 10.0, 0, 1)) if all_v else 0.0
        series_len  = min(13, len(per_frame_counts))
        norm_counts = np.clip(np.array(per_frame_counts[:series_len]) / 10.0, 0, 1)
        feat[7:7 + series_len] = norm_counts.astype(np.float32)
        return feat


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 6 — FUSION LAYER  (PATCH-TRACE)
# ══════════════════════════════════════════════════════════════════════════════
class Stage6Fusion:
    def __init__(self, config: Config):
        self.config = config

    def fuse(self,
             ae_result:        dict,
             ts_binary_result: dict,
             ts_multi_result:  dict,
             yolo_result:      dict,
             audio_result:     dict,
             tracking_result:  dict) -> dict:
        reasoning_trace: List[str] = []
        threshold     = ae_result.get("threshold", self.config.SAFE_THRESHOLD_LEGACY)
        ae_max        = float(ae_result.get("max_error", 0.0))
        ae_score_norm = float(np.clip(ae_max / max(threshold, 1e-8), 0.0, 1.0))
        ts_binary_susp = (ts_binary_result.get("classification") == "Suspicious")
        ts_binary_conf = float(ts_binary_result.get("confidence", 0.0)
                               if ts_binary_susp else 0.0)
        weapon_flag  = bool(yolo_result.get("weapon_flag", False))
        weapon_count = int(yolo_result.get("weapon_count", 0))
        audio_score   = float(audio_result.get("audio_score", 0.0))
        audio_anomaly = (audio_result.get("status") == "ANOMALY")
        motion_int    = float(tracking_result.get("motion_intensity", 0.0))
        behaviors     = tracking_result.get("behaviors", [])
        fast_movement = "FAST_MOVEMENT" in behaviors

        fused_risk = ae_score_norm * 0.50
        reasoning_trace.append(
            f"Baseline: the AE normalised score is {ae_score_norm:.3f} "
            f"(50% weight), giving an initial base risk of {fused_risk:.3f}.")

        if weapon_flag and ae_max > threshold:
            fused_risk = max(fused_risk, 0.90)
            reasoning_trace.append(
                f"Rule 1 TRIGGERED: A weapon was detected AND the AE score "
                f"exceeds the threshold. Risk is escalated to CRITICAL (minimum 0.90).")
        elif ts_binary_susp:
            ts_boost = ts_binary_conf * 0.20
            fused_risk += ts_boost
            reasoning_trace.append(
                f"Rule 2: The binary TimeSformer classified the clip as Suspicious "
                f"with {ts_binary_conf:.2f} confidence (20% weight), adding {ts_boost:.3f}.")

        if fast_movement and audio_anomaly and ae_max > threshold:
            escalation = self.config.FUSION_MOTION_BOOST + self.config.FUSION_AUDIO_BOOST
            fused_risk += escalation
            reasoning_trace.append(
                f"Rule 3 FULLY TRIGGERED: Fast movement, audio anomaly, and AE anomaly "
                f"all present. Adding {escalation:.3f}.")
        elif fast_movement:
            boost = self.config.FUSION_MOTION_BOOST * 0.5
            fused_risk += boost
            reasoning_trace.append(
                f"Rule 3 (partial): Fast movement detected. Adding {boost:.3f}.")
        elif audio_anomaly:
            boost = self.config.FUSION_AUDIO_BOOST * 0.5
            fused_risk += boost
            reasoning_trace.append(
                f"Rule 3 (partial): Audio anomaly detected. Adding {boost:.3f}.")

        has_support = (ts_binary_susp or weapon_flag or audio_anomaly or fast_movement)
        if ae_max > threshold and not has_support:
            fused_risk -= self.config.FUSION_NO_SUPPORT_PENALTY
            reasoning_trace.append(
                f"Rule 4: AE score exceeds threshold but no supporting signal found. "
                f"Applying caution deduction of {self.config.FUSION_NO_SUPPORT_PENALTY:.3f}.")

        if weapon_flag:
            pre_floor  = fused_risk
            fused_risk = max(fused_risk, self.config.FUSION_WEAPON_BOOST)
            if fused_risk != pre_floor:
                reasoning_trace.append(
                    f"Rule 5: Weapon detected, risk floored at minimum "
                    f"{self.config.FUSION_WEAPON_BOOST:.2f}.")

        fused_risk = float(np.clip(fused_risk, 0.0, 1.0))

        if weapon_flag and ae_max > threshold:
            risk_level = "CRITICAL"
        elif fused_risk >= 0.70:
            risk_level = "HIGH"
        elif fused_risk >= 0.40:
            risk_level = "MODERATE"
        else:
            risk_level = "LOW"

        reasoning_trace.append(
            f"Final assessment: the fused risk score is {fused_risk:.4f}, "
            f"classified as {risk_level}.")

        context_vector = np.array([
            float(np.clip(ae_score_norm, 0.0, 1.0)),
            float(ts_binary_conf),
            float(np.clip(audio_score, 0.0, 1.0)),
            float(weapon_flag),
            float(np.clip(motion_int, 0.0, 1.0)),
        ], dtype=np.float32)

        log.info(f"[Stage6-Fusion] {risk_level} | fused_risk={fused_risk:.4f}")
        return {
            "fused_risk":       fused_risk,
            "risk_level":       risk_level,
            "reasoning_trace":  reasoning_trace,
            "context_vector":   context_vector,
            "ae_score_norm":    ae_score_norm,
            "ts_binary_conf":   ts_binary_conf,
            "audio_score":      audio_score,
            "weapon_flag":      weapon_flag,
            "motion_intensity": motion_int,
        }


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 7 — RL3033 (EXPERIMENTAL)
# ══════════════════════════════════════════════════════════════════════════════
class Stage7RL3033:
    def __init__(self, config: Config):
        self.config = config
        self.agent  = None

    def load_agent(self):
        if not RL_AVAILABLE:
            log.warning("[Stage7-RL3033] stable-baselines3 unavailable.")
            return self
        path = self.config.PATHS["RL_AGENT"]
        if not os.path.exists(path):
            log.warning(f"[Stage7-RL3033] Agent not found: {path}")
            return self
        log.info("[Stage7-RL3033] Loading RL3033 agent...")
        try:
            self.agent = PPO.load(path, device="cpu")
            log.info(f"[Stage7-RL3033] Loaded. obs_space={self.agent.observation_space}")
        except Exception as e:
            log.error(f"[Stage7-RL3033] Load failed: {e}")
            self.agent = None
        return self

    def _build_latent_motion(self, ae_signals, tracking_result):
        ae_flat    = ae_signals.flatten()
        track_feat = tracking_result.get("tracking_features", np.zeros(20, dtype=np.float32))
        delta_ch = ae_signals[:, 1]; accel_ch = ae_signals[:, 2]
        spike_ch = ae_signals[:, 5]; err_ch   = ae_signals[:, 0]
        def ch_stats(ch):
            return np.array([
                np.mean(ch), np.std(ch), np.max(ch), np.min(ch),
                np.percentile(ch, 75), np.percentile(ch, 25),
                np.median(ch), float(np.sum(ch > 0)) / max(len(ch), 1),
            ], dtype=np.float32)
        ch_desc = np.concatenate([ch_stats(err_ch), ch_stats(delta_ch),
                                   ch_stats(accel_ch), ch_stats(spike_ch)])
        motion_int = float(tracking_result.get("motion_intensity", 0.0))
        fast_mv    = float("FAST_MOVEMENT" in tracking_result.get("behaviors", []))
        static     = float("STATIC"        in tracking_result.get("behaviors", []))
        cluster    = float("CLUSTERING"    in tracking_result.get("behaviors", []))
        n_tracks   = float(np.clip(tracking_result.get("track_count", 0) / 10.0, 0, 1))
        motion_sum = np.array([motion_int, fast_mv, static, cluster, n_tracks], dtype=np.float32)
        base = np.concatenate([ae_flat, ch_desc, track_feat, motion_sum])
        return np.tile(base, math.ceil(1536 / len(base)))[:1536].astype(np.float32)

    def _build_latent_stack(self, ae_signals, ts_binary_logits, ts_multi_logits,
                             audio_result, tracking_result, fusion_result):
        ae_flat       = ae_signals.flatten().astype(np.float32)
        ts_bin_logits = ts_binary_logits.astype(np.float32)
        ts_mul_logits = ts_multi_logits[:7].astype(np.float32)
        audio_feat    = audio_result.get("audio_features_vec", np.zeros(5, dtype=np.float32))
        track_feat    = tracking_result.get("tracking_features", np.zeros(20, dtype=np.float32))
        context_vec   = fusion_result.get("context_vector", np.zeros(5, dtype=np.float32))
        fusion_signals = np.array([
            float(fusion_result.get("fused_risk",       0.0)),
            float(fusion_result.get("ae_score_norm",    0.0)),
            float(fusion_result.get("ts_binary_conf",   0.0)),
            float(fusion_result.get("audio_score",      0.0)),
            float(fusion_result.get("weapon_flag",      False)),
            float(fusion_result.get("motion_intensity", 0.0)),
            float(fusion_result.get("risk_level") == "CRITICAL"),
            float(fusion_result.get("risk_level") == "HIGH"),
        ], dtype=np.float32)
        base = np.concatenate([ae_flat, ts_bin_logits, ts_mul_logits,
                                audio_feat, track_feat, context_vec, fusion_signals])
        return np.tile(base, math.ceil(2560 / len(base)))[:2560].astype(np.float32)

    def _build_spatial(self, yolo_result, tracking_result, frames, peak_frame_idx):
        spatial      = np.zeros(196, dtype=np.float32)
        yolo_heatmap = yolo_result.get("spatial_heatmap", np.zeros(196, dtype=np.float32))
        spatial     += yolo_heatmap * 0.5
        if frames and peak_frame_idx < len(frames):
            frame = frames[peak_frame_idx]
            gray  = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY).astype(np.float32)
            H, W  = gray.shape
            bh, bw = H // 14, W // 14
            if bh > 0 and bw > 0:
                for gy in range(14):
                    for gx in range(14):
                        block = gray[gy*bh:(gy+1)*bh, gx*bw:(gx+1)*bw]
                        spatial[gy*14 + gx] += float(np.var(block)) / 10000.0
        motion_int = float(tracking_result.get("motion_intensity", 0.0))
        for cy in range(4, 10):
            for cx in range(4, 10):
                spatial[cy*14 + cx] += motion_int * 0.3
        s_max = spatial.max()
        if s_max > 0:
            spatial /= s_max
        return spatial

    def _build_observation(self, ae_signals, ts_binary_logits, ts_multi_logits,
                            yolo_result, audio_result, tracking_result,
                            fusion_result, frames, peak_frame_idx):
        context_vector = fusion_result.get("context_vector", np.zeros(5, dtype=np.float32))
        latent_motion = self._build_latent_motion(ae_signals, tracking_result)
        latent_stack  = self._build_latent_stack(ae_signals, ts_binary_logits,
                                                   ts_multi_logits, audio_result,
                                                   tracking_result, fusion_result)
        spatial = self._build_spatial(yolo_result, tracking_result, frames, peak_frame_idx)
        obs = {
            "ae_signals":    ae_signals.astype(np.float32),
            "latent_motion": latent_motion.astype(np.float32),
            "latent_stack":  latent_stack.astype(np.float32),
            "spatial":       spatial.astype(np.float32),
            "context":       context_vector.astype(np.float32),
        }
        expected = {
            "ae_signals":    (self.config.RL_AE_WINDOW, self.config.RL_AE_CHANNELS),
            "latent_motion": (self.config.RL_LATENT_MOT,),
            "latent_stack":  (self.config.RL_LATENT_STK,),
            "spatial":       (self.config.RL_SPATIAL_DIM,),
            "context":       (self.config.RL_CONTEXT_DIM,),
        }
        for key, exp_shape in expected.items():
            if obs[key].shape != exp_shape:
                if len(exp_shape) == 1:
                    flat = obs[key].flatten()
                    obs[key] = (flat[:exp_shape[0]] if len(flat) >= exp_shape[0]
                                else np.pad(flat, (0, exp_shape[0] - len(flat))))
                elif len(exp_shape) == 2:
                    r, c  = exp_shape; needed = r * c
                    flat  = obs[key].flatten()
                    obs[key] = (flat[:needed].reshape(r, c) if len(flat) >= needed
                                else np.pad(flat, (0, needed - len(flat))).reshape(r, c))
        return obs

    def analyze(self, ae_signals, ts_binary_logits, ts_multi_logits,
                yolo_result, audio_result, tracking_result,
                fusion_result, frames, peak_frame_idx):
        if self.agent is None:
            self.load_agent()
        if self.agent is None:
            fused_risk   = float(fusion_result.get("fused_risk", 0.0))
            tfb_score    = float(fusion_result.get("ts_binary_conf", 0.0))
            tfm_score    = float(fusion_result.get("ae_score_norm", 0.0))
            # RL unavailable: treat rl_score = fused_risk as best available proxy
            final_risk   = float(np.clip(
                0.70 * tfb_score + 0.20 * tfm_score +
                0.05 * fused_risk + 0.05 * fused_risk,
                0.0, 1.0))
            if fusion_result.get("weapon_flag", False):
                final_risk = max(final_risk, 0.85)
            status = (
                "CRITICAL"  if final_risk >= 0.85 else
                "HIGH_RISK" if final_risk >= 0.65 else
                "MODERATE"  if final_risk >= 0.40 else
                "LOW_RISK"
            )
            return {
                "final_risk":     final_risk,
                "fused_risk":     fused_risk,
                "status":         status,
                "interpretation": (
                    f"RL agent unavailable. Final risk from weighted formula: "
                    f"70% TFB ({tfb_score:.2%}) + 20% TFM ({tfm_score:.2%}) + "
                    f"10% Fused ({fused_risk:.2%}) = {final_risk:.2%} [{status}]."
                ),
            }
        try:
            obs = self._build_observation(ae_signals, ts_binary_logits, ts_multi_logits,
                                           yolo_result, audio_result, tracking_result,
                                           fusion_result, frames, peak_frame_idx)
            action, _ = self.agent.predict(obs, deterministic=True)
            if isinstance(action, np.ndarray):
                rl_raw = float(action.item() if action.ndim == 0 else action.flat[0])
            else:
                rl_raw = float(action)
            rl_raw     = float(np.clip(rl_raw, 0.0, 1.0))
            fused_risk = float(fusion_result.get("fused_risk", 0.0))
            # ── NEW WEIGHTED FORMULA ──────────────────────────────
            # Retrieve the upstream stage scores needed for the formula.
            # TFB confidence: binary TimeSformer confidence when Suspicious,
            #                 else (1 - confidence) so a high-confidence Normal
            #                 still contributes a LOW risk value.
            _ts_binary_conf_raw = float(fusion_result.get("ts_binary_conf", 0.0))
            # ts_binary_conf in fusion_result is already 0 when Normal (see Stage6Fusion.fuse).
            # For a Suspicious clip it equals the model confidence directly.
            tfb_score  = _ts_binary_conf_raw          # 0‒1
            # TFM: use ae_score_norm as a proxy for the multi-class signal magnitude
            # (the actual logit-based confidence is not stored in fusion_result, but
            # ae_score_norm encodes how anomalous the peak frame is, which directly
            # corroborates the multi-class prediction strength).
            tfm_score  = float(fusion_result.get("ae_score_norm", 0.0))   # 0‒1
            fused_score = float(np.clip(fused_risk, 0.0, 1.0))
            rl_score    = float(np.clip(rl_raw, 0.0, 1.0))
            # Weighted combination: TFB 70 %, TFM 20 %, FUSED 5 %, RL 5 %
            final_risk = float(np.clip(
                0.70 * tfb_score +
                0.20 * tfm_score +
                0.05 * fused_score +
                0.05 * rl_score,
                0.0, 1.0))
            if fusion_result.get("weapon_flag", False):
                final_risk = max(final_risk, 0.85)
            status = (
                "CRITICAL"  if final_risk >= 0.85 else
                "HIGH_RISK" if final_risk >= 0.65 else
                "MODERATE"  if final_risk >= 0.40 else
                "LOW_RISK"
            )
            log.info(f"[Stage7-RL3033] tfb={tfb_score:.4f} tfm={tfm_score:.4f} "
                     f"fused={fused_score:.4f} rl={rl_score:.4f} "
                     f"final={final_risk:.4f} => {status}")
            return {
                "fused_risk":     fused_risk,
                "final_risk":     final_risk,
                "status":         status,
                "tfb_score":      tfb_score,
                "tfm_score":      tfm_score,
                "rl_raw":         rl_raw,
                "interpretation": (
                    f"Final risk computed as: "
                    f"70% TFB ({tfb_score:.2%}) + "
                    f"20% TFM ({tfm_score:.2%}) + "
                    f"5% Fused ({fused_score:.2%}) + "
                    f"5% RL ({rl_score:.2%}) = {final_risk:.2%} [{status}]."
                ),
            }
        except Exception as e:
            import traceback
            log.error(f"[Stage7-RL3033] Inference failed: {e}")
            traceback.print_exc()
            fused_risk = float(fusion_result.get("fused_risk", 0.0))
            return {"final_risk": fused_risk, "status": "ERROR", "error": str(e)}


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 8 — LLAVA (EXPERIMENTAL)
# LOGIC-NORMAL: always runs when enable_llava=True (no Stage 1 gate).
# Uses neutral investigative prompts regardless of Normal/Suspicious result.
# ══════════════════════════════════════════════════════════════════════════════
class Stage8LLaVA:
    def __init__(self, config: Config):
        self.config    = config
        self.model     = None
        self.processor = None

    def load_model(self):
        if not TRANSFORMERS_AVAILABLE:
            return self
        path = self.config.PATHS["LLAVA"]
        if not os.path.exists(path):
            log.warning(f"[Stage8-LLaVA] Model not found: {path}")
            return self
        log.info("[Stage8-LLaVA] Loading Fire-LLaVA...")
        try:
            self.model = LlavaForConditionalGeneration.from_pretrained(
                path, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True)
            self.processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
            self.processor.patch_size = 14
            self.processor.vision_feature_select_strategy = "default"
            self.model.eval()
            log.info("[Stage8-LLaVA] Loaded.")
        except Exception as e:
            log.error(f"[Stage8-LLaVA] Load failed: {e}")
            self.model = None
        return self

    @staticmethod
    def _build_context_prompt(role, frame_number, yolo_summary,
                               motion_behaviors, audio_status, audio_score,
                               binary_result: str = "Unknown"):
        temporal_labels = {
            "baseline":   "Frame 1 of 4 (beginning of clip)",
            "anom_start": "Frame 2 of 4 (area of elevated AE error onset)",
            "peak":       "Frame 3 of 4 (highest AE reconstruction error)",
            "anom_end":   "Frame 4 of 4 (end of elevated-error region)",
        }
        t_label       = temporal_labels.get(role, f"Frame {frame_number} of 4")
        behaviors_str = ", ".join(motion_behaviors) if motion_behaviors else "STATIC"
        weapon_info   = yolo_summary if "weapon" in yolo_summary.lower() else "No weapons detected"
        role_instructions = {
            "baseline":   "Describe the scene at the start of the clip: who is present and what is happening.",
            "anom_start": "Describe what is visible at the onset of the region with elevated reconstruction error.",
            "peak":       "Describe what is most notable in this frame at the reconstruction error peak.",
            "anom_end":   "Describe how the scene looks at the end of the elevated-error region.",
        }
        instruction = role_instructions.get(role, "Describe what you see.")
        binary_note = (f"The binary classifier result is: {binary_result}. "
                       "Use this as context, but form your own description from the image.")
        return (
            f"USER: <image>\n"
            f"You are a forensic security analyst reviewing surveillance footage.\n\n"
            f"=== TEMPORAL CONTEXT ===\n"
            f"Position: {t_label}\n"
            f"Classifier note: {binary_note}\n\n"
            f"=== DETECTED INFORMATION ===\n"
            f"Objects/Weapons: {weapon_info}\n"
            f"Motion Pattern: {behaviors_str}\n"
            f"Audio Status: {audio_status} (score={audio_score:.2f})\n\n"
            f"=== YOUR TASK ===\n"
            f"{instruction}\n"
            f"Be concise (2-3 sentences). Focus on safety-relevant details.\n"
            f"ASSISTANT:"
        )

    def _describe_frame(self, image, role, frame_number, context, binary_result="Unknown"):
        prompt = self._build_context_prompt(
            role=role, frame_number=frame_number,
            yolo_summary=context.get("yolo_summary", "Unknown"),
            motion_behaviors=context.get("behaviors", []),
            audio_status=context.get("audio_status", "Unknown"),
            audio_score=context.get("audio_score", 0.0),
            binary_result=binary_result,
        )
        inputs = self.processor(text=prompt, images=image.convert("RGB"), return_tensors="pt")
        inputs = {k: v.to(self.config.DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            with torch.amp.autocast("cuda"):
                output_ids = self.model.generate(
                    **inputs, max_new_tokens=130, do_sample=True, temperature=0.7,
                    top_p=0.9, top_k=50, repetition_penalty=1.4,
                    no_repeat_ngram_size=3, early_stopping=True)
        text = self.processor.batch_decode(output_ids, skip_special_tokens=True)[0]
        if "ASSISTANT:" in text:
            text = text.split("ASSISTANT:")[-1].strip()
        return self._clean_text(text)

    @staticmethod
    def _clean_text(text):
        if not text or len(text) < 20:
            return text
        sentences = text.replace("!", ".").replace("?", ".").split(".")
        seen, out = set(), []
        for s in sentences:
            s   = s.strip()
            key = s.lower()
            if s and len(s) > 10 and key not in seen:
                seen.add(key)
                out.append(s)
        result = ". ".join(out)
        return result + ("." if result and not result.endswith(".") else "")

    @staticmethod
    def synthesise_narrative(frame_descriptions: dict,
                              context: dict,
                              yolo_result: dict,
                              audio_result: dict,
                              tracking_result: dict,
                              binary_result: str = "Unknown") -> str:
        """
        PDF-LLAVA-NARR: Combine per-frame descriptions into one coherent,
        non-redundant, human-readable scene narrative suitable for a report.
        Deduplicates content, eliminates noise, and maintains temporal logic.
        """
        baseline   = frame_descriptions.get("baseline", "").strip()
        anom_start = frame_descriptions.get("anom_start", "").strip()
        peak       = frame_descriptions.get("peak", "").strip()
        anom_end   = frame_descriptions.get("anom_end", "").strip()

        # Build context enrichment strings
        weapon_flag    = bool(yolo_result.get("weapon_flag", False))
        weapon_class   = yolo_result.get("dominant_class", "None")
        weapon_conf    = float(yolo_result.get("best_conf", 0.0))
        audio_status   = audio_result.get("status", "UNKNOWN")
        audio_score    = float(audio_result.get("audio_score", 0.0))
        behaviors      = tracking_result.get("behaviors", ["STATIC"])
        motion_int     = float(tracking_result.get("motion_intensity", 0.0))
        behaviors_str  = ", ".join(behaviors) if behaviors else "STATIC"

        # Deduplicate sentences across all four frame descriptions
        seen_keys: set = set()
        unique_parts: List[str] = []
        for raw in [baseline, anom_start, peak, anom_end]:
            if not raw:
                continue
            for sent in raw.replace("!", ".").replace("?", ".").split("."):
                sent = sent.strip()
                key  = " ".join(sent.lower().split())
                if len(sent) > 15 and key not in seen_keys:
                    seen_keys.add(key)
                    unique_parts.append(sent)

        # Assemble narrative: scene opening, incident development, peak, resolution
        narrative_parts: List[str] = []

        # Opening: what the scene looks like at baseline
        if baseline:
            narrative_parts.append(baseline)

        # Transition: onset of the anomaly region
        if anom_start and anom_start != baseline:
            # Only add if it contributes genuinely new information
            anom_key = " ".join(anom_start.lower().split())
            base_key = " ".join(baseline.lower().split())
            if anom_key != base_key:
                narrative_parts.append(anom_start)

        # Peak: the most significant moment
        if peak:
            peak_key = " ".join(peak.lower().split())
            existing_keys = set(" ".join(p.lower().split()) for p in narrative_parts)
            if peak_key not in existing_keys:
                narrative_parts.append(peak)

        # Resolution: how the scene concludes
        if anom_end and anom_end not in [baseline, anom_start, peak]:
            end_key = " ".join(anom_end.lower().split())
            existing_keys = set(" ".join(p.lower().split()) for p in narrative_parts)
            if end_key not in existing_keys:
                narrative_parts.append(anom_end)

        # If no valid descriptions produced, return a safe fallback
        if not narrative_parts:
            return "Visual analysis did not produce a usable scene description for this clip."

        # Join into a flowing paragraph
        combined = " ".join(narrative_parts)

        # Append system-derived context enrichment
        context_note_parts = []
        if weapon_flag:
            context_note_parts.append(
                f"Instrument detection flagged a {weapon_class} with {weapon_conf:.0%} confidence."
            )
        if audio_status in ("ANOMALY", "SUSPICIOUS"):
            context_note_parts.append(
                f"The audio track registered as {audio_status} (composite score {audio_score:.2f}), "
                f"suggesting acoustically unusual activity."
            )
        if "FAST_MOVEMENT" in behaviors:
            context_note_parts.append(
                f"Motion analysis recorded rapid subject movement "
                f"(intensity {motion_int:.2f}), consistent with an escalating incident."
            )
        elif "CLUSTERING" in behaviors:
            context_note_parts.append(
                f"Motion analysis detected converging trajectories between subjects "
                f"(intensity {motion_int:.2f}), a pattern associated with confrontational proximity."
            )
        elif motion_int < 0.15:
            context_note_parts.append(
                f"Motion analysis recorded minimal movement (intensity {motion_int:.2f}), "
                f"indicating either a static scene or a post-event lull."
            )

        if context_note_parts:
            combined = combined.rstrip(".") + ". " + " ".join(context_note_parts)

        # Ensure clean ending
        combined = combined.strip()
        if combined and not combined.endswith("."):
            combined += "."

        return combined

    def analyze(self, frames, peak_idx, anomaly_start_frame, anomaly_end_frame,
                yolo_result, audio_result, tracking_result, binary_result="Unknown"):
        if self.model is None or self.processor is None:
            self.load_model()
        if self.model is None or not frames:
            return {"description": "LLaVA not available",
                    "analysis": "", "frame_descriptions": {},
                    "unified_narrative": "LLaVA not available"}
        if anomaly_end_frame < 0:
            anomaly_end_frame = min(peak_idx + 30, len(frames) - 1)
        context = {
            "yolo_summary": yolo_result.get("summary",  "No weapon detection"),
            "behaviors":    tracking_result.get("behaviors", ["STATIC"]),
            "audio_status": audio_result.get("status",   "UNKNOWN"),
            "audio_score":  float(audio_result.get("audio_score", 0.0)),
        }
        keyframes = {
            "baseline":   (frames[0],                                       1),
            "anom_start": (frames[min(anomaly_start_frame, len(frames)-1)], 2),
            "peak":       (frames[min(peak_idx,            len(frames)-1)], 3),
            "anom_end":   (frames[min(anomaly_end_frame,   len(frames)-1)], 4),
        }
        log.info(f"[Stage8-LLaVA] 4-frame investigative analysis (binary={binary_result})...")
        descriptions = {}
        for role, (frame_arr, fn) in keyframes.items():
            log.info(f"[Stage8-LLaVA]   Describing {role} (frame {fn}/4)...")
            try:
                img  = Image.fromarray(frame_arr) if isinstance(frame_arr, np.ndarray) else frame_arr
                desc = self._describe_frame(img, role, fn, context, binary_result)
                descriptions[role] = desc
            except Exception as e:
                log.warning(f"[Stage8-LLaVA] Frame '{role}' failed: {e}")
                descriptions[role] = f"[{role} analysis unavailable]"

        # Build the unified narrative from all frame descriptions
        unified_narrative = self.synthesise_narrative(
            frame_descriptions=descriptions,
            context=context,
            yolo_result=yolo_result,
            audio_result=audio_result,
            tracking_result=tracking_result,
            binary_result=binary_result,
        )

        weapon_str = (f"WEAPON DETECTED: {yolo_result.get('dominant_class','?')}"
                      if yolo_result.get("weapon_flag") else "No weapons detected")
        motion_str = ", ".join(tracking_result.get("behaviors", ["STATIC"]))
        audio_str  = f"{audio_result.get('status','?')} (score={context['audio_score']:.2f})"
        unified = (
            f"[CONTEXT] {weapon_str}, Motion: {motion_str}, Audio: {audio_str}\n"
            f"[FRAME 1 (BEGINNING)] {descriptions.get('baseline','')}\n"
            f"[FRAME 2 (AE ONSET)]  {descriptions.get('anom_start','')}\n"
            f"[FRAME 3 (AE PEAK)]   {descriptions.get('peak','')}\n"
            f"[FRAME 4 (AE END)]    {descriptions.get('anom_end','')}"
        ).strip()
        cleanup_memory()
        return {
            "description":        unified,
            "analysis":           unified,
            "frame_descriptions": descriptions,
            "unified_narrative":  unified_narrative,
            "context_used":       context,
            "binary_result":      binary_result,
        }


# ══════════════════════════════════════════════════════════════════════════════
# ALERT SYSTEM
# ══════════════════════════════════════════════════════════════════════════════
class AlertSystem:
    def __init__(self, config: Config):
        self.config    = config
        self._handlers: List[Any] = []
        self.register(self._default_log_handler)

    def register(self, handler):
        self._handlers.append(handler)
        return self

    def evaluate_and_fire(self, results: dict) -> Optional[dict]:
        stage7 = results.get("stage7") or {}
        stage6 = results.get("stage6") or {}
        stage3 = results.get("stage3") or {}
        stage4 = results.get("stage4") or {}
        final_risk = float(stage7.get("final_risk", stage6.get("fused_risk", 0.0)))
        level = (
            "HIGH"   if final_risk >= self.config.ALERT_RISK_HIGH   else
            "MEDIUM" if final_risk >= self.config.ALERT_RISK_MEDIUM  else
            "LOW"
        )
        if level == "LOW":
            return None
        alert = {
            "timestamp":       datetime.utcnow().isoformat() + "Z",
            "level":           level,
            "final_risk":      round(final_risk, 4),
            "fused_risk":      round(float(stage6.get("fused_risk",    0.0)), 4),
            "weapon_flag":     bool(stage3.get("weapon_flag", False)),
            "weapon_summary":  stage3.get("summary", ""),
            "audio_score":     round(float(stage4.get("audio_score", 0.0)), 4),
            "risk_level":      stage6.get("risk_level", "UNKNOWN"),
            "reasoning_trace": stage6.get("reasoning_trace", []),
            "rl_status":       stage7.get("status", "UNKNOWN"),
            "video_path":      results.get("video_path", ""),
        }
        for handler in self._handlers:
            try:
                handler(alert)
            except Exception as e:
                log.error(f"[AlertSystem] Handler failed: {e}")
        return alert

    @staticmethod
    def _default_log_handler(alert: dict):
        log.warning(
            f"[ALERT] {alert['level']}, "
            f"final_risk={alert['final_risk']:.2%}, "
            f"fused_risk={alert['fused_risk']:.2%}, "
            f"weapon={'YES' if alert['weapon_flag'] else 'NO'}, "
            f"rl_status={alert['rl_status']}"
        )

    @staticmethod
    def file_alert_handler(log_path: str = "/tmp/scenesolver_alerts.jsonl"):
        def _handler(alert: dict):
            with open(log_path, "a") as f:
                f.write(json.dumps(alert) + "\n")
        return _handler


# ══════════════════════════════════════════════════════════════════════════════
# REAL-TIME STREAM PROCESSOR
# ══════════════════════════════════════════════════════════════════════════════
class RealTimeStream:
    def __init__(self, pipeline: "SceneSolverPipeline"):
        self.pipeline  = pipeline
        self._thread   = None
        self._stop_evt = threading.Event()

    def start(self, source: str, window_frames: int = 50, on_alert: Any = None):
        self._stop_evt.clear()
        self._thread = threading.Thread(
            target=self._run, args=(source, window_frames, on_alert),
            daemon=True, name="RealTimeStream")
        self._thread.start()
        log.info(f"[RealTimeStream] Started: {source}")

    def stop(self):
        self._stop_evt.set()
        if self._thread:
            self._thread.join(timeout=10)
        log.info("[RealTimeStream] Stopped.")

    def _run(self, source, window_frames, on_alert):
        cfg       = self.pipeline.config
        ae_stage  = self.pipeline.stage0
        ae_stage.load_model()
        alert_sys = self.pipeline.alert_system
        transform = ae_stage.transform
        cap = cv2.VideoCapture(int(source) if source.isdigit() else source)
        if not cap.isOpened():
            log.error(f"[RealTimeStream] Cannot open: {source}")
            return
        buffer: List[np.ndarray] = []
        scores: List[float]      = []
        frame_count              = 0
        while not self._stop_evt.is_set():
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            buffer.append(frame_rgb)
            frame_count += 1
            if frame_count % cfg.AE_FRAME_STRIDE == 0 and ae_stage.model:
                img    = Image.fromarray(frame_rgb)
                tensor = transform(img).unsqueeze(0).to(cfg.DEVICE)
                with torch.no_grad():
                    recon = ae_stage.model(tensor)
                score = calculate_enhanced_score(tensor, recon)
                scores.append(score)
                del tensor, recon
            if len(buffer) >= window_frames:
                threshold = compute_dynamic_threshold(scores)
                max_err   = max(scores) if scores else 0.0
                if max_err > threshold:
                    mini_results = {
                        "stage0":  {"max_error": max_err},
                        "stage6":  {"fused_risk": float(np.clip(max_err / max(threshold, 1e-8), 0, 1))},
                        "stage7":  {"final_risk": float(np.clip(max_err / max(threshold, 1e-8), 0, 1)), "status": "STREAM"},
                        "stage3":  {}, "stage4":  {},
                        "threshold":        threshold,
                        "video_path":       source,
                        "output_directory": "",
                    }
                    alert = alert_sys.evaluate_and_fire(mini_results)
                    if alert and on_alert:
                        try:
                            on_alert(alert)
                        except Exception as e:
                            log.error(f"[RealTimeStream] on_alert failed: {e}")
                half   = window_frames // 2
                buffer = buffer[-half:]
                scores = scores[-(half // cfg.AE_FRAME_STRIDE):]
        cap.release()
        log.info("[RealTimeStream] Stream ended.")


# ══════════════════════════════════════════════════════════════════════════════
# PDF REPORT GENERATOR
# ══════════════════════════════════════════════════════════════════════════════
class EnhancedReportGenerator:
    """
    PDF Layout (up to 11 pages):
      Page  1 : Cover + Executive Summary
      Page  2 : Stage 0 (AE) temporal signal graph
      Page  3 : Stage 1 (TF Binary)
      Page  4 : Stage 2 (TF Multi-class)
      Page  5 : Stage 3 (Weapon Detection, EXPERIMENTAL)
      Page  6 : Stage 4 (Audio)
      Page  7 : Stage 5 (Tracking)
      Page  8 : Stage 6 (Fusion)
      Page  9 : Stage 7 (RL3033, EXPERIMENTAL)
      Page 10 : Top anomalous frames
      Page 11 : Stage 8 (LLaVA, EXPERIMENTAL)
    """

    DISCLAIMER = (
        "This report is produced by machine learning models and should be treated "
        "as an investigative aid, not a definitive conclusion. Human expert review "
        "is strongly recommended before any operational action is taken."
    )

    C_DARK   = "#1a252f"
    C_HEADER = "#2c3e50"
    C_RED    = "#c0392b"
    C_GREEN  = "#1e8449"
    C_BLUE   = "#2471a3"
    C_ORANGE = "#d35400"
    C_PURPLE = "#6c3483"
    C_TEAL   = "#0e6655"
    C_GREY   = "#5d6d7e"
    C_LIGHT  = "#f2f3f4"

    HEADER_H = 52
    MARGIN_X = 48

    @staticmethod
    def _body_top(H: float) -> float:
        return H - EnhancedReportGenerator.HEADER_H - 16

    @staticmethod
    def _page_header(c, title: str, bg_hex: str,
                     W: float = letter[0], H: float = letter[1]):
        c.setFillColor(colors.HexColor(bg_hex))
        c.rect(0, H - EnhancedReportGenerator.HEADER_H, W,
               EnhancedReportGenerator.HEADER_H, fill=True, stroke=False)
        c.setFillColor(colors.white)
        c.setFont("Helvetica-Bold", 15)
        c.drawCentredString(W / 2,
                            H - EnhancedReportGenerator.HEADER_H + 16,
                            title)

    @staticmethod
    def _footer(c, ts: str, W: float = letter[0]):
        c.setStrokeColor(colors.HexColor("#aab7b8"))
        c.setLineWidth(0.5)
        c.line(40, 28, W - 40, 28)
        c.setFont("Helvetica-Oblique", 7)
        c.setFillColor(colors.HexColor("#7f8c8d"))
        c.drawCentredString(W / 2, 14,
            f"Scene Solver v3  |  {ts}  |  Confidential, Investigative Use Only")

    @staticmethod
    def _section_title(c, text: str, y: float, x: float = None,
                       color_hex: str = "#2c3e50",
                       W: float = letter[0]) -> float:
        if x is None:
            x = EnhancedReportGenerator.MARGIN_X
        c.setFillColor(colors.HexColor(color_hex))
        c.rect(x, y - 2, 4, 14, fill=True, stroke=False)
        c.setFillColor(colors.HexColor(color_hex))
        c.setFont("Helvetica-Bold", 11)
        c.drawString(x + 8, y, text)
        c.setFillColor(colors.black)
        return y - 20

    @staticmethod
    def _kv_row(c, key: str, value: str, x: float, y: float,
                key_w: float = 170, line_h: float = 14,
                font_size: int = 9) -> float:
        c.setFont("Helvetica-Bold", font_size)
        c.setFillColor(colors.HexColor("#2c3e50"))
        c.drawString(x, y, key + ":")
        c.setFont("Helvetica", font_size)
        c.setFillColor(colors.black)
        c.drawString(x + key_w, y, value)
        return y - line_h

    @staticmethod
    def _wrap_text(c, text: str, x: float, y: float,
                   max_width: float, font: str = "Helvetica",
                   size: int = 9, line_h: int = 12,
                   min_y: float = 50) -> float:
        c.setFont(font, size)
        c.setFillColor(colors.black)
        words, line = text.split(), ""
        for w in words:
            test = line + w + " "
            if c.stringWidth(test, font, size) < max_width:
                line = test
            else:
                if y < min_y:
                    break
                c.drawString(x, y, line.rstrip())
                y -= line_h
                line = w + " "
        if line.strip() and y >= min_y:
            c.drawString(x, y, line.rstrip())
            y -= line_h
        return y

    @staticmethod
    def _disclaimer_box(c, y: float, W: float = letter[0]) -> float:
        MX    = EnhancedReportGenerator.MARGIN_X
        box_w = W - 2 * MX
        c.setFillColor(colors.HexColor("#fdfefe"))
        c.setStrokeColor(colors.HexColor("#aab7b8"))
        c.setLineWidth(0.6)
        c.roundRect(MX, y - 28, box_w, 32, 4, fill=True, stroke=True)
        c.setFont("Helvetica-Oblique", 7.5)
        c.setFillColor(colors.HexColor("#626567"))
        y_text = y - 9
        y_text = EnhancedReportGenerator._wrap_text(
            c, EnhancedReportGenerator.DISCLAIMER,
            MX + 6, y_text, box_w - 12, "Helvetica-Oblique", 7.5, 10)
        return y - 40

    # ── PDF-BADGE: full body-width badge, parentheses text ─────────────────
    @staticmethod
    def _status_badge(c, text: str, y: float, color_hex: str,
                      W: float = letter[0],
                      MX: float = None) -> float:
        if MX is None:
            MX = EnhancedReportGenerator.MARGIN_X
        bw = W - 2 * MX
        bh = 38
        br = 8
        bx = MX
        c.setFillColor(colors.HexColor(color_hex))
        c.roundRect(bx, y - bh, bw, bh, br, fill=True, stroke=False)
        c.setFillColor(colors.white)
        c.setFont("Helvetica-Bold", 14)
        c.drawCentredString(W / 2, y - bh + 12, text)
        return y - bh - 10

    @staticmethod
    def _experimental_warning_box(c, text: str, y: float,
                                   W: float = letter[0],
                                   MX: float = None) -> float:
        """
        PDF-EXPBOX3: Experimental disclaimer box.
        - Text flows as a single continuous paragraph with NO internal line breaks.
        - Box height is computed dynamically from actual wrapped line count.
        - No leading blank line. Text is fully contained inside the box.
        - Orange border and warm background consistent with experimental branding.
        """
        if MX is None:
            MX = EnhancedReportGenerator.MARGIN_X
        box_w = W - 2 * MX
        font, size, line_h = "Helvetica-Bold", 8.5, 12

        # Normalise the input text to a single paragraph (strip internal newlines)
        # so the box always renders as flowing prose, never fragmented lines.
        normalised_text = " ".join(text.split())

        # Pre-measure lines using a temporary canvas to compute box height
        import io
        from reportlab.pdfgen.canvas import Canvas as _TmpCanvas
        tmp_buf = io.BytesIO()
        tmp_c   = _TmpCanvas(tmp_buf)
        tmp_c.setFont(font, size)
        words    = normalised_text.split()
        line_buf = ""
        lines    = []
        for w in words:
            test = line_buf + w + " "
            if tmp_c.stringWidth(test, font, size) < box_w - 16:
                line_buf = test
            else:
                lines.append(line_buf.rstrip())
                line_buf = w + " "
        if line_buf.strip():
            lines.append(line_buf.rstrip())
        n_lines = max(len(lines), 1)
        # 8 px top padding + 8 px bottom padding
        box_h = n_lines * line_h + 16

        # Draw box
        c.setFillColor(colors.HexColor("#fef9e7"))
        c.setStrokeColor(colors.HexColor("#d35400"))
        c.setLineWidth(1.2)
        c.roundRect(MX, y - box_h, box_w, box_h, 5, fill=True, stroke=True)

        # Render text lines inside the box starting just below the top edge
        c.setFont(font, size)
        c.setFillColor(colors.HexColor("#d35400"))
        text_y = y - 8 - size  # align first baseline flush inside top padding, no gap
        for ln in lines:
            c.drawString(MX + 8, text_y, ln)
            text_y -= line_h

        c.setFillColor(colors.black)
        return y - box_h - 8

    # ── Graph generators ───────────────────────────────────────────────────
    @staticmethod
    def generate_deviation_graph(scores, threshold, peak_score_indices,
                                  sampled_indices, fps,
                                  output_path="/tmp/deviation_graph.png"):
        fig, ax = plt.subplots(figsize=(13, 4.2))
        ts_labels = [frame_to_timestamp(
                         sampled_indices[i] if i < len(sampled_indices) else i * 5, fps)
                     for i in range(len(scores))]
        x = list(range(len(scores)))
        ax.plot(x, scores, linewidth=2.2, color="#c0392b",
                label="Reconstruction Error", zorder=2)
        ax.axhline(y=threshold, color="#1e8449", linestyle="--",
                   label=f"Dynamic Threshold ({threshold:.5f})", linewidth=1.8, zorder=1)
        valid_peaks = [i for i in peak_score_indices if i < len(scores)]
        if valid_peaks:
            ax.scatter(valid_peaks, [scores[i] for i in valid_peaks],
                       color="#f39c12", s=220, marker="*",
                       edgecolors="#333", linewidths=1.2,
                       label="Peak Anomaly", zorder=3)
        ax.fill_between(x, scores, threshold,
                        where=[s > threshold for s in scores],
                        color="#c0392b", alpha=0.15)
        n = len(scores)
        tick_step = max(1, n // 8)
        tick_pos  = list(range(0, n, tick_step))
        ax.set_xticks(tick_pos)
        ax.set_xticklabels([ts_labels[i] for i in tick_pos], rotation=25,
                           fontsize=7.5, ha="right")
        ax.set_xlabel("Timestamp (MM:SS.mmm)", fontsize=10, fontweight="bold")
        ax.set_ylabel("Reconstruction Error", fontsize=10, fontweight="bold")
        ax.set_title("Autoencoder Temporal Reconstruction Error", fontsize=12,
                     fontweight="bold", pad=10)
        ax.legend(loc="upper right", fontsize=9, framealpha=0.92)
        ax.grid(True, alpha=0.25, linestyle=":", linewidth=0.8)
        fig.tight_layout()
        fig.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        return output_path

    @staticmethod
    def generate_classification_pie(probabilities, output_path="/tmp/class_pie.png"):
        """
        PDF-PIE-LABELS: Percentage labels removed from the pie wedges to prevent
        overlapping. Users are directed to the numbered legend panel for class names
        and exact percentages. Numbers 1-7 remain on wedges for cross-reference.
        """
        items  = sorted(probabilities.items(), key=lambda x: x[1], reverse=True)
        labels = [i[0] for i in items]
        sizes  = [max(i[1], 1e-6) for i in items]
        clist  = ["#c0392b", "#2471a3", "#d35400", "#6c3483",
                  "#117a65", "#1a5276", "#7d6608"]
        fig, (ax_pie, ax_leg) = plt.subplots(1, 2, figsize=(10, 5),
                                              gridspec_kw={"width_ratios": [1.3, 1]})
        wedges, texts = ax_pie.pie(
            sizes,
            labels=[str(i + 1) for i in range(len(sizes))],
            colors=clist[:len(sizes)],
            startangle=90,
            textprops={"fontsize": 10, "weight": "bold"},
            explode=[0.04 if i == 0 else 0 for i in range(len(sizes))],
            labeldistance=1.10,
        )
        ax_pie.set_title("Anomaly Class Distribution\n(see legend for percentages)",
                         fontsize=10, fontweight="bold", pad=8)
        ax_leg.axis("off")
        for i, (cls, prob) in enumerate(zip(labels, sizes)):
            rect = plt.Rectangle((0, 0.85 - i * 0.12), 0.12, 0.09,
                                  color=clist[i % len(clist)])
            ax_leg.add_patch(rect)
            ax_leg.text(0.16, 0.855 - i * 0.12 + 0.01,
                        f"{i+1}. {cls}  ({prob*100:.1f}%)",
                        va="center", fontsize=8.5, fontweight="bold",
                        transform=ax_leg.transAxes)
        ax_leg.set_xlim(0, 1); ax_leg.set_ylim(0, 1)
        fig.tight_layout()
        fig.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        return output_path

    @staticmethod
    def generate_confidence_bars(top3, output_path="/tmp/conf_bars.png"):
        if not top3:
            return None
        classes = [t[0] for t in top3]
        confs   = [t[1] * 100 for t in top3]
        fig, ax = plt.subplots(figsize=(7, 3.2))
        bar_colors = ["#c0392b", "#d35400", "#5d6d7e"]
        bars = ax.barh(classes, confs, color=bar_colors[:len(top3)],
                       edgecolor="#2c3e50", linewidth=0.8, height=0.5)
        for bar, cv in zip(bars, confs):
            ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height() / 2,
                    f"{cv:.1f}%", va="center", fontsize=9, fontweight="bold",
                    color="#2c3e50")
        ax.set_xlabel("Confidence (%)", fontsize=9, fontweight="bold")
        ax.set_title("Top-3 Anomaly Predictions", fontsize=10, fontweight="bold", pad=8)
        ax.set_xlim(0, 118)
        ax.grid(axis="x", alpha=0.25, linestyle="--", linewidth=0.7)
        ax.invert_yaxis()
        fig.tight_layout()
        fig.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        return output_path

    @staticmethod
    def extract_top_frames(frames, scores, sampled_indices, n=3):
        if not scores or not frames:
            return []
        top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        out = []
        for si in top[:n]:
            fi = sampled_indices[si] if si < len(sampled_indices) else si * 5
            if fi < len(frames):
                out.append({"frame": frames[fi], "index": fi, "score": scores[si]})
        return out

    # ── Main PDF generator ─────────────────────────────────────────────────
    @staticmethod
    def generate_enhanced_pdf(results: dict,
                               output_path: str = "/tmp/forensic_report.pdf") -> str:
        c    = rl_canvas.Canvas(output_path, pagesize=letter)
        W, H = letter
        ts   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        G    = EnhancedReportGenerator
        MX   = G.MARGIN_X
        BODY_W = W - 2 * MX

        s0 = results.get("stage0") or {}
        s1 = results.get("stage1") or {}
        s2 = results.get("stage2") or {}
        s3 = results.get("stage3") or {}
        s4 = results.get("stage4") or {}
        s5 = results.get("stage5") or {}
        s6 = results.get("stage6") or {}
        s7 = results.get("stage7") or {}
        s8 = results.get("stage8") or {}
        al = results.get("alert")

        fps       = float(s0.get("fps", 25.0))
        threshold = results.get("threshold", Config.SAFE_THRESHOLD_LEGACY)
        overall   = results.get("status", "SAFE")
        is_crit   = (overall == "CRITICAL")

        # Check Stage 1 result for conditional sections
        stage1_normal = (s1.get("classification", "Unknown") == "Normal")

        def ts_of(frame_idx):
            return frame_to_timestamp(int(frame_idx), fps)

        # ═══════════════════════════════════════════════════════════
        # PAGE 1 — COVER + EXECUTIVE SUMMARY
        # ═══════════════════════════════════════════════════════════
        G._page_header(c, "FORENSIC ANALYSIS DOSSIER, Scene Solver v3",
                        G.C_HEADER, W, H)
        y = G._body_top(H)

        c.setFont("Helvetica", 8)
        c.setFillColor(colors.HexColor(G.C_GREY))
        c.drawCentredString(W / 2, y, f"Report generated: {ts}")
        y -= 14

        badge_col  = G.C_RED if is_crit else G.C_GREEN
        badge_text = "ANOMALY DETECTED (CRITICAL)" if is_crit else "NO ANOMALY DETECTED (SAFE)"
        y = G._status_badge(c, badge_text, y, badge_col, W, MX)
        y -= 6

        y = G._section_title(c, "EXECUTIVE SUMMARY", y, MX, G.C_HEADER)

        def _field(label, value, key_w=185):
            nonlocal y
            if y < 60:
                return
            c.setFont("Helvetica-Bold", 9)
            c.setFillColor(colors.HexColor(G.C_DARK))
            c.drawString(MX + 10, y, label + ":")
            c.setFont("Helvetica", 9)
            c.setFillColor(colors.black)
            c.drawString(MX + 10 + key_w, y, str(value))
            y -= 13

        def _spacer(h=5):
            nonlocal y; y -= h

        _field("Overall Status",          overall)
        _field("Dynamic Threshold",
               f"{threshold:.6f}  (mean plus 2 standard deviations)")
        _field("AE Max Reconstruction Error", f"{s0.get('max_error', 0):.6f}")
        _field("AE Avg Reconstruction Error", f"{s0.get('avg_error', 0):.6f}")
        _field("Peak Event Timestamp",    ts_of(s0.get('peak_frame_idx', 0)))
        _field("Anomaly Window",
               f"{ts_of(s0.get('anomaly_start_frame', 0))}  to  "
               f"{ts_of(s0.get('anomaly_end_frame', 0))}")
        _spacer()

        if s1:
            _field("Stage 1 (TF Binary)",
                   f"{s1.get('classification','?')}  ({s1.get('confidence',0):.1%})")
        if s2:
            sm = s2.get("sampling_method", "")
            samp_note = "  [AE-weighted sampling]" if sm == "segmentwise_ae" else ""
            skipped2  = s2.get("class") == "Skipped"
            if skipped2:
                _field("Stage 2 (TF Multi)", "Skipped (Stage 1 = Normal)")
            else:
                _field("Stage 2 (TF Multi)",
                       f"{s2.get('class','?')}  ({s2.get('confidence',0):.1%})"
                       + samp_note)
        _spacer()

        if s3:
            wf = bool(s3.get("weapon_flag", False))
            _field("Stage 3 (Weapon Detection)",
                   "Weapons detected" if wf else "No weapons detected",
                   key_w=210)
            if wf:
                _field("  Dominant Class",    s3.get("dominant_class", "?"), key_w=210)
                _field("  Best Confidence",   f"{s3.get('best_conf', 0):.2%}", key_w=210)
        _spacer()

        if s4:
            audio_ok = s4.get("audio_present", True)
            audio_st = s4.get("status", "?")
            _field("Stage 4 (Audio)",
                   "No audio or silent track, section skipped"
                   if not audio_ok or audio_st in ("NO_AUDIO", "SILENT")
                   else f"{audio_st}  (score = {s4.get('audio_score', 0):.4f})")
        if s5:
            _field("Stage 5 (Tracking)",
                   f"Motion = {s5.get('motion_intensity',0):.3f},  "
                   f"Behaviors: {', '.join(s5.get('behaviors',[])) or 'None'}")
        _spacer()

        if s6:
            _field("Stage 6 (Fusion)",
                   f"{s6.get('fused_risk',0):.1%}  [{s6.get('risk_level','?')}]")
        if s7:
            _field("Stage 7 (RL3033, Experimental)",
                   f"Final risk = {s7.get('final_risk',0):.1%}  [{s7.get('status','?')}]")
        _spacer()

        if al:
            c.setFont("Helvetica-Bold", 9)
            c.setFillColor(colors.HexColor(G.C_RED))
            c.drawString(MX + 10, y,
                f"ALERT FIRED: {al['level']}  |  Final risk = {al['final_risk']:.1%}")
            c.setFillColor(colors.black)
            y -= 13

        # ── Enriched interpretive summary paragraph ───────────────
        _spacer(8)
        y = G._section_title(c, "PIPELINE INTERPRETATION", y, MX, G.C_HEADER)
        risk_val    = float(s7.get("final_risk", s6.get("fused_risk", 0.0)))
        risk_label  = s6.get("risk_level", "LOW")
        cls1_label  = s1.get("classification", "Unknown")
        cls1_conf   = float(s1.get("confidence", 0.0))
        cls2_label  = s2.get("class", "Unknown")
        cls2_conf   = float(s2.get("confidence", 0.0))
        weapon_flag = bool(s3.get("weapon_flag", False))
        audio_ok    = s4.get("audio_present", True)
        audio_st    = s4.get("status", "NO_AUDIO")
        motion_val  = float(s5.get("motion_intensity", 0.0))
        behaviors   = s5.get("behaviors", [])

        interp_sentences = []
        interp_sentences.append(
            f"The pipeline classified this video as {overall} with a final risk score "
            f"of {risk_val:.1%} ({risk_label})."
        )
        if cls1_label == "Suspicious":
            interp_sentences.append(
                f"The semantic binary classifier (Stage 1) identified the clip as "
                f"Suspicious at {cls1_conf:.0%} confidence, activating downstream "
                f"anomaly classification and weapon scanning."
            )
            if cls2_label not in ("Unknown", "Skipped", "Error"):
                interp_sentences.append(
                    f"Stage 2 classified the event type as {cls2_label} "
                    f"({cls2_conf:.0%} confidence) using AE-weighted temporal sampling."
                )
        else:
            interp_sentences.append(
                f"The semantic binary classifier returned Normal ({cls1_conf:.0%} "
                f"confidence), overriding the AE signal and suppressing anomaly "
                f"sub-classification and weapon scanning."
            )
        if weapon_flag:
            interp_sentences.append(
                f"A weapon ({s3.get('dominant_class','?')}) was detected at "
                f"{s3.get('best_conf', 0.0):.0%} confidence, triggering a hard "
                f"risk escalation in the Fusion layer."
            )
        if audio_ok and audio_st not in ("NO_AUDIO", "SILENT", "NORMAL"):
            interp_sentences.append(
                f"Audio analysis reported {audio_st} status "
                f"(score {s4.get('audio_score',0):.3f}), contributing to the "
                f"composite risk calculation."
            )
        if "FAST_MOVEMENT" in behaviors:
            interp_sentences.append(
                f"Motion tracking recorded fast subject movement "
                f"(intensity {motion_val:.2f}), a behavioural indicator that "
                f"boosted the fused risk score."
            )
        elif "CLUSTERING" in behaviors:
            interp_sentences.append(
                f"Motion tracking detected converging subject trajectories "
                f"(intensity {motion_val:.2f}), consistent with a confrontational approach pattern."
            )

        interp_text = " ".join(interp_sentences)
        y = G._wrap_text(c, interp_text, MX + 6, y, BODY_W - 12, "Helvetica", 8.5, 12)
        _spacer(8)

        y = G._disclaimer_box(c, y, W)
        G._footer(c, ts, W)

        # ═══════════════════════════════════════════════════════════
        # PAGE 2 — STAGE 0 (AE)
        # PDF-AE-NORMAL: suppress anomaly detail when Stage 1 = Normal
        # ═══════════════════════════════════════════════════════════
        c.showPage()
        G._page_header(c, "STAGE 0 (AUTOENCODER): TEMPORAL ANOMALY DETECTION",
                        G.C_RED, W, H)
        y = G._body_top(H)
        c.setFont("Helvetica-Oblique", 8.5)
        c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "The Autoencoder (AE) is trained exclusively on normal footage. It learns "
            "to reconstruct normal frames accurately. When an unusual event occurs the "
            "AE cannot reconstruct the frame well, producing a high reconstruction "
            "error. This error is the primary anomaly indicator that drives all "
            "downstream stages. The threshold is computed dynamically per video as the "
            "mean error plus two standard deviations.",
            MX, y, BODY_W, "Helvetica-Oblique", 8.5, 12)
        y -= 8

        gp = results.get("graph_path")
        if gp and os.path.exists(gp):
            img_h = 200
            c.drawImage(gp, MX, y - img_h, width=BODY_W, height=img_h,
                        preserveAspectRatio=True)
            y -= img_h + 10

        # PDF-AE-NORMAL: if Stage 1 = Normal, show a concise safe note instead of
        # the full anomaly breakdown.
        if stage1_normal:
            y = G._section_title(c, "AE Score Summary", y, MX, G.C_GREEN)
            y = G._wrap_text(c,
                "The binary TimeSformer (Stage 1) classified this video as NORMAL. "
                "The AE reconstruction error is shown in the graph above for reference, "
                "but it does not contribute to an anomaly determination in this run. "
                "Downstream anomaly-specific stages (multi-class classification, weapon "
                "detection) were skipped. The overall pipeline status is SAFE.",
                MX + 6, y, BODY_W - 12, "Helvetica", 9, 13)
            y -= 8
            pairs_safe = [
                ("Max Reconstruction Error",  f"{s0.get('max_error',0):.6f}"),
                ("Avg Reconstruction Error",  f"{s0.get('avg_error',0):.6f}"),
                ("Dynamic Threshold",         f"{threshold:.6f}"),
                ("Threshold Exceeded by AE",
                 "YES" if s0.get('max_error', 0) > threshold else "NO"),
                ("Note",
                 "AE signal overridden to SAFE by Stage 1 semantic classifier"),
                ("Total Sampled Frames",      str(len(s0.get("scores", [])))),
                ("Video FPS",                 f"{fps:.1f}"),
                ("Pipeline Status",           "SAFE (Stage 1 override)"),
            ]
            for k, v in pairs_safe:
                y = G._kv_row(c, k, v, MX + 6, y, key_w=220)
                if y < 60: break
        else:
            y = G._section_title(c, "AE Score Summary", y, MX, G.C_RED)
            ae_over_thresh = s0.get('max_error', 0) > threshold
            pairs = [
                ("Max Reconstruction Error",  f"{s0.get('max_error',0):.6f}"),
                ("Avg Reconstruction Error",  f"{s0.get('avg_error',0):.6f}"),
                ("Dynamic Threshold",
                 f"{threshold:.6f}  (mean + 2 std dev, floor = {Config.SAFE_THRESHOLD_FLOOR})"),
                ("Threshold Exceeded",
                 "YES" if ae_over_thresh else "NO"),
                ("Excess Above Threshold",
                 f"{s0.get('max_error', 0) - threshold:.6f}"
                 if ae_over_thresh else "N/A"),
                ("Peak Event Timestamp",      ts_of(s0.get('peak_frame_idx', 0))),
                ("Anomaly Start Timestamp",   ts_of(s0.get('anomaly_start_frame', 0))),
                ("Anomaly End Timestamp",     ts_of(s0.get('anomaly_end_frame', 0))),
                ("Anomaly Duration (frames)",
                 str(s0.get('anomaly_end_frame', 0) - s0.get('anomaly_start_frame', 0))),
                ("Total Sampled Frames",      str(len(s0.get("scores", [])))),
                ("Sampling Stride",           f"Every {Config.AE_FRAME_STRIDE} frames"),
                ("Video FPS",                 f"{fps:.1f}"),
            ]
            for k, v in pairs:
                y = G._kv_row(c, k, v, MX + 6, y, key_w=220)
                if y < 60: break

            # Interpretive note on AE signal quality
            if y > 100:
                y -= 6
                y = G._section_title(c, "Signal Interpretation", y, MX, G.C_RED)
                ratio = s0.get('max_error', 0) / max(threshold, 1e-8)
                if ratio >= 2.0:
                    ae_interp = (
                        f"The peak reconstruction error is {ratio:.1f}x the dynamic "
                        f"threshold, indicating a strong and unambiguous anomaly signal. "
                        f"High-magnitude exceedances like this are rarely produced by "
                        f"camera artefacts or scene transitions alone."
                    )
                elif ratio >= 1.0:
                    ae_interp = (
                        f"The peak reconstruction error exceeds the threshold by a "
                        f"factor of {ratio:.2f}x. This is a meaningful signal but "
                        f"may benefit from corroboration from downstream stages "
                        f"(audio, tracking, semantic classification)."
                    )
                else:
                    ae_interp = (
                        f"The reconstruction error remains below the dynamic threshold "
                        f"(ratio {ratio:.3f}). The AE alone does not flag an anomaly, "
                        f"though downstream semantic stages may still identify concerns."
                    )
                y = G._wrap_text(c, ae_interp, MX + 6, y, BODY_W - 12, "Helvetica", 9, 13)

        y -= 6
        y = G._disclaimer_box(c, y, W)
        G._footer(c, ts, W)

        # ═══════════════════════════════════════════════════════════
        # PAGE 3 — STAGE 1 (TF BINARY)
        # ═══════════════════════════════════════════════════════════
        c.showPage()
        G._page_header(c, "STAGE 1 (TIMESFORMER BINARY): SEMANTIC CLASSIFICATION",
                        G.C_BLUE, W, H)
        y = G._body_top(H)
        c.setFont("Helvetica-Oblique", 8.5)
        c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "The binary TimeSformer is trained on surveillance footage labelled Normal "
            "or Suspicious. It classifies a 32-frame clip centred on the AE peak frame "
            "and outputs a confidence score. When the result is Normal, the overall "
            "pipeline status is overridden to SAFE regardless of the AE signal, because "
            "the semantic model takes priority over reconstruction error alone. "
            "A Normal result also skips Stage 2 (multi-class classification) and "
            "Stage 3 (weapon scanning), conserving compute and reducing false positives.",
            MX, y, BODY_W, "Helvetica-Oblique", 8.5, 12)
        y -= 12

        cls1  = s1.get("classification", "?")
        conf1 = s1.get("confidence", 0.0)
        badge_col1 = G.C_RED if cls1 == "Suspicious" else G.C_GREEN
        y = G._status_badge(c, f"{cls1.upper()}  ({conf1:.1%})", y, badge_col1, W, MX)
        y -= 10

        y = G._section_title(c, "Classification Details", y, MX, G.C_BLUE)
        kv_pairs = [
            ("Result",            cls1),
            ("Confidence",        f"{conf1:.2%}"),
            ("Clip Centre",       ts_of(s0.get('peak_frame_idx', 0))),
            ("Clip Window",       "Peak timestamp, plus or minus 16 frames (32 frames total)"),
            ("Sampling Strategy", "Fixed uniform interval (unchanged from training)"),
            ("Gates YOLO Stage 3",
             "YES, weapon scan activated" if cls1 == "Suspicious" else "NO, skipped"),
            ("Gates Stage 2",
             "YES, multi-class classification runs" if cls1 == "Suspicious"
             else "NO, skipped (Stage 1 = Normal)"),
            ("LLaVA Stage 8",
             "Runs with full context" if cls1 == "Suspicious"
             else "Runs with neutral investigative prompts"),
            ("Overall Status Effect",
             "Follows AE signal" if cls1 == "Suspicious"
             else "Overrides AE — pipeline forced to SAFE"),
        ]
        for k, v in kv_pairs:
            y = G._kv_row(c, k, v, MX + 6, y, key_w=210)
        y -= 8
        y = G._section_title(c, "Interpretation and Operational Significance", y, MX, G.C_BLUE)
        interp = (
            f"The clip centred at {ts_of(s0.get('peak_frame_idx',0))} was classified "
            f"as SUSPICIOUS with {conf1:.1%} confidence. The semantic model has observed "
            "patterns in this clip inconsistent with normal surveillance footage. "
            "Weapon detection and multi-class anomaly classification are now active. "
            "The overall pipeline risk follows the AE signal and downstream fusion output."
            if cls1 == "Suspicious" else
            f"The clip was classified as NORMAL with {conf1:.1%} confidence. "
            "The TimeSformer sees no semantic evidence of an anomalous event in the "
            "32-frame clip centred on the AE peak. This acts as a semantic veto: even "
            "if the AE reconstruction error is elevated, the system defers to the "
            "higher-level semantic understanding. Multi-class classification is skipped. "
            "LLaVA visual analysis runs with neutral investigative prompts to provide "
            "an independent scene description without a suspicious framing bias."
        )
        y = G._wrap_text(c, interp, MX + 6, y, BODY_W - 12, "Helvetica", 9, 13)
        y -= 6
        y = G._disclaimer_box(c, y, W)
        G._footer(c, ts, W)

        # ═══════════════════════════════════════════════════════════
        # PAGE 4 — STAGE 2 (TF MULTI-CLASS)
        # ═══════════════════════════════════════════════════════════
        c.showPage()
        G._page_header(c, "STAGE 2 (TIMESFORMER MULTI-CLASS): ANOMALY TYPE",
                        G.C_PURPLE, W, H)
        y = G._body_top(H)

        cls2      = s2.get("class", "Unknown")
        conf2     = s2.get("confidence", 0.0)
        samp_meth = s2.get("sampling_method", "?")
        stage2_skipped = (cls2 == "Skipped" or not s2.get("all_probabilities"))

        c.setFont("Helvetica-Oblique", 8.5)
        c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "The multi-class TimeSformer classifies the anomaly into one of seven "
            "crime categories. This stage only runs when Stage 1 (TF Binary) "
            "classifies the video as Suspicious. It uses segment-wise AE-weighted "
            "temporal sampling: the video is divided into 16 equal segments and the "
            "frame with the highest AE error in each segment is selected, ensuring "
            "the model focuses on the most anomalous moments rather than a uniform "
            "time window.",
            MX, y, BODY_W, "Helvetica-Oblique", 8.5, 12)
        y -= 8

        if stage2_skipped:
            y = G._status_badge(c, "STAGE 2 SKIPPED (STAGE 1 = NORMAL)", y, G.C_GREY, W, MX)
            y -= 8
            c.setFont("Helvetica-Oblique", 9)
            c.setFillColor(colors.HexColor(G.C_GREY))
            c.drawString(MX + 6, y,
                "Multi-class classification does not run when the binary classifier "
                "returns Normal.")
            y -= 14
            y -= 6
            y = G._section_title(c, "Why This Stage Was Skipped", y, MX, G.C_GREY)
            y = G._wrap_text(c,
                "Stage 1 (TF Binary) classified the clip as Normal, which overrides "
                "the AE anomaly signal and prevents all anomaly-specific downstream "
                "processing. Running multi-class classification on a Normal-labelled "
                "clip would produce unreliable category assignments because the model "
                "was trained to distinguish between anomaly types, not to operate in "
                "the absence of an anomaly. Skipping this stage also avoids producing "
                "false category labels that could mislead an analyst.",
                MX + 6, y, BODY_W - 12, "Helvetica", 9, 13)
        else:
            y = G._status_badge(c, f"{cls2.upper()}  ({conf2:.1%})", y, G.C_PURPLE, W, MX)
            y -= 4
            c.setFont("Helvetica-Oblique", 8)
            c.setFillColor(colors.HexColor(G.C_GREY))
            c.drawCentredString(W / 2, y,
                f"Sampling: {'Segment-wise AE-weighted (16 segments, best per segment)' if samp_meth == 'segmentwise_ae' else samp_meth}")
            y -= 12

            # PDF-PIE-LABELS note to user
            c.setFont("Helvetica-Oblique", 8)
            c.setFillColor(colors.HexColor(G.C_PURPLE))
            c.drawCentredString(W / 2, y,
                "Chart shows class numbers only. Refer to the numbered legend on the right for class names and percentages.")
            y -= 10

            all_probs = s2.get("all_probabilities", {})
            top3      = s2.get("top3", [])
            if all_probs:
                pie_path = "/tmp/cls_pie_v3.png"
                G.generate_classification_pie(all_probs, pie_path)
                if os.path.exists(pie_path):
                    c.drawImage(pie_path, MX, y - 200, width=310, height=200,
                                preserveAspectRatio=True)
            if top3:
                bar_path = "/tmp/conf_bars_v3.png"
                G.generate_confidence_bars(top3, bar_path)
                if os.path.exists(bar_path):
                    c.drawImage(bar_path, MX + 318, y - 150, width=220, height=150,
                                preserveAspectRatio=True)
            y -= 212
            y = G._section_title(c, "Top-3 Predictions", y, MX, G.C_PURPLE)
            for rank, (cls_name, prob) in enumerate(top3[:3], 1):
                c.setFont("Helvetica", 9); c.setFillColor(colors.black)
                c.drawString(MX + 12, y, f"{rank}.  {cls_name}  ({prob:.2%})")
                y -= 13
            y -= 4
            y = G._section_title(c, "All Class Probabilities", y, MX, G.C_PURPLE)
            for cls_name, prob in sorted(all_probs.items(), key=lambda x: x[1], reverse=True):
                c.setFont("Helvetica", 8.5); c.setFillColor(colors.black)
                c.drawString(MX + 12, y,
                             f"{cls_name:<22s}  {prob:.4f}  ({prob * 100:.2f}%)")
                y -= 12
                if y < 70: break

            # Interpretive insight for top class
            if y > 100 and top3:
                y -= 6
                y = G._section_title(c, "Classification Insight", y, MX, G.C_PURPLE)
                top_cls, top_conf = top3[0]
                insight = (
                    f"The model assigned the highest probability to '{top_cls}' "
                    f"({top_conf:.1%}). "
                )
                if top_conf >= 0.60:
                    insight += (
                        "This is a high-confidence prediction, suggesting the visual "
                        "pattern strongly matches the training signature for this category."
                    )
                elif top_conf >= 0.35:
                    insight += (
                        "This is a moderate-confidence prediction. The second and "
                        "third candidates should also be considered by the analyst."
                    )
                else:
                    insight += (
                        "The low confidence indicates the model is uncertain. The "
                        "anomaly pattern may not align cleanly with any trained category, "
                        "or the event may be partially occluded."
                    )
                y = G._wrap_text(c, insight, MX + 6, y, BODY_W - 12, "Helvetica", 9, 13)

        y -= 4
        y = G._disclaimer_box(c, y, W)
        G._footer(c, ts, W)

        # ═══════════════════════════════════════════════════════════
        # PAGE 5 — STAGE 3 (WEAPON DETECTION, EXPERIMENTAL)
        # ═══════════════════════════════════════════════════════════
        c.showPage()
        weapon_flag = bool(s3.get("weapon_flag", False))
        hdr_col3    = G.C_RED if weapon_flag else G.C_TEAL
        G._page_header(c, "STAGE 3 (WEAPON DETECTION, EXPERIMENTAL)", hdr_col3, W, H)
        y = G._body_top(H)

        exp_text3 = (
            "EXPERIMENTAL: This module is under active development. "
            "Detection results should be independently verified before any action is taken."
        )
        y = G._experimental_warning_box(c, exp_text3, y, W, MX)

        c.setFont("Helvetica-Oblique", 8.5)
        c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "A custom-trained YOLOv8 model scans frames within the anomaly window "
            "for two weapon categories: Melee weapons and Firearms. This stage is "
            "only activated when Stage 1 (TF Binary) returns Suspicious. A confirmed "
            "weapon detection triggers a hard escalation in the Fusion layer (Stage 6), "
            "forcing the risk score to a minimum of 0.90 when combined with an AE anomaly. "
            "Up to 32 frames are scanned at evenly spaced intervals within the anomaly window.",
            MX, y, BODY_W, "Helvetica-Oblique", 8.5, 12)
        y -= 8

        if weapon_flag:
            y = G._status_badge(c,
                f"WEAPONS DETECTED ({s3.get('dominant_class','?').upper()})",
                y, G.C_RED, W, MX)
        else:
            y = G._status_badge(c, "NO WEAPONS DETECTED", y, G.C_TEAL, W, MX)
        y -= 8

        skipped = str(s3.get("summary", "")).lower().startswith("skipped")
        if skipped:
            c.setFont("Helvetica-Oblique", 9)
            c.setFillColor(colors.HexColor(G.C_GREY))
            c.drawString(MX + 6, y,
                "This stage was skipped because Stage 1 classified the video as Normal.")
            y -= 20
            y = G._section_title(c, "Operational Note", y, MX, G.C_GREY)
            y = G._wrap_text(c,
                "Weapon scanning is gated behind Stage 1 to reduce false positives "
                "and unnecessary compute. Running YOLO on a Normal-classified clip "
                "would scan frames with no semantic evidence of an incident, increasing "
                "the chance of spurious detections from everyday objects. When Stage 1 "
                "returns Suspicious, weapon scanning activates automatically for the "
                "anomaly window defined by the AE temporal signal.",
                MX + 6, y, BODY_W - 12, "Helvetica", 9, 13)
        else:
            y = G._section_title(c, "Detection Summary", y, MX, hdr_col3)
            det_pairs = [
                ("Weapon Detected",          "YES" if weapon_flag else "NO"),
                ("Dominant Weapon Class",    s3.get("dominant_class", "None")),
                ("Total Detections",         str(s3.get("weapon_count", 0))),
                ("Best Detection Confidence",
                 f"{s3.get('best_conf', 0.0):.2%}" if weapon_flag else "N/A"),
                ("Frames Scanned",           "Up to 32 frames within the anomaly window"),
                ("Anomaly Window",
                 f"{ts_of(s0.get('anomaly_start_frame',0))} to "
                 f"{ts_of(s0.get('anomaly_end_frame',0))}"),
                ("Fusion Impact",
                 "Risk forced to minimum 0.90 (weapon + AE threshold exceeded)"
                 if weapon_flag and s0.get("max_error", 0) > threshold
                 else "Weapon floor applied (0.40 minimum)" if weapon_flag
                 else "No fusion impact from weapon stage"),
            ]
            for k, v in det_pairs:
                y = G._kv_row(c, k, v, MX + 6, y, key_w=210)
            y -= 6
            if weapon_flag:
                y = G._section_title(c, "Individual Detections (top 10)", y, MX, hdr_col3)
                dets = s3.get("all_detections", [])[:10]
                for di, det in enumerate(dets, 1):
                    line = (
                        f"  {di:2d}.  {det.get('class_name','?'):12s}  "
                        f"Confidence = {det.get('confidence',0):.2%},  "
                        f"at {ts_of(det.get('frame_idx', 0))}"
                    )
                    c.setFont("Helvetica", 8.5); c.setFillColor(colors.black)
                    c.drawString(MX + 8, y, line)
                    y -= 12
                    if y < 130: break
                top_frames = s3.get("top_frames_annotated", [])
                if top_frames:
                    y -= 4
                    y = G._section_title(c,
                        "Top Detection Frames (with bounding boxes)", y, MX, hdr_col3)
                    n_show = min(2, len(top_frames))
                    img_w  = (BODY_W - 10) / max(n_show, 1)
                    img_h  = min(140, img_w * 0.6)
                    if y - img_h < 50:
                        img_h = max(y - 60, 60)
                    for fi_idx, (fi, ann_frame) in enumerate(top_frames[:n_show]):
                        tmp_path = f"/tmp/yolo_ann_{fi_idx}.jpg"
                        Image.fromarray(ann_frame).save(tmp_path)
                        cx = MX + fi_idx * (img_w + 10)
                        c.drawImage(tmp_path, cx, y - img_h, width=img_w - 4,
                                    height=img_h, preserveAspectRatio=True)
                        c.setFont("Helvetica", 7.5)
                        c.setFillColor(colors.HexColor(G.C_DARK))
                        c.drawString(cx, y - img_h - 10, f"Timestamp: {ts_of(fi)}")
                    y -= img_h + 22
        y -= 4
        y = G._disclaimer_box(c, y, W)
        G._footer(c, ts, W)

        # ═══════════════════════════════════════════════════════════
        # PAGE 6 — STAGE 4 (AUDIO)
        # PDF-AUDIO-EXPLAIN: expanded plain-English explanations
        # ═══════════════════════════════════════════════════════════
        audio_present = s4.get("audio_present", True)
        audio_status  = s4.get("status", "NO_AUDIO")
        audio_silent  = (not audio_present or audio_status in ("NO_AUDIO", "SILENT"))
        c.showPage()
        aud_col = (G.C_RED    if audio_status == "ANOMALY"    else
                   G.C_ORANGE if audio_status == "SUSPICIOUS" else
                   G.C_GREEN  if audio_status == "NORMAL"     else G.C_GREY)
        G._page_header(c, "STAGE 4 (AUDIO FEATURE EXTRACTION)", aud_col, W, H)
        y = G._body_top(H)
        c.setFont("Helvetica-Oblique", 8.5)
        c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "Five acoustic features are extracted from the video's audio track. "
            "Each is normalised to a 0-to-1 scale using global range constants and "
            "combined into a composite anomaly score via a weighted sum. "
            "The classification rule is: two or more features scoring above 0.4 = ANOMALY; "
            "exactly one above 0.4 = SUSPICIOUS; none above 0.4 = NORMAL. "
            "The composite score feeds into the Fusion layer (Stage 6) and the "
            "RL3033 observation space (Stage 7).",
            MX, y, BODY_W, "Helvetica-Oblique", 8.5, 12)
        y -= 6

        # Plain-English feature glossary
        y = G._section_title(c, "What Each Feature Measures", y, MX, aud_col)
        feature_glossary = [
            ("Zero Crossing Rate (ZCR)",
             "How often the audio waveform crosses zero per second. High values suggest "
             "noisy or chaotic sounds such as shouting, breaking glass, or gunshots. "
             "Quiet ambient speech produces low ZCR; abrupt impact sounds spike it sharply."),
            ("MFCC Variance",
             "Mel-Frequency Cepstral Coefficients capture the tonal texture of sound, "
             "similar to how humans perceive timbre. High variance means the audio "
             "content is rapidly changing, which is unusual for quiet environments. "
             "Screaming, rapid voice changes, or sudden silence all elevate MFCC variance."),
            ("Spectral Flux",
             "Measures how quickly the frequency content of the audio changes over time. "
             "Sudden loud events such as explosions, impacts, or gunshots produce sharp "
             "spikes in spectral flux. Steady-state sounds like background hum remain low."),
            ("RMS Energy",
             "Root Mean Square energy reflects overall loudness. A sudden surge "
             "indicates a loud, potentially alarming sound event. Sustained high "
             "RMS with no baseline context is also a strong anomaly indicator."),
            ("Spectral Centroid",
             "The centre of gravity of the frequency spectrum. Higher values indicate "
             "brighter, higher-pitched sounds. Dramatic shifts can signal unusual events "
             "such as alarms, screaming, or high-pitched metal impacts."),
        ]
        for fname, fdesc in feature_glossary:
            if y < 90:
                break
            c.setFont("Helvetica-Bold", 8.5)
            c.setFillColor(colors.HexColor(G.C_DARK))
            c.drawString(MX + 8, y, fname)
            y -= 12
            y = G._wrap_text(c, fdesc, MX + 16, y, BODY_W - 24, "Helvetica", 8, 11)
            y -= 4

        y -= 4
        if audio_silent:
            c.setFillColor(colors.HexColor("#eaeded"))
            c.roundRect(MX, y - 44, BODY_W, 48, 6, fill=True, stroke=False)
            c.setFillColor(colors.HexColor(G.C_GREY))
            c.setFont("Helvetica-Bold", 12)
            c.drawCentredString(W / 2, y - 20, "NO AUDIO TRACK OR SILENT TRACK")
            c.setFont("Helvetica", 9)
            c.drawCentredString(W / 2, y - 34,
                "The video has no audio or an entirely silent track.")
            y -= 60
        else:
            y = G._status_badge(c, f"AUDIO STATUS: {audio_status}", y, aud_col, W, MX)
            y -= 8
            y = G._section_title(c, "Raw Feature Values", y, MX, aud_col)
            aud_pairs = [
                ("Audio Composite Score",    f"{s4.get('audio_score',0):.4f}"),
                ("Zero Crossing Rate (ZCR)", f"{s4.get('zcr',0):.6f}"),
                ("MFCC Variance",            f"{s4.get('mfcc_variance',0):.4f}"),
                ("Spectral Flux",            f"{s4.get('spectral_flux',0):.4f}"),
                ("RMS Energy",               f"{s4.get('rms_energy',0):.6f}"),
                ("Spectral Centroid",        f"{s4.get('spectral_centroid',0):.2f} Hz"),
            ]
            for k, v in aud_pairs:
                y = G._kv_row(c, k, v, MX + 6, y, key_w=200)
            norm = s4.get("normalized", {})
            if norm:
                y -= 4
                y = G._section_title(c, "Globally Normalised Values (0 to 1 scale, threshold = 0.4)",
                                     y, MX, aud_col)
                flagged = []
                for k, v, key in [
                    ("ZCR (normalised)",               f"{norm.get('zcr',0):.4f}", "zcr"),
                    ("MFCC Variance (normalised)",     f"{norm.get('mfcc',0):.4f}", "mfcc"),
                    ("Spectral Flux (normalised)",     f"{norm.get('flux',0):.4f}", "flux"),
                    ("RMS Energy (normalised)",        f"{norm.get('rms',0):.4f}", "rms"),
                    ("Spectral Centroid (normalised)", f"{norm.get('sc',0):.4f}", "sc"),
                ]:
                    flag = "  [ABOVE THRESHOLD]" if norm.get(key.replace("mfcc","mfcc"), 0) > 0.4 else ""
                    if flag:
                        flagged.append(k.split(" (")[0])
                    y = G._kv_row(c, k, v + flag, MX + 6, y, key_w=230)
                if flagged and y > 90:
                    y -= 4
                    flag_note = (
                        f"Features above threshold: {', '.join(flagged)}. "
                        f"The classification rule requires 2+ features above 0.4 for ANOMALY, "
                        f"1 for SUSPICIOUS, and 0 for NORMAL."
                    )
                    y = G._wrap_text(c, flag_note, MX + 6, y, BODY_W - 12,
                                     "Helvetica-Oblique", 8, 11)
        y -= 6
        y = G._disclaimer_box(c, y, W)
        G._footer(c, ts, W)

        # ═══════════════════════════════════════════════════════════
        # PAGE 7 — STAGE 5 (TRACKING)
        # ═══════════════════════════════════════════════════════════
        c.showPage()
        G._page_header(c, "STAGE 5 (TRACKING AND MOTION BEHAVIOR)", G.C_TEAL, W, H)
        y = G._body_top(H)
        c.setFont("Helvetica-Oblique", 8.5)
        c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "Multi-object tracking records pixel trajectories in the anomaly window. "
            "Three behavioural patterns are detected: FAST_MOVEMENT (rapid motion), "
            "STATIC (stationary), and CLUSTERING (two subjects converging). "
            "These patterns feed into the Fusion layer and the RL3033 observation space. "
            "When YOLOv8 tracking is available, per-object trajectories are computed. "
            "Otherwise, dense optical flow (Farneback method) is used as a fallback.",
            MX, y, BODY_W, "Helvetica-Oblique", 8.5, 12)
        y -= 10
        motion_int   = float(s5.get("motion_intensity", 0.0))
        behaviors    = s5.get("behaviors", [])
        track_count  = s5.get("track_count", 0)
        track_status = s5.get("status", "?")
        bar_bg_w = BODY_W
        c.setFillColor(colors.HexColor("#d5d8dc"))
        c.rect(MX, y - 16, bar_bg_w, 18, fill=True, stroke=False)
        fill_w  = int(bar_bg_w * min(motion_int, 1.0))
        bar_col = (G.C_RED if motion_int > 0.6 else
                   G.C_ORANGE if motion_int > 0.3 else G.C_GREEN)
        c.setFillColor(colors.HexColor(bar_col))
        c.rect(MX, y - 16, fill_w, 18, fill=True, stroke=False)
        c.setFillColor(colors.white if motion_int > 0.15 else colors.HexColor(G.C_DARK))
        c.setFont("Helvetica-Bold", 9)
        c.drawString(MX + 6, y - 9,
                     f"Motion Intensity: {motion_int:.3f}  ({motion_int*100:.1f}%)")
        y -= 28
        y = G._section_title(c, "Tracking Summary", y, MX, G.C_TEAL)
        for k, v in [
            ("Tracking Method",
             track_status + (" (per-object trajectories)" if track_status == "YOLO_TRACK"
                             else " (dense pixel flow)" if "OPTICAL" in track_status
                             else "")),
            ("Tracked Objects",     str(track_count) if track_count else "N/A (optical flow)"),
            ("Motion Intensity",    f"{motion_int:.4f}  (0 = static, 1 = maximum displacement)"),
            ("Detected Behaviours", ", ".join(behaviors) if behaviors else "None"),
            ("Fusion Contribution",
             "FAST_MOVEMENT adds 0.075 to fused risk; CLUSTERING adds 0.075"
             if any(b in behaviors for b in ("FAST_MOVEMENT", "CLUSTERING"))
             else "No behavioural boost applied to fusion risk"),
        ]:
            y = G._kv_row(c, k, v, MX + 6, y, key_w=210)
        y -= 8
        y = G._section_title(c, "Behaviour Definitions", y, MX, G.C_TEAL)
        for bname, bdesc in [
            ("FAST_MOVEMENT",
             "Average per-frame displacement exceeds 8 pixels. Indicates subjects "
             "running, lunging, or otherwise moving rapidly within the scene."),
            ("STATIC",
             "Average displacement below 2 pixels over at least 10 tracked positions. "
             "Subjects are essentially stationary; may indicate surveillance of a scene "
             "or a post-event lull."),
            ("CLUSTERING",
             "Two subjects converge: distance decreases in over 75% of frames by at "
             "least 50 pixels. This trajectory pattern is associated with confrontational "
             "proximity, pursuit, or physical engagement."),
        ]:
            c.setFont("Helvetica-Bold", 9)
            c.setFillColor(colors.HexColor(G.C_DARK))
            c.drawString(MX + 8, y, bname)
            y -= 12
            y = G._wrap_text(c, bdesc, MX + 16, y, BODY_W - 24, "Helvetica", 8.5, 12)
            y -= 4
        y -= 4
        y = G._disclaimer_box(c, y, W)
        G._footer(c, ts, W)

        # ═══════════════════════════════════════════════════════════
        # PAGE 8 — STAGE 6 (FUSION)
        # ═══════════════════════════════════════════════════════════
        c.showPage()
        fused_risk = float(s6.get("fused_risk", 0.0))
        risk_level = s6.get("risk_level", "?")
        fus_col    = (G.C_RED    if risk_level == "CRITICAL" else
                      G.C_ORANGE if risk_level in ("HIGH", "MODERATE") else G.C_GREEN)
        G._page_header(c, "STAGE 6 (FUSION LAYER): MULTI-MODAL REASONING", fus_col, W, H)
        y = G._body_top(H)
        c.setFont("Helvetica-Oblique", 8.5)
        c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "The Fusion layer combines all preceding stage outputs into a single risk "
            "score using explicit rule-based reasoning. Each step is recorded in plain "
            "language below, making this an explainable AI (xAI) decision layer. "
            "The base weight for the AE score is 50%. Semantic classification, motion, "
            "audio, and weapon signals apply additive boosts or penalties on top of this "
            "base. The final fused risk is then passed to Stage 7 (RL3033) for "
            "temporal blending.",
            MX, y, BODY_W, "Helvetica-Oblique", 8.5, 12)
        y -= 8
        y = G._status_badge(c, f"{risk_level}, Fused Risk = {fused_risk:.2%}",
                            y, fus_col, W, MX)
        y -= 10
        y = G._section_title(c, "Input Signal Values", y, MX, fus_col)
        for k, v in [
            ("AE Score (normalised)",
             f"{s6.get('ae_score_norm', 0):.4f}  (50% base weight)"),
            ("Binary TF Confidence",
             f"{s6.get('ts_binary_conf', 0):.4f}  (20% weight if Suspicious)"),
            ("Audio Score",
             f"{s6.get('audio_score', 0):.4f}  (10% boost if ANOMALY)"),
            ("Weapon Detected",
             "YES  (minimum 0.40 floor; 0.90 floor if AE also anomalous)"
             if s6.get("weapon_flag") else "NO"),
            ("Motion Intensity",
             f"{s6.get('motion_intensity', 0):.4f}  (15% boost if FAST_MOVEMENT + audio)"),
        ]:
            y = G._kv_row(c, k, v, MX + 6, y, key_w=230)
        y -= 8
        y = G._section_title(c, "Step-by-Step Reasoning", y, MX, fus_col)
        c.setFont("Helvetica", 8.5); c.setFillColor(colors.black)
        for i, line in enumerate(s6.get("reasoning_trace", []), 1):
            y = G._wrap_text(c, f"Step {i}:  {line}", MX + 8, y,
                             BODY_W - 16, "Helvetica", 8.5, 12)
            y -= 4
            if y < 80: break
        y -= 4
        y = G._disclaimer_box(c, y, W)
        G._footer(c, ts, W)

        # ═══════════════════════════════════════════════════════════
        # PAGE 9 — STAGE 7 (RL3033, EXPERIMENTAL)
        # PDF-RL-EXPLAIN: plain-English explanations for all technical terms
        # ═══════════════════════════════════════════════════════════
        c.showPage()
        final_risk = float(s7.get("final_risk", 0.0))
        rl_status  = s7.get("status", "?")
        rl_col     = (G.C_RED    if rl_status in ("CRITICAL", "ERROR") else
                      G.C_ORANGE if rl_status in ("HIGH_RISK", "MODERATE") else G.C_GREEN)
        G._page_header(c, "STAGE 7 (RL3033, EXPERIMENTAL): TEMPORAL REASONING",
                        rl_col, W, H)
        y = G._body_top(H)

        exp_text7 = (
            "EXPERIMENTAL: The RL3033 agent is actively being trained and refined. "
            "Its output is blended with the deterministic fusion score rather than "
            "used in isolation."
        )
        y = G._experimental_warning_box(c, exp_text7, y, W, MX)

        c.setFont("Helvetica-Oblique", 8.5)
        c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "RL3033 is a Proximal Policy Optimisation (PPO) reinforcement learning agent. "
            "Unlike rule-based models, it was trained by trial and reward: it learned to "
            "predict risk by observing thousands of video sequences and improving its "
            "decisions over time. It outputs a temporal risk score between 0 and 1. "
            "Its score is blended with the rule-based fusion score (70% RL, 30% fusion) "
            "to produce the final risk value shown below.",
            MX, y, BODY_W, "Helvetica-Oblique", 8.5, 12)
        y -= 8

        y = G._status_badge(c, f"FINAL RISK: {final_risk:.2%}  [{rl_status}]",
                            y, rl_col, W, MX)
        y -= 10

        y = G._section_title(c, "Risk Components", y, MX, rl_col)
        fused_rl = float(s7.get("fused_risk", s6.get("fused_risk", 0.0)))
        for k, v in [
            ("Fusion Risk Input (Stage 6)",
             f"{fused_rl:.4f}  (rule-based score passed into the RL agent)"),
            ("Final Blended Risk",
             f"{final_risk:.4f}  (70% RL agent output + 30% fusion score)"),
            ("Weapon Override Applied",
             "YES, final risk floored at 0.85 (weapon forces minimum high-risk threshold)"
             if s6.get("weapon_flag") else "NO"),
            ("RL Agent Status", rl_status),
            ("Risk Classification",
             "CRITICAL (>= 0.85)" if final_risk >= 0.85
             else "HIGH_RISK (>= 0.65)" if final_risk >= 0.65
             else "MODERATE (>= 0.40)" if final_risk >= 0.40
             else "LOW_RISK (< 0.40)"),
        ]:
            y = G._kv_row(c, k, v, MX + 6, y, key_w=250)
        y -= 8

        # PDF-RL-EXPLAIN: expanded observation space with plain-English explanations
        y = G._section_title(c, "What the RL Agent Sees (Observation Space)",
                             y, MX, rl_col)
        c.setFont("Helvetica-Oblique", 8)
        c.setFillColor(colors.HexColor(G.C_GREY))
        y = G._wrap_text(c,
            "At each decision step the agent receives a structured observation made up "
            "of five named inputs. These are described below along with what they represent "
            "and why they are useful for temporal risk assessment.",
            MX + 6, y, BODY_W - 12, "Helvetica-Oblique", 8, 11)
        y -= 6

        rl_obs_entries = [
            (
                "ae_signals  (32 time steps x 6 channels)",
                "A sliding window of AE reconstruction errors over time. "
                "The six channels are: raw error, frame-to-frame change (delta), "
                "rate of change of the delta (acceleration), a fast-smoothed trend "
                "(short EMA), a slow-smoothed baseline (long EMA), and a spike ratio "
                "showing how much the current error exceeds the baseline. "
                "This lets the agent understand whether an anomaly is rising, "
                "falling, or sustained over time, rather than treating each frame in isolation."
            ),
            (
                "latent_motion  (1536 values)",
                "A compact numeric summary combining AE channel statistics "
                "(mean, standard deviation, max, min, percentiles) with motion "
                "tracking data. Gives the agent a sense of how movement in the scene "
                "correlates with reconstruction error patterns, enabling it to "
                "distinguish genuine physical events from camera artefacts or lighting changes."
            ),
            (
                "latent_stack  (2560 values)",
                "A combined vector of all pipeline signals: the AE error window, "
                "the binary and multi-class TimeSformer prediction scores (logits), "
                "audio feature values, tracking features, context vector, and Fusion "
                "layer outputs. This is the agent's full situational picture, allowing "
                "it to integrate evidence from every preceding stage simultaneously."
            ),
            (
                "spatial  (196 values, 14x14 grid)",
                "A spatial heatmap of the video frame divided into a 14-by-14 grid. "
                "Each cell records where weapon detections (YOLO) occurred and how "
                "much pixel-level variation existed in that region. High values "
                "highlight regions of interest within the frame, letting the agent "
                "weight spatially localised evidence more heavily."
            ),
            (
                "context  (5 values)",
                "A short summary vector: normalised AE score, binary classifier "
                "confidence, audio anomaly score, weapon detection flag (0 or 1), "
                "and motion intensity. Provides a high-level snapshot of all signals "
                "in a compact form, acting as a fast-access context cache for the agent."
            ),
        ]

        for entry_title, entry_desc in rl_obs_entries:
            if y < 80:
                break
            c.setFont("Helvetica-Bold", 8.5)
            c.setFillColor(colors.HexColor(G.C_DARK))
            c.drawString(MX + 8, y, entry_title)
            y -= 12
            y = G._wrap_text(c, entry_desc, MX + 16, y, BODY_W - 28,
                             "Helvetica", 8.5, 11)
            y -= 5

        if s7.get("interpretation"):
            y -= 4
            y = G._section_title(c, "Interpretation", y, MX, rl_col)
            y = G._wrap_text(c, s7["interpretation"], MX + 8, y,
                             BODY_W - 16, "Helvetica", 9, 13)
        y -= 4
        y = G._disclaimer_box(c, y, W)
        G._footer(c, ts, W)

        # ═══════════════════════════════════════════════════════════
        # PAGE 10 — TOP ANOMALOUS FRAMES
        # ═══════════════════════════════════════════════════════════
        frames_data  = s0.get("frames", [])
        scores_list  = s0.get("scores", [])
        samp_indices = s0.get("sampled_indices", [])
        if frames_data and scores_list:
            c.showPage()
            G._page_header(c, "TOP ANOMALOUS FRAMES", G.C_RED, W, H)
            y = G._body_top(H)
            c.setFont("Helvetica-Oblique", 8.5)
            c.setFillColor(colors.HexColor(G.C_GREY))
            y = G._wrap_text(c,
                "The three frames with the highest AE reconstruction error are shown below. "
                "High error means the AE could not reproduce the frame accurately from its "
                "learned representation of normal footage. These frames represent the moments "
                "in the clip that deviate most strongly from what the model expects to see.",
                MX, y, BODY_W, "Helvetica-Oblique", 8.5, 12)
            y -= 8
            top_f = G.extract_top_frames(
                frames_data, scores_list,
                samp_indices or list(range(len(scores_list))), n=3)
            for i, fd in enumerate(top_f, 1):
                tp = f"/tmp/af_v3_{i}.jpg"
                Image.fromarray(fd["frame"]).save(tp)
                frame_ts = ts_of(fd["index"])
                ratio    = fd["score"] / max(threshold, 1e-8)
                severity = ("CRITICAL" if ratio >= 2.0 else
                            "HIGH"     if ratio >= 1.0 else "BELOW THRESHOLD")
                c.setFillColor(colors.HexColor(G.C_DARK))
                c.setFont("Helvetica-Bold", 9)
                c.drawString(MX, y,
                    f"Frame {i}  |  {frame_ts}  |  "
                    f"Error: {fd['score']:.6f}  |  "
                    f"{ratio:.2f}x threshold  |  Severity: {severity}")
                y -= 14
                img_h = 155
                c.drawImage(tp, MX, y - img_h, width=BODY_W, height=img_h,
                            preserveAspectRatio=True)
                y -= img_h + 12
                if y < 80: break
        G._footer(c, ts, W)

        # ═══════════════════════════════════════════════════════════
        # PAGE 11 — STAGE 8 (LLAVA, EXPERIMENTAL)
        # PDF-LLAVA-NARR: unified narrative replaces frame-by-frame dump
        # ═══════════════════════════════════════════════════════════
        if s8 and (s8.get("unified_narrative") or s8.get("frame_descriptions")):
            c.showPage()
            G._page_header(c, "STAGE 8 (LLaVA, EXPERIMENTAL): AI VISUAL ANALYSIS",
                            G.C_PURPLE, W, H)
            y = G._body_top(H)

            exp_text8 = (
                "EXPERIMENTAL: LLaVA output is a language model generation and may "
                "contain hallucinations. Always verify descriptions against the actual frames."
            )
            y = G._experimental_warning_box(c, exp_text8, y, W, MX)

            c.setFont("Helvetica-Oblique", 8.5)
            c.setFillColor(colors.HexColor(G.C_GREY))
            binary_note = s8.get("binary_result", "Unknown")
            y = G._wrap_text(c,
                f"LLaVA (Large Language and Vision Assistant) receives four key frames "
                f"together with contextual metadata (weapon detections, motion behaviours, "
                f"audio status) and generates a natural-language description for each. "
                f"The binary classifier result was '{binary_note}'. "
                f"All four frame descriptions are synthesised below into a single coherent "
                f"scene narrative. Raw per-frame outputs are available in results.json.",
                MX, y, BODY_W, "Helvetica-Oblique", 8.5, 12)
            y -= 10

            ctx = s8.get("context_used", {})
            if ctx:
                y = G._section_title(c, "Context Provided to LLaVA", y, MX, G.C_PURPLE)
                for k, v in [
                    ("Weapon Information",  ctx.get("yolo_summary", "?")),
                    ("Motion Behaviours",   ", ".join(ctx.get("behaviors", [])) or "None"),
                    ("Audio Status",        ctx.get("audio_status", "?")),
                    ("Audio Score",         f"{ctx.get('audio_score',0):.3f}"),
                    ("Binary Classifier",   binary_note),
                    ("Frames Analysed",     "4 keyframes: clip start, AE onset, AE peak, AE end"),
                ]:
                    y = G._kv_row(c, k, v, MX + 6, y, key_w=185)
                y -= 8

            # ── Unified narrative (PRIMARY display) ──────────────
            y = G._section_title(c, "Scene Narrative (Synthesised)", y, MX, G.C_PURPLE)
            c.setFont("Helvetica-Oblique", 8)
            c.setFillColor(colors.HexColor(G.C_GREY))
            c.drawString(MX + 6, y,
                "The following is a coherent summary derived from all four frame "
                "analyses, with redundancy removed and context enriched.")
            y -= 14

            narrative = s8.get("unified_narrative", "")
            if not narrative:
                # Fallback: build a basic narrative from frame descriptions if
                # unified_narrative is absent (e.g. generated by an older run)
                fd_map = s8.get("frame_descriptions", {})
                parts  = [fd_map.get(r, "") for r in
                          ("baseline", "anom_start", "peak", "anom_end") if fd_map.get(r)]
                narrative = " ".join(parts) if parts else "No scene description available."

            # Draw narrative in a lightly tinted box for visual prominence
            # First measure how many lines it will need
            import io as _io
            from reportlab.pdfgen.canvas import Canvas as _TC2
            _tb = _io.BytesIO()
            _tc = _TC2(_tb)
            _tc.setFont("Helvetica", 9)
            _words = narrative.split()
            _line  = ""
            _nlines = 0
            for _w in _words:
                _test = _line + _w + " "
                if _tc.stringWidth(_test, "Helvetica", 9) < BODY_W - 24:
                    _line = _test
                else:
                    _nlines += 1
                    _line = _w + " "
            if _line.strip():
                _nlines += 1
            narr_box_h = _nlines * 13 + 24
            narr_box_h = min(narr_box_h, y - 90)  # don't overflow page

            c.setFillColor(colors.HexColor("#f4ecf7"))
            c.setStrokeColor(colors.HexColor("#6c3483"))
            c.setLineWidth(0.8)
            c.roundRect(MX, y - narr_box_h, BODY_W, narr_box_h, 5,
                        fill=True, stroke=True)
            y_narr = y - 11
            G._wrap_text(c, narrative, MX + 10, y_narr, BODY_W - 24,
                         "Helvetica", 9, 13, min_y=y - narr_box_h + 10)
            y -= narr_box_h + 14

            # ── Keyframes used in Scene Narrative ─────────────────
            fd_map = s8.get("frame_descriptions", {})
            frame_roles = [
                ("baseline",   "Frame 1: Beginning of clip"),
                ("anom_start", "Frame 2: AE Onset"),
                ("peak",       "Frame 3: AE Peak"),
                ("anom_end",   "Frame 4: AE End"),
            ]
            # Collect the actual frame arrays for the 4 keyframes
            frames_data_llava = s0.get("frames", [])
            peak_idx_llava    = int(s0.get("peak_frame_idx", 0))
            anom_s_llava      = int(s0.get("anomaly_start_frame", 0))
            anom_e_llava      = int(s0.get("anomaly_end_frame", peak_idx_llava))
            n_frames_llava    = len(frames_data_llava)
            frame_idx_map = {
                "baseline":   0,
                "anom_start": min(anom_s_llava, n_frames_llava - 1) if n_frames_llava else 0,
                "peak":       min(peak_idx_llava, n_frames_llava - 1) if n_frames_llava else 0,
                "anom_end":   min(anom_e_llava, n_frames_llava - 1) if n_frames_llava else 0,
            }
            if frames_data_llava and y > 130:
                y = G._section_title(c, "Keyframes Used in Scene Narrative", y, MX, G.C_PURPLE)
                c.setFont("Helvetica-Oblique", 8)
                c.setFillColor(colors.HexColor(G.C_GREY))
                c.drawString(MX + 6, y,
                    "The four frames below were provided to LLaVA as visual input for the scene narrative above.")
                y -= 14
                n_kf   = len(frame_roles)
                kf_w   = (BODY_W - (n_kf - 1) * 6) / n_kf
                kf_h   = min(90, kf_w * 0.65)
                if y - kf_h - 36 > 40:
                    for kf_i, (role, label) in enumerate(frame_roles):
                        fi = frame_idx_map[role]
                        tmp_kf = f"/tmp/llava_kf_{kf_i}.jpg"
                        try:
                            Image.fromarray(frames_data_llava[fi]).save(tmp_kf)
                            cx = MX + kf_i * (kf_w + 6)
                            c.drawImage(tmp_kf, cx, y - kf_h, width=kf_w - 4, height=kf_h,
                                        preserveAspectRatio=True)
                            c.setFont("Helvetica-Bold", 7)
                            c.setFillColor(colors.HexColor(G.C_DARK))
                            c.drawString(cx, y - kf_h - 11, label)
                            raw_desc = fd_map.get(role, "")
                            if raw_desc:
                                # Clip snippet to fit within kf_w - 4 px column
                                c.setFont("Helvetica-Oblique", 6)
                                c.setFillColor(colors.HexColor(G.C_GREY))
                                max_chars = max(1, int((kf_w - 8) / 3.3))
                                snippet = raw_desc[:max_chars]
                                if len(raw_desc) > max_chars:
                                    snippet = snippet.rstrip() + "…"
                                c.drawString(cx, y - kf_h - 22, snippet)
                        except Exception:
                            pass
                    y -= kf_h + 36
                y -= 8

            # ── Key signal cross-reference ────────────────────────
            if y > 130:
                y = G._section_title(c, "Signal Cross-Reference", y, MX, G.C_PURPLE)
                c.setFont("Helvetica-Oblique", 8)
                c.setFillColor(colors.HexColor(G.C_GREY))
                y = G._wrap_text(c,
                    "The narrative above was generated with the following system signals "
                    "as ground truth context. These values are deterministic and should "
                    "be used to validate or challenge the language model's description.",
                    MX + 6, y, BODY_W - 12, "Helvetica-Oblique", 8, 11)
                y -= 6
                xref_pairs = [
                    ("AE Peak Timestamp",       ts_of(s0.get("peak_frame_idx", 0))),
                    ("AE Max Error vs Threshold",
                     f"{s0.get('max_error',0):.6f} vs {threshold:.6f} "
                     f"({'exceeds' if s0.get('max_error',0) > threshold else 'below'})"),
                    ("Binary Classification",   f"{s1.get('classification','?')} ({s1.get('confidence',0):.1%})"),
                    ("Anomaly Type",
                     f"{s2.get('class','N/A')} ({s2.get('confidence',0):.1%})"
                     if s2.get("class") not in ("Skipped", "Unknown", None) else "N/A"),
                    ("Weapon Detected",         "YES" if s3.get("weapon_flag") else "NO"),
                    ("Audio Status",            s4.get("status", "N/A")),
                    ("Motion Behaviours",        ", ".join(s5.get("behaviors", [])) or "None"),
                    ("Final Risk",              f"{s7.get('final_risk', s6.get('fused_risk', 0)):.1%}"),
                ]
                for k, v in xref_pairs:
                    y = G._kv_row(c, k, v, MX + 6, y, key_w=210)
                    if y < 70: break

            y -= 4
            y = G._disclaimer_box(c, y, W)
            G._footer(c, ts, W)

        c.save()
        return output_path

    @staticmethod
    def generate_all(results: dict) -> dict:
        s0        = results.get("stage0") or {}
        scores    = s0.get("scores", [])
        samp      = s0.get("sampled_indices", [])
        threshold = results.get("threshold", Config.SAFE_THRESHOLD_LEGACY)
        peak_si   = s0.get("peak_score_idx", 0)
        fps       = float(s0.get("fps", 25.0))
        if scores:
            gp = EnhancedReportGenerator.generate_deviation_graph(
                scores, threshold, [peak_si], samp, fps,
                "/tmp/deviation_graph_v3.png")
            results["graph_path"] = gp
        pdf = EnhancedReportGenerator.generate_enhanced_pdf(
            results, "/tmp/forensic_report_v3.pdf")
        results["pdf_path"] = pdf
        return results


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE — SCENE SOLVER v3
# ══════════════════════════════════════════════════════════════════════════════
class SceneSolverPipeline:
    """
    Stage ordering (enforced):
      Stage 0: AE              — primary anomaly signal
      Stage 1: TF Binary       — semantic understanding (GATES everything)
      Stage 2: TF Multi-class  — SKIPPED when Stage 1 = Normal  (LOGIC-NORMAL)
      Stage 3: YOLO Weapon     — SKIPPED when Stage 1 = Normal
      Stage 4: Audio
      Stage 5: Tracking
      Stage 6: Fusion
      Stage 7: RL3033 (Experimental, FINAL)
      Stage 8: LLaVA (Experimental, always runs when enable_llava=True)  (LOGIC-NORMAL)

    LOGIC-NORMAL: when Stage 1 = Normal,
      - results['status'] is overridden to 'SAFE'
      - Stage 2 is skipped and returns a placeholder dict
      - Stage 3 is skipped (existing behaviour, unchanged)
      - LLaVA still runs with neutral investigative prompts
    """

    def __init__(self, config: Optional[Config] = None):
        self.config       = config or Config()
        self.stage0       = Stage0AE(self.config)
        self.stage1       = Stage1TSBinary(self.config)
        self.stage2       = Stage2TSMulti(self.config)
        self.stage3       = Stage3YOLOWeapon(self.config)
        self.stage4       = Stage4Audio(self.config)
        self.stage5       = Stage5Tracking(self.config)
        self.stage6       = Stage6Fusion(self.config)
        self.stage7       = Stage7RL3033(self.config)
        self.stage8       = Stage8LLaVA(self.config)
        self.alert_system = AlertSystem(self.config)
        self.reporter     = EnhancedReportGenerator()

    def _make_output_dir(self, video_path: str, base: str) -> str:
        name = os.path.splitext(os.path.basename(video_path))[0]
        ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
        out  = os.path.join(base, "analysis_results", name, ts)
        os.makedirs(out, exist_ok=True)
        return out

    def analyze(self,
                video_path:      str,
                base_output_dir: Optional[str] = None,
                enable_llava:    bool           = True,
                enable_rl:       bool           = True,
                enable_tracking: bool           = True) -> dict:
        base = base_output_dir or self.config.DRIVE_ROOT
        out  = self._make_output_dir(video_path, base)
        log.info("=" * 65)
        log.info("  SCENE SOLVER v3, REASONING PIPELINE")
        log.info("=" * 65)
        log.info(f"  Video  : {os.path.basename(video_path)}")
        log.info(f"  Output : {out}")
        log.info("=" * 65)
        results: dict = {"video_path": video_path, "output_directory": out}

        # ── STAGE 0: AE ───────────────────────────────────────────
        log.info("\n== STAGE 0: AE ==")
        stage0 = self.stage0.analyze(video_path)
        results["stage0"]    = stage0
        results["threshold"] = stage0["threshold"]

        frames              = stage0["frames"]
        scores              = stage0["scores"]
        sampled_indices     = stage0["sampled_indices"]
        peak_frame_idx      = stage0["peak_frame_idx"]
        peak_score_idx      = stage0.get("peak_score_idx", 0)
        threshold           = stage0["threshold"]
        anomaly_start_frame = stage0.get("anomaly_start_frame", 0)
        anomaly_end_frame   = stage0.get("anomaly_end_frame", peak_frame_idx)
        fps                 = stage0.get("fps", 25.0)

        ae_signals = self.stage0.build_ae_signals(scores, peak_score_idx)

        # ── STAGE 1: TimeSformer Binary ───────────────────────────
        log.info("\n== STAGE 1: TimeSformer Binary ==")
        stage1 = self.stage1.analyze(frames, peak_frame_idx)
        results["stage1"] = stage1
        is_suspicious    = (stage1.get("classification") == "Suspicious")
        ts_binary_logits = self.stage1.get_logits()

        # LOGIC-NORMAL: override overall status when TF Binary = Normal
        if is_suspicious:
            results["status"] = stage0["status"]
        else:
            results["status"] = "SAFE"
            log.info("[Pipeline] Stage 1 = Normal => overall status overridden to SAFE")

        # ── STAGE 2: TimeSformer Multi-class ─────────────────────
        log.info("\n== STAGE 2: TimeSformer Multi-class ==")
        if is_suspicious:
            stage2 = self.stage2.analyze(
                frames=frames, peak_idx=peak_frame_idx,
                ae_scores=scores, sampled_indices=sampled_indices,
            )
        else:
            log.info("[Stage2-TSMulti] Skipped (Stage 1 = Normal)")
            stage2 = {
                "class":             "Skipped",
                "confidence":        0.0,
                "top3":              [],
                "all_probabilities": {},
                "logits":            [0.0] * 7,
                "sampling_method":   "skipped",
            }
        results["stage2"] = stage2
        ts_multi_logits   = self.stage2.get_logits()

        # ── STAGE 3: YOLO Weapon (conditional on Stage 1) ────────
        log.info("\n== STAGE 3: YOLO Weapon ==")
        if is_suspicious:
            stage3 = self.stage3.analyze(
                frames=frames,
                anomaly_start_frame=anomaly_start_frame,
                anomaly_end_frame=anomaly_end_frame,
            )
        else:
            stage3 = {
                "weapon_flag":          False,
                "weapon_count":         0,
                "dominant_class":       "None",
                "best_detection":       {},
                "all_detections":       [],
                "top_frames_annotated": [],
                "spatial_heatmap":      np.zeros(196, dtype=np.float32),
                "summary":              "Skipped, Stage 1 = Normal",
            }
            log.info("[Stage3-YOLO] Skipped (Stage1=Normal)")
        results["stage3"] = stage3

        # ── STAGE 4: Audio ────────────────────────────────────────
        log.info("\n== STAGE 4: Audio ==")
        stage4 = self.stage4.analyze(video_path)
        results["stage4"] = stage4

        # ── STAGE 5: Tracking ─────────────────────────────────────
        if enable_tracking:
            log.info("\n== STAGE 5: Tracking ==")
            stage5 = self.stage5.analyze(
                frames=frames,
                anomaly_start_frame=anomaly_start_frame,
                n_frames=32,
            )
        else:
            log.info("\n== STAGE 5: Tracking DISABLED ==")
            stage5 = {
                "motion_intensity":  0.0,
                "behaviors":         ["STATIC"],
                "track_count":       0,
                "tracking_features": np.zeros(20, dtype=np.float32),
                "status":            "DISABLED",
            }
        results["stage5"] = stage5

        # ── STAGE 6: Fusion ───────────────────────────────────────
        log.info("\n== STAGE 6: Fusion ==")
        stage6 = self.stage6.fuse(
            ae_result=stage0, ts_binary_result=stage1,
            ts_multi_result=stage2, yolo_result=stage3,
            audio_result=stage4, tracking_result=stage5,
        )
        results["stage6"] = stage6

        # ── STAGE 7: RL3033 (EXPERIMENTAL, FINAL) ─────────────────
        if enable_rl:
            log.info("\n== STAGE 7: RL3033 ==")
            stage7 = self.stage7.analyze(
                ae_signals=ae_signals,
                ts_binary_logits=ts_binary_logits,
                ts_multi_logits=ts_multi_logits,
                yolo_result=stage3,
                audio_result=stage4,
                tracking_result=stage5,
                fusion_result=stage6,
                frames=frames,
                peak_frame_idx=peak_frame_idx,
            )
        else:
            log.info("\n== STAGE 7: RL3033 DISABLED ==")
            stage7 = {
                "final_risk":     float(stage6.get("fused_risk", 0.0)),
                "status":         "DISABLED",
                "interpretation": "RL agent disabled. Using fused risk from Stage 6.",
            }
        results["stage7"] = stage7
        # Sync overall status and risk_level to Stage 7 final risk
        _fr = float(stage7.get("final_risk", stage6.get("fused_risk", 0.0)))
        _rl = ("CRITICAL"  if _fr >= 0.85 else
               "HIGH"      if _fr >= 0.70 else
               "MODERATE"  if _fr >= 0.40 else "LOW")
        stage6["risk_level"] = _rl
        if _fr >= 0.40 and results.get("status") == "SAFE":
            pass  # Stage 1 semantic veto is preserved; only label is synced
        results["risk_label_final"] = _rl

        # ── STAGE 8: LLaVA (EXPERIMENTAL) ─────────────────────────
        if enable_llava:
            log.info("\n== STAGE 8: LLaVA (neutral prompts) ==")
            stage8 = self.stage8.analyze(
                frames=frames,
                peak_idx=peak_frame_idx,
                anomaly_start_frame=anomaly_start_frame,
                anomaly_end_frame=anomaly_end_frame,
                yolo_result=stage3,
                audio_result=stage4,
                tracking_result=stage5,
                binary_result=stage1.get("classification", "Unknown"),
            )
        else:
            stage8 = None
        results["stage8"] = stage8

        # ── Alert ─────────────────────────────────────────────────
        alert = self.alert_system.evaluate_and_fire(results)
        results["alert"] = alert

        # ── Visualisations + PDF ──────────────────────────────────
        results = EnhancedReportGenerator.generate_all(results)

        # ── Save ──────────────────────────────────────────────────
        self._save_outputs(results, out, video_path)
        self._print_summary(results)

        return results

    def _save_outputs(self, results: dict, out_dir: str, video_path: str):
        def _default(o):
            if isinstance(o, np.ndarray):
                return o.tolist()
            return str(o)
        save = {k: v for k, v in results.items()}
        if "stage0" in save and save["stage0"]:
            save["stage0"] = {k: v for k, v in save["stage0"].items() if k != "frames"}
        for skey in ("stage3", "stage4", "stage5", "stage6"):
            if skey in save and save[skey]:
                sx = dict(save[skey])
                for arr_key in ("spatial_heatmap", "audio_features_vec",
                                "tracking_features", "context_vector"):
                    if arr_key in sx and isinstance(sx[arr_key], np.ndarray):
                        sx[arr_key] = sx[arr_key].tolist()
                if skey == "stage3":
                    sx.pop("all_detections", None)
                    sx.pop("top_frames_annotated", None)
                save[skey] = sx
        json_path = os.path.join(out_dir, "results.json")
        with open(json_path, "w") as f:
            json.dump(save, f, indent=2, default=_default)

        s0 = results.get("stage0") or {}
        fps_sv = float(s0.get("fps", 25.0))
        def ts_s(fi): return frame_to_timestamp(int(fi), fps_sv)

        s1 = results.get("stage1") or {}
        s2 = results.get("stage2") or {}
        s3 = results.get("stage3") or {}
        s4 = results.get("stage4") or {}
        s5 = results.get("stage5") or {}
        s6 = results.get("stage6") or {}
        s7 = results.get("stage7") or {}
        s8 = results.get("stage8") or {}

        txt_path = os.path.join(out_dir, "summary.txt")
        with open(txt_path, "w") as f:
            f.write(f"SCENE SOLVER v3, {os.path.basename(video_path)}\n")
            f.write("=" * 65 + "\n\n")
            f.write(f"Status              : {results.get('status')}\n")
            f.write(f"Threshold           : {results.get('threshold',0):.6f}\n")
            f.write(f"AE Max Error        : {s0.get('max_error',0):.6f}\n")
            f.write(f"Peak Timestamp      : {ts_s(s0.get('peak_frame_idx',0))}\n")
            if s1:
                f.write(f"TF Binary           : {s1.get('classification')} ({s1.get('confidence',0):.1%})\n")
            if s2:
                skipped_note = " [Skipped, Stage 1 = Normal]" if s2.get("class") == "Skipped" else f" ({s2.get('confidence',0):.1%}) [{s2.get('sampling_method','?')}]"
                f.write(f"Anomaly Type        : {s2.get('class','?')}{skipped_note}\n")
            if s3:
                wf = s3.get("weapon_flag", False)
                f.write(f"Weapon Detection    : {'Weapons detected' if wf else 'No weapons detected'}\n")
            if s4:
                audio_ok = s4.get("audio_present", True)
                f.write(f"Audio               : {'No audio or silent' if not audio_ok else s4.get('status','?')}"
                        + (f"  score={s4.get('audio_score',0):.3f}" if audio_ok else "") + "\n")
            if s5:
                f.write(f"Tracking            : motion={s5.get('motion_intensity',0):.3f} behaviors={s5.get('behaviors',[])} [{s5.get('status')}]\n")
            if s6:
                f.write(f"Fusion Risk         : {s6.get('fused_risk',0):.1%} [{s6.get('risk_level')}]\n")
                for line in s6.get("reasoning_trace", []):
                    f.write(f"  {line}\n")
            if s7:
                f.write(f"RL3033 Final        : {s7.get('final_risk',0):.1%} [{s7.get('status')}]\n")
            if results.get("alert"):
                al = results["alert"]
                f.write(f"ALERT               : {al['level']} risk={al['final_risk']:.1%}\n")
            if s8 and s8.get("unified_narrative"):
                f.write(f"\nAI Visual Narrative:\n{s8['unified_narrative']}\n")
            elif s8 and s8.get("description"):
                f.write(f"\nAI Visual Description:\n{s8['description']}\n")

        if results.get("pdf_path") and os.path.exists(results["pdf_path"]):
            dest = os.path.join(out_dir, "forensic_report.pdf")
            shutil.copy2(results["pdf_path"], dest)
            results["pdf_path_local"] = dest
            colab_dest = "/content/scene_solver_report.pdf"
            try:
                shutil.copy2(results["pdf_path"], colab_dest)
                log.info(f"[Output] PDF also saved to {colab_dest}")
                try:
                    from google.colab import files as colab_files
                    colab_files.download(colab_dest)
                except Exception:
                    pass
            except Exception:
                pass
        log.info(f"[Output] Saved to {out_dir}")

    def _print_summary(self, results: dict):
        s0 = results.get("stage0") or {}
        fps_p = float(s0.get("fps", 25.0))
        def ts_p(fi): return frame_to_timestamp(int(fi), fps_p)
        s1 = results.get("stage1") or {}
        s2 = results.get("stage2") or {}
        s3 = results.get("stage3") or {}
        s4 = results.get("stage4") or {}
        s5 = results.get("stage5") or {}
        s6 = results.get("stage6") or {}
        s7 = results.get("stage7") or {}
        s8 = results.get("stage8") or {}
        al = results.get("alert")
        print("\n" + "=" * 65)
        print("  SCENE SOLVER v3, SUMMARY")
        print("=" * 65)
        print(f"  Overall Status        : {results.get('status')}")
        print(f"  Dynamic Threshold     : {results.get('threshold',0):.6f}")
        print(f"\n  [Stage 0 (AE)]")
        print(f"    Max Error           : {s0.get('max_error',0):.6f}")
        print(f"    Avg Error           : {s0.get('avg_error',0):.6f}")
        print(f"    Peak Timestamp      : {ts_p(s0.get('peak_frame_idx',0))}")
        print(f"\n  [Stage 1 (TF Binary)]")
        print(f"    Classification      : {s1.get('classification','?')} ({s1.get('confidence',0):.2%})")
        if s2:
            print(f"\n  [Stage 2 (TF Multi)]")
            print(f"    Class               : {s2.get('class','?')} ({s2.get('confidence',0):.2%})")
            print(f"    Sampling            : {s2.get('sampling_method','?')}")
        if s3:
            wf = s3.get("weapon_flag", False)
            print(f"\n  [Stage 3 (YOLO Weapon, Experimental)]")
            print(f"    Result              : {'Weapons detected' if wf else 'No weapons detected'}")
        if s4:
            audio_ok = s4.get("audio_present", True)
            print(f"\n  [Stage 4 (Audio)]")
            print(f"    Status              : {'No audio or silent' if not audio_ok else s4.get('status','?')}")
        if s5:
            print(f"\n  [Stage 5 (Tracking)]")
            print(f"    Motion Intensity    : {s5.get('motion_intensity',0):.3f}")
            print(f"    Behaviours          : {s5.get('behaviors',[])}")
        if s6:
            print(f"\n  [Stage 6 (Fusion)]")
            print(f"    Fused Risk          : {s6.get('fused_risk',0):.2%}")
            print(f"    Risk Level          : {s6.get('risk_level','?')}")
        if s7:
            print(f"\n  [Stage 7 (RL3033, Experimental)]")
            print(f"    Final Risk          : {s7.get('final_risk',0):.2%}")
            print(f"    Status              : {s7.get('status','?')}")
        if al:
            print(f"\n  ALERT: {al['level']}  |  final_risk={al['final_risk']:.2%}")
        if s8 and s8.get("unified_narrative"):
            narr = s8["unified_narrative"]
            print(f"\n  [Stage 8 (LLaVA, Experimental)]")
            print("    " + (narr[:300] + "..." if len(narr) > 300 else narr))
        elif s8 and s8.get("description"):
            desc = s8["description"]
            print(f"\n  [Stage 8 (LLaVA, Experimental)]")
            print("    " + (desc[:300] + "..." if len(desc) > 300 else desc))
        print("=" * 65 + "\n")

    def start_stream(self, source: str, window_frames: int = 50,
                     on_alert: Any = None):
        stream = RealTimeStream(self)
        stream.start(source, window_frames=window_frames, on_alert=on_alert)
        return stream


# ══════════════════════════════════════════════════════════════════════════════
# READY MESSAGE
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("[STATUS] SCENE SOLVER v3, REASONING SYSTEM READY")
print("=" * 65)
print(f"Device             : {Config.DEVICE}")
print(f"RL3033 Available   : {RL_AVAILABLE}")
print(f"YOLO Available     : {YOLO_AVAILABLE}")
print(f"Librosa Available  : {LIBROSA_AVAILABLE}")
print()
print("Usage:")
print("  pipeline = SceneSolverPipeline()")
print("  results  = pipeline.analyze('/path/to/video.mp4')")
print()
print("Improvements in this revision (PDF readability + content density):")
print("  PDF-EXPBOX3     : Experimental boxes: single continuous paragraph, no fragmentation.")
print("  PDF-LLAVA-NARR  : LLaVA section now shows one unified scene narrative.")
print("  PDF-DENSITY     : All sections enriched with interpretive context and insights.")
print("  PDF-AE-SIGNAL   : AE page now shows signal ratio, severity, and interpretation.")
print("  PDF-STAGE2-SKIP : Stage 2 skip page now explains WHY it was skipped in detail.")
print("  PDF-STAGE3-OPS  : Stage 3 skipped page now includes operational note.")
print("  PDF-AUDIO-FLAGS : Audio page now flags which normalised features exceeded threshold.")
print("  PDF-TRACKING+   : Tracking page enriched with fusion contribution and method notes.")
print("  PDF-FUSION+     : Fusion input table now shows weight context for each signal.")
print("  PDF-FRAMES+     : Anomalous frames now show severity label and threshold ratio.")
print("  PDF-INTERP      : Page 1 now includes a full PIPELINE INTERPRETATION paragraph.")
print()
print("Architecture (enforced reasoning order):")
print("  Stage 0 (AE)                  primary anomaly signal")
print("  Stage 1 (TF Binary)           semantic gate (overrides status to SAFE if Normal)")
print("  Stage 2 (TF Multi)            skipped if Stage 1 = Normal")
print("  Stage 3 (YOLO Weapon, Exp.)   skipped if Stage 1 = Normal")
print("  Stage 4 (Audio)               always runs")
print("  Stage 5 (Tracking)            always runs")
print("  Stage 6 (Fusion)              always runs")
print("  Stage 7 (RL3033, Exp.)        always runs, FINAL stage")
print("  Stage 8 (LLaVA, Exp.)         always runs when enable_llava=True")
print("=" * 65)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[STATUS] SCENE SOLVER v3, REASONING SYSTEM READY
Device             : cuda
RL3033 Available   : True
YOLO Available     : True
Librosa Available  : True

Usage:
  pipeline = SceneSolverPipeline()
  results  = pipeline.analyze('/path/to/video.mp4')

Improvements in this revision (PDF readability + content density):
  PDF-EXPBOX3     : Experimental boxes: single continuous paragraph, no fragmentation.
  PDF-LLAVA-NARR  : LLaVA section now shows one unified scene narrative.
  PDF-DENSITY     : All sections enriched with interpretive context and insights.
  PDF-AE-SIGNAL   : AE page now shows signal ratio, severity, and interpretation.
  PDF-STAGE2-SKIP : Stage 2 skip page now explain

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
# Mount drive first
from google.colab import drive
drive.mount('/content/drive')

# Then run the cell above, then:
pipeline = SceneSolverPipeline()
#results = pipeline.analyze('/content/drive/MyDrive/solver1/AnomalyDetectionDataset/Explosion/Explosion001_x264.mp4')

# Without LLaVA (faster):
#results = pipeline.analyze('/path/to/video.mp4', enable_llava=False)

Mounted at /content/drive


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Install (run once)
# !pip install -q gradio
# ══════════════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — CrimeSceneSolver Gradio Web App  (v7 — all fixes applied)
# Paste AFTER the main pipeline cell. Do NOT modify pipeline code.
#
# CHANGES vs v6:
#  FIX-1  NAV TABS       : Added explicit gap/padding so tabs are evenly spaced.
#  FIX-2  ANALYSE BUTTON : onclick now uses a more robust multi-strategy selector
#                          (data-tab-id, aria-controls, text-match) so navigation
#                          works reliably across Gradio versions.
#  FIX-3  REPORT STYLING : "Risk Assessment & Report" label + section header text
#                          are bolder and more visually prominent.
#  FIX-4  AE DEVIATION   : AE Deviation graph is now rendered inline inside the
#                          Report HTML via a base64-encoded <img> tag so it is
#                          visible in the Risk Assessment section as well as in
#                          the PDF. The graph_path is passed through process_video.
#  FIX-5  DISCLAIMER     : Full disclaimer text added to footer on every page
#                          (Home, Analyse, Contact) in a clearly visible banner.
# ══════════════════════════════════════════════════════════════════════════════

import gradio as gr
import numpy as np
import json, os, traceback, smtplib, ssl, re, base64
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from PIL import Image

# ─────────────────────────────────────────────────────────────────────────────
# EMAIL CONFIG
# ─────────────────────────────────────────────────────────────────────────────
ADMIN_EMAIL  = "anumulasamarth008@gmail.com"
SMTP_SERVER  = "smtp.gmail.com"
SMTP_PORT    = 465
SENDER_EMAIL = os.environ.get("SENDER_EMAIL", "")
SENDER_PASS  = os.environ.get("SENDER_PASS",  "")

def _send_contact_email(name, email, phone, message):
    if not SENDER_EMAIL or not SENDER_PASS:
        with open("/tmp/css_contact.log", "a") as f:
            f.write(f"[CONTACT] {name} | {email} | {phone}\n")
        return False
    try:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = f"[CrimeSceneSolver] Contact: {name}"
        msg["From"]    = SENDER_EMAIL
        msg["To"]      = ADMIN_EMAIL
        body = f"Name:{name}\nEmail:{email}\nPhone:{phone}\nMessage:\n{message}\n"
        msg.attach(MIMEText(body, "plain"))
        ctx = ssl.create_default_context()
        with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT, context=ctx) as srv:
            srv.login(SENDER_EMAIL, SENDER_PASS)
            srv.sendmail(SENDER_EMAIL, ADMIN_EMAIL, msg.as_string())
        return True
    except Exception:
        return False

# ─────────────────────────────────────────────────────────────────────────────
# PIPELINE WRAPPER
# ─────────────────────────────────────────────────────────────────────────────
def run_pipeline(video_path: str) -> dict:
    pipeline = SceneSolverPipeline()
    results  = pipeline.analyze(
        video_path=video_path,
        base_output_dir="/tmp/css_webapp",
        enable_llava=True, enable_rl=True, enable_tracking=True,
    )
    s0 = results.get("stage0") or {}
    s1 = results.get("stage1") or {}
    s2 = results.get("stage2") or {}
    s3 = results.get("stage3") or {}
    s4 = results.get("stage4") or {}
    s5 = results.get("stage5") or {}
    s6 = results.get("stage6") or {}
    s7 = results.get("stage7") or {}
    s8 = results.get("stage8") or {}
    fps = float(s0.get("fps", 25.0))

    def ts(fi): return frame_to_timestamp(int(fi), fps)

    risk_score = float(s7.get("final_risk", s6.get("fused_risk", 0.0)))
    rl_status   = s7.get("status", "")
    if rl_status in ("CRITICAL", "HIGH_RISK", "MODERATE", "LOW_RISK"):
        rl_label_map = {"CRITICAL":"CRITICAL","HIGH_RISK":"HIGH","MODERATE":"MODERATE","LOW_RISK":"LOW"}
        risk_label = rl_label_map.get(rl_status, s6.get("risk_level","LOW"))
    else:
        risk_label = results.get("risk_label_final") or s6.get("risk_level", "LOW")

    status    = results.get("status", "UNKNOWN")
    narrative = (s8 or {}).get("unified_narrative") or (s8 or {}).get("description") or ""

    report = {
        "overall_status":   status,
        "final_risk_score": f"{risk_score:.1%}",
        "risk_level":       risk_label,
        "dynamic_threshold": f"{results.get('threshold',0):.6f}",
        "ae_max_error":     f"{s0.get('max_error',0):.6f}",
        "ae_avg_error":     f"{s0.get('avg_error',0):.6f}",
        "peak_timestamp":   ts(s0.get("peak_frame_idx",0)),
        "anomaly_window":   f"{ts(s0.get('anomaly_start_frame',0))} → {ts(s0.get('anomaly_end_frame',0))}",
        "stage1_tf_binary": f"{s1.get('classification','?')}  ({s1.get('confidence',0):.1%})",
        "stage2_anomaly_type": f"{s2.get('class','N/A')}  ({s2.get('confidence',0):.1%})",
        "stage2_top3":      s2.get("top3", []),
        "stage2_all_probs": s2.get("all_probabilities", {}),
        "stage3_weapon":    "YES — " + str(s3.get("dominant_class","?")) if s3.get("weapon_flag") else "None detected",
        "stage4_audio":     f"{s4.get('status','N/A')}  (score: {s4.get('audio_score',0):.4f})",
        "stage5_motion":    f"{s5.get('motion_intensity',0):.3f}  |  {', '.join(s5.get('behaviors',[]) or ['STATIC'])}",
        "stage6_fusion_risk": f"{s6.get('fused_risk',0):.1%}  [{s6.get('risk_level','?')}]",
        "stage6_reasoning": s6.get("reasoning_trace", []),
        "stage7_rl_final":  f"{s7.get('final_risk',0):.1%}  [{rl_status}]",
        "stage7_interpretation": s7.get("interpretation",""),
        "llava_narrative":  narrative,
        # raw values for charts
        "_stage1_conf":     float(s1.get("confidence", 0)),
        "_stage2_conf":     float(s2.get("confidence", 0)),
        "_stage2_top3_raw": s2.get("top3", []),
        "_stage2_all_probs": s2.get("all_probabilities", {}),
        "_risk_score":      risk_score,
        "_fused_risk":      float(s6.get("fused_risk", 0)),
        "_audio_score":     float(s4.get("audio_score", 0)),
        "_motion":          float(s5.get("motion_intensity", 0)),
    }

    frames_l = s0.get("frames", [])
    scores_l = s0.get("scores", [])
    samp_idx = s0.get("sampled_indices", [])
    top_frame = None
    if frames_l and scores_l:
        psi = int(s0.get("peak_score_idx",0))
        pfi = samp_idx[psi] if psi < len(samp_idx) else 0
        pfi = min(pfi, len(frames_l)-1)
        top_frame = frames_l[pfi]

    def _s(o):
        if isinstance(o, np.ndarray): return o.tolist()
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        return str(o)

    jd = {
        "status": status, "risk_score": round(risk_score,4), "risk_level": risk_label,
        "threshold": round(float(results.get("threshold",0)),6),
        "stage0": {"max_error": round(float(s0.get("max_error",0)),6),
                   "avg_error": round(float(s0.get("avg_error",0)),6),
                   "peak_timestamp": ts(s0.get("peak_frame_idx",0)),
                   "anomaly_start": ts(s0.get("anomaly_start_frame",0)),
                   "anomaly_end": ts(s0.get("anomaly_end_frame",0))},
        "stage1": {"classification": s1.get("classification","?"),
                   "confidence": round(float(s1.get("confidence",0)),4)},
        "stage2": {"class": s2.get("class","N/A"),
                   "confidence": round(float(s2.get("confidence",0)),4),
                   "top3": s2.get("top3",[])},
        "stage3": {"weapon_flag": bool(s3.get("weapon_flag",False)),
                   "dominant_class": s3.get("dominant_class","None"),
                   "weapon_count": int(s3.get("weapon_count",0))},
        "stage4": {"status": s4.get("status","N/A"),
                   "audio_score": round(float(s4.get("audio_score",0)),4)},
        "stage5": {"motion_intensity": round(float(s5.get("motion_intensity",0)),4),
                   "behaviors": s5.get("behaviors",[])},
        "stage6": {"fused_risk": round(float(s6.get("fused_risk",0)),4),
                   "risk_level": s6.get("risk_level","?"),
                   "reasoning": s6.get("reasoning_trace",[])},
        "stage7": {"final_risk": round(risk_score,4),
                   "status": rl_status,
                   "interpretation": s7.get("interpretation","")},
        "llava_narrative": narrative,
    }

    pdf = (results.get("pdf_path_local") or results.get("pdf_path")
           or "/tmp/forensic_report_v3.pdf")
    if not os.path.exists(pdf): pdf = None

    # FIX-4: also pass the AE deviation graph path through
    graph_path = results.get("graph_path", "/tmp/deviation_graph_v3.png")
    if not os.path.exists(graph_path): graph_path = None

    return {"risk_score": risk_score, "risk_label": risk_label,
            "status": status, "report": report,
            "top_frame": top_frame, "json_data": jd, "pdf_path": pdf,
            "graph_path": graph_path}


def _risk_col(r: float) -> str:
    if r >= 0.70: return "#FF4500"
    if r >= 0.40: return "#FF8C00"
    return "#4CAF50"


# ─────────────────────────────────────────────────────────────────────────────
# FIX-4: Helper — encode a PNG/JPG file to a base64 data-URI for inline HTML
# ─────────────────────────────────────────────────────────────────────────────
def _img_to_data_uri(path: str) -> str:
    """Return a base64 data-URI for the image at *path*, or empty string."""
    if not path or not os.path.exists(path):
        return ""
    ext = os.path.splitext(path)[1].lower().lstrip(".")
    mime = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg"}.get(ext, "image/png")
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime};base64,{encoded}"


# ─────────────────────────────────────────────────────────────────────────────
# REPORT HTML BUILDER — FIX-3 (bold section labels) + FIX-4 (inline AE graph)
# ─────────────────────────────────────────────────────────────────────────────
def _build_report_html(report: dict, risk_score: float, risk_label: str,
                       status: str, graph_path: str = None) -> str:
    col = _risk_col(risk_score)

    def numfmt(v):
        return re.sub(r'([\d]+\.?[\d]*%?)', r"<span style='font-family:\"Times New Roman\",Times,serif;'>\1</span>", str(v))

    def row(label, value, mono=False):
        vd = numfmt(str(value))
        vfont = "font-family:'Roboto Mono',monospace;font-size:.8rem;" if mono else "font-size:.82rem;"
        return (f"<tr>"
                f"<td style='padding:9px 14px;color:#aaa;font-size:.78rem;"
                f"border-bottom:1px solid #222;white-space:nowrap;width:38%;'>{label}</td>"
                f"<td style='padding:9px 14px;color:#eee;{vfont}"
                f"border-bottom:1px solid #222;'>{vd}</td>"
                f"</tr>")

    # FIX-3: section headers are now font-weight:800 and slightly larger
    def section_hdr(title):
        return (f"<div style='font-size:.65rem;letter-spacing:.16em;text-transform:uppercase;"
                f"color:#FF8C00;padding:14px 0 6px 2px;border-bottom:2px solid #FF8C00;"
                f"margin-bottom:0;font-family:\"Roboto Mono\",monospace;font-weight:800;"
                f"-webkit-font-smoothing:antialiased;'>{title}</div>")

    def section(title, rows_html, extra=""):
        return (f"<div style='margin-bottom:26px;'>"
                + section_hdr(title)
                + f"<table style='width:100%;border-collapse:collapse;'>{rows_html}</table>"
                + extra
                + "</div>")

    # ── Risk badge ────────────────────────────────────────────────────────────
    is_crit = risk_score >= 0.70
    badge_text = f"{'ANOMALY DETECTED' if is_crit else 'NO ANOMALY DETECTED'} ({status})"
    badge = (f"<div style='text-align:center;padding:28px 20px;margin-bottom:28px;"
             f"background:{col};border-radius:4px;'>"
             f"<div style='font-size:3.8rem;font-weight:700;color:#fff;"
             f"font-family:\"Times New Roman\",Times,serif;line-height:1;letter-spacing:.02em;'>"
             f"{risk_score:.1%}</div>"
             f"<div style='color:rgba(255,255,255,.85);font-size:.75rem;margin-top:10px;"
             f"letter-spacing:.14em;text-transform:uppercase;"
             f"font-family:\"Roboto Mono\",monospace;font-weight:600;"
             f"-webkit-font-smoothing:antialiased;'>"
             f"{badge_text} &nbsp;|&nbsp; {risk_label} RISK</div>"
             f"</div>")

    # ── Pull raw chart data ───────────────────────────────────────────────────
    stage1_conf  = float(report.get("_stage1_conf", 0))
    stage2_conf  = float(report.get("_stage2_conf", 0))
    top3_raw     = report.get("_stage2_top3_raw", []) or []
    all_probs    = report.get("_stage2_all_probs", {}) or {}
    fused_risk   = float(report.get("_fused_risk", risk_score))
    audio_score  = float(report.get("_audio_score", 0))
    motion       = float(report.get("_motion", 0))

    # ── Donut pie chart (SVG) ─────────────────────────────────────────────────
    PIE_COLORS = ["#c0392b","#2471a3","#d35400","#6c3483","#117a65","#1a5276","#7d6608"]

    def _donut_pie_svg(all_probs_dict):
        if not all_probs_dict:
            return ""
        import math
        items = sorted(all_probs_dict.items(), key=lambda x: x[1], reverse=True)
        labels = [i[0] for i in items]
        sizes  = [max(float(i[1]), 1e-6) for i in items]
        total  = sum(sizes)
        cx, cy, R, r = 90, 90, 70, 36
        wedges = ""
        start_angle = -90
        for idx, (label, val) in enumerate(zip(labels, sizes)):
            sweep = (val / total) * 360
            end_angle = start_angle + sweep
            x1o = cx + R * math.cos(math.radians(start_angle))
            y1o = cy + R * math.sin(math.radians(start_angle))
            x2o = cx + R * math.cos(math.radians(end_angle))
            y2o = cy + R * math.sin(math.radians(end_angle))
            x1i = cx + r * math.cos(math.radians(end_angle))
            y1i = cy + r * math.sin(math.radians(end_angle))
            x2i = cx + r * math.cos(math.radians(start_angle))
            y2i = cy + r * math.sin(math.radians(start_angle))
            lg = 1 if sweep > 180 else 0
            fill = PIE_COLORS[idx % len(PIE_COLORS)]
            mid_a = start_angle + sweep / 2
            lx = cx + (R + r) / 2 * math.cos(math.radians(mid_a))
            ly = cy + (R + r) / 2 * math.sin(math.radians(mid_a))
            num_lbl = (f"<text x='{lx:.1f}' y='{ly:.1f}' text-anchor='middle' "
                       f"dominant-baseline='middle' fill='#fff' font-family='Helvetica,sans-serif' "
                       f"font-size='9' font-weight='bold'>{idx+1}</text>") if sweep > 15 else ""
            wedges += (f"<path d='M {x1o:.2f} {y1o:.2f} A {R} {R} 0 {lg} 1 {x2o:.2f} {y2o:.2f} "
                       f"L {x1i:.2f} {y1i:.2f} A {r} {r} 0 {lg} 0 {x2i:.2f} {y2i:.2f} Z' "
                       f"fill='{fill}' stroke='#000' stroke-width='1'/>{num_lbl}")
            start_angle = end_angle
        legend_rows = ""
        for idx, (cls, prob) in enumerate(zip(labels, sizes)):
            fc = PIE_COLORS[idx % len(PIE_COLORS)]
            legend_rows += (f"<div style='display:flex;align-items:center;gap:7px;margin-bottom:5px;'>"
                            f"<div style='width:11px;height:11px;min-width:11px;background:{fc};border-radius:2px;'></div>"
                            f"<span style='font-family:\"Roboto Mono\",monospace;font-size:.62rem;"
                            f"color:#ccc;-webkit-font-smoothing:antialiased;'>"
                            f"{idx+1}. {cls}&nbsp;"
                            f"<span style='font-family:\"Times New Roman\",Times,serif;color:#fff;font-weight:700;'>"
                            f"({prob/total*100:.1f}%)</span></span>"
                            f"</div>")
        svg = (f"<svg width='180' height='180' viewBox='0 0 180 180' xmlns='http://www.w3.org/2000/svg'>"
               f"{wedges}"
               f"<circle cx='{cx}' cy='{cy}' r='{r-2}' fill='#050505'/>"
               f"<text x='{cx}' y='{cy}' text-anchor='middle' dominant-baseline='middle' "
               f"fill='#666' font-family='Roboto Mono,monospace' font-size='8'>ANOMALY</text>"
               f"<text x='{cx}' y='{cy+11}' text-anchor='middle' dominant-baseline='middle' "
               f"fill='#666' font-family='Roboto Mono,monospace' font-size='8'>CLASSES</text>"
               f"</svg>")
        return (f"<div style='display:flex;gap:16px;align-items:flex-start;"
                f"padding:16px;background:#050505;border:1px solid #1e1e1e;border-radius:3px;margin-top:6px;'>"
                f"<div style='flex-shrink:0;'>{svg}"
                f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.56rem;color:#555;"
                f"text-align:center;margin-top:4px;letter-spacing:.08em;'>See legend for %</div>"
                f"</div>"
                f"<div style='flex:1;'>{legend_rows}</div>"
                f"</div>")

    # ── Horizontal confidence bar chart ───────────────────────────────────────
    def _conf_bar_chart(top3):
        if not top3:
            return ""
        BAR_COLORS = ["#c0392b","#d35400","#5d6d7e"]
        rows = ""
        for idx, (cls_name, prob) in enumerate(top3[:3]):
            pct = prob * 100
            bc  = BAR_COLORS[idx]
            rows += (f"<div style='margin-bottom:10px;'>"
                     f"<div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:4px;'>"
                     f"<span style='font-family:\"Roboto Mono\",monospace;font-size:.68rem;color:#ccc;"
                     f"-webkit-font-smoothing:antialiased;'>{cls_name}</span>"
                     f"<span style='font-family:\"Times New Roman\",Times,serif;color:#fff;font-weight:700;"
                     f"font-size:.82rem;margin-left:8px;'>{pct:.1f}%</span>"
                     f"</div>"
                     f"<div style='background:#1a1a1a;border-radius:2px;height:10px;overflow:hidden;'>"
                     f"<div style='background:{bc};height:10px;border-radius:2px;"
                     f"width:{pct:.1f}%;transition:width .8s ease;'></div>"
                     f"</div></div>")
        return (f"<div style='padding:14px 16px;background:#050505;"
                f"border:1px solid #1e1e1e;border-radius:3px;margin-top:6px;'>"
                f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.6rem;"
                f"letter-spacing:.14em;color:#FF8C00;text-transform:uppercase;margin-bottom:12px;"
                f"-webkit-font-smoothing:antialiased;'>Top-3 Anomaly Predictions</div>"
                f"{rows}</div>")

    # ── Signal gauge bars ─────────────────────────────────────────────────────
    def risk_gauge(val, label, color):
        pct = min(100, int(val * 100))
        return (f"<div style='margin-bottom:10px;'>"
                f"<div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:3px;'>"
                f"<span style='font-family:\"Roboto Mono\",monospace;font-size:.65rem;color:#aaa;"
                f"-webkit-font-smoothing:antialiased;'>{label}</span>"
                f"<span style='font-family:\"Times New Roman\",Times,serif;color:#fff;font-weight:700;"
                f"font-size:.8rem;'>{val:.1%}</span>"
                f"</div>"
                f"<div style='background:#1a1a1a;border-radius:2px;height:8px;overflow:hidden;'>"
                f"<div style='background:{color};height:8px;border-radius:2px;width:{pct}%;'></div>"
                f"</div></div>")

    # ── Small donut pies ──────────────────────────────────────────────────────
    def _small_donut(pct, color, label, size=84):
        import math
        cx = cy = size // 2
        R = cx - 8
        r = R - 12
        sweep = pct * 360
        end_a = -90 + sweep
        x1 = cx + R * math.cos(math.radians(-90))
        y1 = cy + R * math.sin(math.radians(-90))
        x2 = cx + R * math.cos(math.radians(end_a))
        y2 = cy + R * math.sin(math.radians(end_a))
        xi1 = cx + r * math.cos(math.radians(end_a))
        yi1 = cy + r * math.sin(math.radians(end_a))
        xi2 = cx + r * math.cos(math.radians(-90))
        yi2 = cy + r * math.sin(math.radians(-90))
        lg = 1 if sweep > 180 else 0
        if pct >= 0.999:
            arc = (f"<circle cx='{cx}' cy='{cy}' r='{R}' fill='none' stroke='{color}' stroke-width='12'/>"
                   f"<circle cx='{cx}' cy='{cy}' r='{r}' fill='#050505'/>")
        else:
            arc = (f"<circle cx='{cx}' cy='{cy}' r='{R}' fill='none' stroke='#1a1a1a' stroke-width='12'/>"
                   f"<path d='M {x1:.2f} {y1:.2f} A {R} {R} 0 {lg} 1 {x2:.2f} {y2:.2f} "
                   f"L {xi1:.2f} {yi1:.2f} A {r} {r} 0 {lg} 0 {xi2:.2f} {yi2:.2f} Z' fill='{color}'/>"
                   f"<circle cx='{cx}' cy='{cy}' r='{r}' fill='#050505'/>")
        return (f"<div style='text-align:center;'>"
                f"<svg width='{size}' height='{size}' viewBox='0 0 {size} {size}' xmlns='http://www.w3.org/2000/svg'>"
                f"{arc}"
                f"<text x='{cx}' y='{cy}' text-anchor='middle' dominant-baseline='middle' "
                f"fill='#fff' font-family='Times New Roman,Times,serif' font-size='11' font-weight='700'>"
                f"{pct:.0%}</text>"
                f"</svg>"
                f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.55rem;color:#666;"
                f"margin-top:2px;-webkit-font-smoothing:antialiased;'>{label}</div>"
                f"</div>")

    charts_cls = (f"<div style='display:flex;gap:20px;padding:14px 16px;background:#050505;"
                  f"border:1px solid #1e1e1e;border-radius:3px;margin-top:6px;align-items:flex-start;flex-wrap:wrap;'>"
                  + _small_donut(stage1_conf, '#FF8C00', 'TF BINARY · Stage 1')
                  + _small_donut(stage2_conf, '#c0392b', 'ANOMALY · Stage 2')
                  + f"<div style='flex:1;min-width:200px;'>{_conf_bar_chart(top3_raw)}</div>"
                  + (f"<div style='width:100%;margin-top:10px;'>{_donut_pie_svg(all_probs)}</div>" if all_probs else "")
                  + "</div>")

    charts_sensing = (f"<div style='padding:14px 16px;background:#050505;"
                      f"border:1px solid #1e1e1e;border-radius:3px;margin-top:6px;'>"
                      f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.6rem;"
                      f"letter-spacing:.14em;color:#FF8C00;text-transform:uppercase;margin-bottom:12px;"
                      f"-webkit-font-smoothing:antialiased;'>Signal Intensities</div>"
                      + risk_gauge(fused_risk, "Fusion Risk (Stage 6)", _risk_col(fused_risk))
                      + risk_gauge(audio_score, "Audio Score (Stage 4)", "#2471a3")
                      + risk_gauge(min(motion, 1.0), "Motion Intensity (Stage 5)", "#117a65")
                      + "</div>")

    # ── Build table rows ──────────────────────────────────────────────────────
    ae_rows = (row("Overall Status",     report.get("overall_status",""))
             + row("Dynamic Threshold",  report.get("dynamic_threshold",""), mono=True)
             + row("AE Max Error",       report.get("ae_max_error",""),      mono=True)
             + row("AE Avg Error",       report.get("ae_avg_error",""),      mono=True)
             + row("Peak Timestamp",     report.get("peak_timestamp",""),    mono=True)
             + row("Anomaly Window",     report.get("anomaly_window",""),    mono=True))

    cl_rows  = (row("TF Binary (Stage 1)",   report.get("stage1_tf_binary",""))
              + row("Anomaly Type (Stage 2)", report.get("stage2_anomaly_type","")))
    top3 = report.get("stage2_top3", [])
    if top3:
        for i, (cls, prob) in enumerate(top3, 1):
            cl_rows += row(f"  Top-{i}", f"{cls}  ({prob:.2%})", mono=True)

    det_rows = (row("Weapon Detection (Stage 3)", report.get("stage3_weapon",""))
              + row("Audio Status (Stage 4)",      report.get("stage4_audio",""))
              + row("Motion (Stage 5)",            report.get("stage5_motion","")))

    fus_rows = row("Fusion Risk (Stage 6)", report.get("stage6_fusion_risk",""))
    for i, line in enumerate(report.get("stage6_reasoning",[]), 1):
        fus_rows += row(f"  Step {i}", line)

    rl_rows = (row("RL3033 Final Risk (Stage 7)", report.get("stage7_rl_final",""), mono=True)
             + row("Interpretation", report.get("stage7_interpretation","")))

    narr = report.get("llava_narrative","")
    narr_html = ""
    if narr:
        narr_html = (f"<div style='margin-bottom:26px;'>"
                     + section_hdr("AI Visual Narrative (LLaVA — Stage 8)")
                     + f"<div style='color:#ccc;font-size:.85rem;line-height:1.78;"
                     f"padding:16px 18px;background:#050505;border:1px solid #1e1e1e;"
                     f"border-radius:3px;margin-top:6px;'>{narr}</div></div>")

    # ── FIX-4: AE Deviation Graph inline block ────────────────────────────────
    ae_graph_html = ""
    if graph_path:
        data_uri = _img_to_data_uri(graph_path)
        if data_uri:
            ae_graph_html = (
                f"<div style='margin-bottom:26px;'>"
                + section_hdr("AE Deviation Graph — Temporal Reconstruction Error")
                + f"<div style='padding:14px;background:#050505;border:1px solid #1e1e1e;"
                f"border-radius:3px;margin-top:6px;'>"
                f"<img src='{data_uri}' alt='AE Deviation Graph' "
                f"style='width:100%;border-radius:2px;display:block;'/>"
                f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.58rem;"
                f"color:#555;margin-top:8px;letter-spacing:.08em;text-align:center;'>"
                f"Red line = reconstruction error &nbsp;·&nbsp; "
                f"Green dashed = dynamic threshold &nbsp;·&nbsp; "
                f"Star = peak anomaly frame"
                f"</div></div></div>"
            )

    return (f"<div style='background:#0a0a0a;padding:28px 26px;border:1px solid #1e1e1e;border-radius:4px;'>"
            f"{badge}"
            + ae_graph_html
            + section("Autoencoder — Stage 0", ae_rows)
            + section("Classification — Stages 1 & 2", cl_rows, charts_cls)
            + section("Detection & Sensing — Stages 3, 4, 5", det_rows, charts_sensing)
            + section("Fusion Layer — Stage 6", fus_rows)
            + section("RL3033 Temporal Reasoning — Stage 7", rl_rows)
            + narr_html
            + "</div>")


def process_video(video):
    if video is None:
        return (_no_result_html(), None, None, None)
    try:
        out   = run_pipeline(video)
        r     = out["risk_score"]
        report_html = _build_report_html(
            out["report"], r, out["risk_label"], out["status"],
            graph_path=out.get("graph_path"))    # FIX-4: pass graph_path
        fi = Image.fromarray(out["top_frame"]) if out["top_frame"] is not None else None
        jd_str = json.dumps(out["json_data"], indent=2, default=str)
        return (report_html, fi, jd_str, out["pdf_path"])
    except Exception:
        tb = traceback.format_exc()
        err_html = (f"<div style='color:#FF4500;font-family:\"Roboto Mono\",monospace;"
                    f"font-size:.8rem;padding:20px;background:#050505;'>"
                    f"Analysis failed:\n\n{tb}</div>")
        return (err_html, None, None, None)


def _no_result_html():
    return ("<div style='text-align:center;padding:60px 20px;color:#333;"
            "font-family:Charter,serif;font-size:1rem;letter-spacing:.04em;'>"
            "Upload footage and press ANALYSE to begin.</div>")


# ─────────────────────────────────────────────────────────────────────────────
# SPINNER HTML helper
# ─────────────────────────────────────────────────────────────────────────────
def _spinner_html():
    return """
<div style="text-align:center;padding:18px 0;">
  <div style="display:inline-block;width:40px;height:40px;
    border:3px solid rgba(255,255,255,0.07);
    border-top-color:#FF8C00;
    border-right-color:rgba(255,140,0,0.28);
    border-radius:50%;
    animation:arc-spin .75s cubic-bezier(0.6,0.2,0.4,0.8) infinite;">
  </div>
  <div style="font-family:'Roboto Mono',monospace;font-size:.64rem;
    letter-spacing:.12em;color:#FF8C00;margin-top:11px;
    text-transform:uppercase;-webkit-font-smoothing:antialiased;font-weight:600;">
    Processing&hellip; this may take several minutes
  </div>
</div>"""


# ─────────────────────────────────────────────────────────────────────────────
# DISCLAIMER BANNER — FIX-5: consistent across all pages
# ─────────────────────────────────────────────────────────────────────────────
_DISCLAIMER_BANNER = """
<div style="background:#0d0d0d;border-top:1px solid #1a1a1a;border-bottom:1px solid #1a1a1a;
            padding:14px 32px;margin:0 0 0 0;">
  <div style="max-width:860px;margin:0 auto;display:flex;align-items:flex-start;gap:12px;">
    <svg style="flex-shrink:0;margin-top:2px;" width="15" height="15" viewBox="0 0 24 24"
         fill="none" stroke="#FF8C00" stroke-width="2">
      <circle cx="12" cy="12" r="10"/>
      <line x1="12" y1="8" x2="12" y2="12"/>
      <line x1="12" y1="16" x2="12.01" y2="16"/>
    </svg>
    <p style="font-family:'Roboto Mono',monospace;font-size:.62rem;
              letter-spacing:.04em;color:#888;line-height:1.7;margin:0;
              font-weight:500;-webkit-font-smoothing:antialiased;">
      <span style="color:#FF8C00;font-weight:700;">DISCLAIMER&nbsp;&mdash;&nbsp;</span>
      This report is produced by machine learning models and should be treated as an
      investigative aid, not a definitive conclusion. Human expert review is strongly
      recommended before any operational action is taken.
    </p>
  </div>
</div>
"""

# ─────────────────────────────────────────────────────────────────────────────
# CSS — FIX-1: tab spacing improved
# ─────────────────────────────────────────────────────────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Roboto+Mono:wght@300;400;500;600;700&display=swap');

:root {
  --bg:      #000000;
  --surface: #080808;
  --border:  #1e1e1e;
  --white:   #FFFFFF;
  --dim:     #888888;
  --orange:  #FF8C00;
  --red:     #FF4500;
  --mono:    'Roboto Mono', monospace;
  --serif:   Charter, 'Bitstream Charter', Georgia, serif;
}

*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

html, body, .gradio-container, .gradio-container > * {
  background: var(--bg) !important;
  color: var(--white) !important;
  font-family: var(--serif) !important;
}

* { -webkit-font-smoothing: antialiased; -moz-osx-font-smoothing: grayscale; }

::-webkit-scrollbar { width: 4px; height: 4px; }
::-webkit-scrollbar-track { background: var(--bg); }
::-webkit-scrollbar-thumb { background: #333; border-radius: 2px; }

@keyframes arc-spin {
  from { transform: rotate(0deg); }
  to   { transform: rotate(360deg); }
}

/* ── FIX-1: TABS — evenly spaced, clear separation ── */
.tab-nav {
  background: var(--bg) !important;
  border-bottom: 1px solid var(--border) !important;
  padding: 0 32px !important;
  display: flex !important;
  gap: 8px !important;           /* explicit gap between tab buttons */
}
.tab-nav button {
  font-family: var(--mono) !important;
  font-size: .72rem !important;
  letter-spacing: .13em !important;
  color: #999 !important;
  background: transparent !important;
  border: none !important;
  border-bottom: 2px solid transparent !important;
  padding: 14px 32px !important;  /* generous horizontal padding for each tab */
  text-transform: uppercase !important;
  transition: color .18s, border-color .18s !important;
  font-weight: 600 !important;
  -webkit-font-smoothing: antialiased !important;
  white-space: nowrap !important;
  flex: 0 0 auto !important;      /* prevent tabs from shrinking into each other */
}
.tab-nav button:hover { color: #ddd !important; }
.tab-nav button.selected {
  color: #FFFFFF !important;
  border-bottom-color: var(--orange) !important;
  font-weight: 700 !important;
}

/* ── GRADIO INTERNALS ── */
.gradio-container .block, .gradio-container .panel,
.gradio-container .form, .gradio-container fieldset,
.gradio-container .wrap, .gradio-container .container,
.gradio-container > div { background: var(--bg) !important; border: none !important; }

/* ── INPUTS ── */
input, textarea, select {
  background: #0d0d0d !important;
  border: 1px solid #2a2a2a !important;
  border-radius: 3px !important;
  color: #f0f0f0 !important;
  font-family: var(--serif) !important;
  font-size: .88rem !important;
  padding: 12px 14px !important;
  transition: border-color .18s !important;
  width: 100% !important;
}
input:focus, textarea:focus {
  border-color: var(--orange) !important;
  outline: none !important;
  box-shadow: 0 0 0 1px var(--orange) !important;
}
label, .label-wrap span {
  font-family: var(--mono) !important;
  font-size: .65rem !important;
  letter-spacing: .1em !important;
  text-transform: uppercase !important;
  color: #ccc !important;
  font-weight: 600 !important;
}

/* ── BUTTONS ── */
.gr-button {
  background: var(--white) !important;
  color: var(--bg) !important;
  border-radius: 3px !important;
  font-family: var(--mono) !important;
  font-size: .72rem !important;
  letter-spacing: .12em !important;
  font-weight: 700 !important;
  -webkit-font-smoothing: antialiased !important;
}
.gr-button:hover { background: var(--orange) !important; }

#ana-btn, #ana-btn button {
  width: auto !important;
  max-width: 260px !important;
  min-height: 44px !important;
  height: 44px !important;
  padding: 0 30px !important;
}

/* ── VIDEO UPLOAD ── */
#vid-up { min-height: 150px !important; max-height: 200px !important; }
#vid-up .upload-container, #vid-up .wrap {
  background: #060606 !important;
  border: 1px dashed #2a2a2a !important;
  border-radius: 3px !important;
  min-height: 150px !important;
  max-height: 200px !important;
}
#vid-up .wrap:hover { border-color: var(--orange) !important; }

/* ── REPORT HTML ── */
#report-html { background: var(--bg) !important; }

/* ── IMAGE ── */
#kf-img, #kf-img img {
  border-radius: 3px !important;
  border: 1px solid var(--border) !important;
  width: 100% !important;
}

/* ── JSON ── */
#json-out, #json-out textarea {
  background: #060606 !important;
  color: #bbb !important;
  font-family: var(--mono) !important;
  font-size: .75rem !important;
  line-height: 1.7 !important;
  border: 1px solid var(--border) !important;
  border-radius: 3px !important;
}

/* ── PDF FILE ── */
#pdf-out { border: 1px solid var(--border) !important; border-radius: 3px !important; }
#pdf-out .file-preview { background: #060606 !important; }

/* ── DIVIDER ── */
.rule { border: none; border-top: 1px solid #111; margin: 28px 0; }

/* ── CONTACT FORM ── */
.contact-card {
  background: #080808;
  border: 1px solid #1e1e1e;
  border-radius: 4px;
  padding: 28px 28px 20px;
}
.contact-info-block {
  background: #060606;
  border: 1px solid #222;
  border-radius: 3px;
  padding: 18px 20px;
  margin-top: 16px;
}

footer { display: none !important; }
"""

# ─────────────────────────────────────────────────────────────────────────────
# STATIC HTML — NAV
# ─────────────────────────────────────────────────────────────────────────────
_NAV = """
<div style="background:#000;border-bottom:1px solid #1a1a1a;
            padding:16px 36px;display:flex;justify-content:space-between;align-items:center;">
  <span style="font-family:'Roboto Mono',monospace;font-size:.92rem;
               letter-spacing:.15em;color:#fff;text-transform:uppercase;
               font-weight:700;-webkit-font-smoothing:antialiased;">
    CrimeSceneSolver
  </span>
  <span style="font-family:'Roboto Mono',monospace;font-size:.6rem;
               letter-spacing:.1em;color:#555;text-transform:uppercase;
               font-weight:500;-webkit-font-smoothing:antialiased;">
    Scene Solver v3
  </span>
</div>
"""

# ─────────────────────────────────────────────────────────────────────────────
# STATIC HTML — HOME
# FIX-2: Analyse Footage button uses a more robust multi-strategy click handler
# ─────────────────────────────────────────────────────────────────────────────
_HOME = """
<div style="max-width:860px;margin:0 auto;padding:0 28px 64px;">

  <!-- HERO -->
  <div style="padding:80px 0 64px;border-bottom:1px solid #111;">
    <div style="font-family:'Roboto Mono',monospace;font-size:.64rem;
                letter-spacing:.18em;color:#FF8C00;text-transform:uppercase;
                margin-bottom:18px;font-weight:600;-webkit-font-smoothing:antialiased;">
      Forensic AI Platform
    </div>
    <h1 style="font-family:Charter,'Bitstream Charter',Georgia,serif;
               font-size:3.4rem;font-weight:400;color:#fff;line-height:1.1;
               margin-bottom:20px;">
      Uncover the Fact.
    </h1>
    <p style="font-family:Charter,'Bitstream Charter',Georgia,serif;
              font-size:1.06rem;color:#777;max-width:520px;line-height:1.74;
              margin-bottom:38px;">
      An intelligent, secure platform for modern crime scene investigation.
      Multi-stage AI pipeline — Autoencoder, TimeSformer, YOLO, RL3033, LLaVA —
      for faster, more accurate case resolution.
    </p>

    <!-- FIX-2: Multi-strategy tab navigation button -->
    <button
      onclick="(function(){
        /* Strategy 1: match by button text containing 'analyse' */
        var found = false;
        var allBtns = document.querySelectorAll('.tab-nav button, [role=tab]');
        for (var i = 0; i < allBtns.length; i++) {
          var txt = (allBtns[i].textContent || allBtns[i].innerText || '').trim().toLowerCase();
          if (txt.indexOf('analyse') !== -1 || txt.indexOf('analyze') !== -1) {
            allBtns[i].click();
            found = true;
            break;
          }
        }
        /* Strategy 2: fallback — click the second tab (index 1) */
        if (!found) {
          var tabs = document.querySelectorAll('.tab-nav button, [role=tab]');
          if (tabs.length > 1) { tabs[1].click(); found = true; }
        }
        /* Strategy 3: fallback — click any button whose id contains 'analyse' */
        if (!found) {
          var byId = document.querySelector('[id*=analyse], [id*=analyze]');
          if (byId) byId.click();
        }
      })()"
      style="background:#fff;color:#000;font-family:'Roboto Mono',monospace;
             font-size:.74rem;letter-spacing:.14em;text-transform:uppercase;
             padding:14px 36px;border-radius:3px;border:none;cursor:pointer;
             font-weight:700;-webkit-font-smoothing:antialiased;
             transition:background .2s,color .2s;"
      onmouseover="this.style.background='#FF8C00'"
      onmouseout="this.style.background='#fff'">
      Analyse Footage &rarr;
    </button>
  </div>

  <!-- CAPABILITIES -->
  <div style="padding:56px 0 48px;border-bottom:1px solid #111;">
    <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;
                letter-spacing:.16em;color:#FF8C00;text-transform:uppercase;
                margin-bottom:34px;font-weight:600;-webkit-font-smoothing:antialiased;">Capabilities</div>
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px;">

      <div style="padding:26px 22px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="width:30px;height:30px;margin-bottom:16px;">
          <svg viewBox="0 0 24 24" fill="none" stroke="#FF8C00" stroke-width="1.5">
            <path d="M9 5H7a2 2 0 00-2 2v12a2 2 0 002 2h10a2 2 0 002-2V7a2 2 0 00-2-2h-2"/>
            <rect x="9" y="3" width="6" height="4" rx="1"/>
            <path d="M9 12h6M9 16h4"/>
          </svg>
        </div>
        <div style="font-family:Charter,Georgia,serif;font-size:.95rem;color:#fff;margin-bottom:8px;">Evidence Tracker</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:1.65;">
          Securely log and trace every piece of evidence from scene to court with immutable audit trails.
        </div>
      </div>

      <div style="padding:26px 22px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="width:30px;height:30px;margin-bottom:16px;">
          <svg viewBox="0 0 24 24" fill="none" stroke="#FF8C00" stroke-width="1.5">
            <circle cx="11" cy="11" r="8"/><path d="M21 21l-4.35-4.35M11 8v6M8 11h6"/>
          </svg>
        </div>
        <div style="font-family:Charter,Georgia,serif;font-size:.95rem;color:#fff;margin-bottom:8px;">AI Pattern Detection</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:1.65;">
          Uncover hidden connections and predict crime hotspots with nine-stage forensic reasoning.
        </div>
      </div>

      <div style="padding:26px 22px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="width:30px;height:30px;margin-bottom:16px;">
          <svg viewBox="0 0 24 24" fill="none" stroke="#FF8C00" stroke-width="1.5">
            <path d="M17 21v-2a4 4 0 00-4-4H5a4 4 0 00-4 4v2"/>
            <circle cx="9" cy="7" r="4"/>
            <path d="M23 21v-2a4 4 0 00-3-3.87M16 3.13a4 4 0 010 7.75"/>
          </svg>
        </div>
        <div style="font-family:Charter,Georgia,serif;font-size:.95rem;color:#fff;margin-bottom:8px;">Live Collaboration</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:1.65;">
          Work in real time with your team across locations, sharing critical case information securely.
        </div>
      </div>
    </div>
  </div>

  <!-- HOW IT WORKS -->
  <div style="padding:56px 0 48px;border-bottom:1px solid #111;">
    <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;
                letter-spacing:.16em;color:#FF8C00;text-transform:uppercase;
                margin-bottom:34px;font-weight:600;-webkit-font-smoothing:antialiased;">How It Works</div>
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:36px;">
      <div>
        <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;color:#444;margin-bottom:10px;font-weight:600;">01</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.95rem;color:#fff;margin-bottom:8px;">Upload Case Data</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:1.65;">Digitize and securely upload CCTV footage and case files.</div>
      </div>
      <div>
        <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;color:#444;margin-bottom:10px;font-weight:600;">02</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.95rem;color:#fff;margin-bottom:8px;">Analyse Evidence</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:1.65;">Nine AI stages process the footage in parallel for a complete forensic profile.</div>
      </div>
      <div>
        <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;color:#444;margin-bottom:10px;font-weight:600;">03</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.95rem;color:#fff;margin-bottom:8px;">Resolve Cases</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:1.65;">Receive structured reports, risk scores, and AI narratives to inform decisions.</div>
      </div>
    </div>
  </div>

  <!-- PRICING -->
  <div style="padding:56px 0 48px;border-bottom:1px solid #111;">
    <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;
                letter-spacing:.16em;color:#FF8C00;text-transform:uppercase;
                margin-bottom:34px;font-weight:600;-webkit-font-smoothing:antialiased;">Pricing</div>
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px;">
      <div style="padding:26px 22px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.64rem;letter-spacing:.1em;color:#666;margin-bottom:12px;font-weight:600;">BASIC</div>
        <div style="font-size:2rem;font-weight:400;color:#fff;margin-bottom:16px;font-family:'Times New Roman',Times,serif;">Free</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:2.1;">Evidence Tracker<br>5 analyses / month<br>PDF reports</div>
      </div>
      <div style="padding:26px 22px;background:#080808;border:1px solid #FF8C00;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.64rem;letter-spacing:.1em;color:#FF8C00;margin-bottom:12px;font-weight:600;">PRO</div>
        <div style="font-size:2rem;font-weight:400;color:#fff;margin-bottom:16px;font-family:'Times New Roman',Times,serif;">$49/mo</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#888;line-height:2.1;">AI Pattern Detection<br>Unlimited analyses<br>LLaVA narratives<br>Priority support</div>
      </div>
      <div style="padding:26px 22px;background:#080808;border:1px solid #1e1e1e;border-radius:3px;">
        <div style="font-family:'Roboto Mono',monospace;font-size:.64rem;letter-spacing:.1em;color:#666;margin-bottom:12px;font-weight:600;">ENTERPRISE</div>
        <div style="font-size:2rem;font-weight:400;color:#fff;margin-bottom:16px;font-family:'Times New Roman',Times,serif;">Custom</div>
        <div style="font-family:Charter,Georgia,serif;font-size:.8rem;color:#555;line-height:2.1;">Custom models<br>Dedicated support<br>SLA guarantee</div>
      </div>
    </div>
  </div>

  <!-- TESTIMONIAL -->
  <div style="padding:56px 0 24px;">
    <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;letter-spacing:.16em;
                color:#FF8C00;text-transform:uppercase;margin-bottom:26px;
                font-weight:600;-webkit-font-smoothing:antialiased;">From the Field</div>
    <blockquote style="font-family:Charter,'Bitstream Charter',Georgia,serif;
                       font-size:1.1rem;color:#aaa;line-height:1.74;
                       border-left:2px solid #FF8C00;padding-left:22px;
                       font-style:italic;margin:0 0 14px;">
      "We closed a complex 6-month investigation in under 2 weeks after implementing
      SceneSolver. The AI-driven analysis and real-time collaboration features
      were a game changer for our team."
    </blockquote>
    <div style="font-family:'Roboto Mono',monospace;font-size:.68rem;letter-spacing:.08em;
                color:#666;font-weight:500;-webkit-font-smoothing:antialiased;">
      — Detective Lara Monroe, Forensics Division
    </div>
  </div>

</div>
"""

# FIX-5: Footer now contains the full disclaimer text (used on every page)
_FOOTER = """
<div style="border-top:1px solid #0d0d0d;margin-top:16px;">
  <!-- Disclaimer banner -->
  <div style="background:#0d0d0d;border-bottom:1px solid #1a1a1a;padding:14px 32px;">
    <div style="max-width:860px;margin:0 auto;display:flex;align-items:flex-start;gap:12px;">
      <svg style="flex-shrink:0;margin-top:2px;" width="14" height="14" viewBox="0 0 24 24"
           fill="none" stroke="#FF8C00" stroke-width="2">
        <circle cx="12" cy="12" r="10"/>
        <line x1="12" y1="8" x2="12" y2="12"/>
        <line x1="12" y1="16" x2="12.01" y2="16"/>
      </svg>
      <p style="font-family:'Roboto Mono',monospace;font-size:.62rem;
                letter-spacing:.04em;color:#888;line-height:1.7;margin:0;
                font-weight:500;-webkit-font-smoothing:antialiased;">
        <span style="color:#FF8C00;font-weight:700;">DISCLAIMER&nbsp;&mdash;&nbsp;</span>
        This report is produced by machine learning models and should be treated as an
        investigative aid, not a definitive conclusion. Human expert review is strongly
        recommended before any operational action is taken.
      </p>
    </div>
  </div>
  <!-- Standard footer line -->
  <div style="text-align:center;padding:18px;
              font-family:'Roboto Mono',monospace;font-size:.6rem;
              letter-spacing:.1em;color:#444;text-transform:uppercase;
              font-weight:500;-webkit-font-smoothing:antialiased;">
    CrimeSceneSolver &nbsp;&middot;&nbsp; Scene Solver v3 &nbsp;&middot;&nbsp;
    Confidential &nbsp;&middot;&nbsp; ML outputs require expert review
  </div>
</div>
"""

# ─────────────────────────────────────────────────────────────────────────────
# SECTION LABEL HELPER — FIX-3: bolder label style
# ─────────────────────────────────────────────────────────────────────────────
def _sec_label(text):
    return (f"<div style='font-family:\"Roboto Mono\",monospace;font-size:.62rem;"
            f"letter-spacing:.14em;text-transform:uppercase;color:#ddd;"
            f"margin-bottom:10px;font-weight:800;-webkit-font-smoothing:antialiased;"
            f"border-left:3px solid #FF8C00;padding-left:10px;'>{text}</div>")


# ─────────────────────────────────────────────────────────────────────────────
# BUILD APP
# ─────────────────────────────────────────────────────────────────────────────
with gr.Blocks(css=CSS, title="CrimeSceneSolver") as demo:

    gr.HTML(_NAV)

    with gr.Tabs():

        # ══════════════════════════════════════════════════════════
        # HOME
        # ══════════════════════════════════════════════════════════
        with gr.TabItem("Home"):
            gr.HTML(_HOME)
            gr.HTML(_FOOTER)   # FIX-5: disclaimer footer on Home page

        # ══════════════════════════════════════════════════════════
        # ANALYSE
        # ══════════════════════════════════════════════════════════
        with gr.TabItem("Analyse"):
            gr.HTML("""
            <div style="max-width:780px;margin:0 auto;padding:48px 28px 0;">
              <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;
                          letter-spacing:.16em;color:#FF8C00;text-transform:uppercase;
                          margin-bottom:12px;font-weight:600;-webkit-font-smoothing:antialiased;">Forensic Analysis</div>
              <h2 style="font-family:Charter,Georgia,serif;font-size:1.9rem;
                         font-weight:400;color:#fff;margin-bottom:8px;">Case File Analysis</h2>
              <p style="font-family:Charter,Georgia,serif;font-size:.88rem;
                        color:#555;margin-bottom:28px;">
                Upload CCTV footage to run the full nine-stage forensic pipeline.
              </p>
              <hr class="rule">
            </div>
            """)

            with gr.Column():
                gr.HTML('<div style="max-width:780px;margin:0 auto;padding:0 28px;">')

                # Upload
                gr.HTML(_sec_label("Upload Footage"))
                vid_in = gr.Video(label="", elem_id="vid-up", sources=["upload"],
                                   show_label=False, height=175)

                gr.HTML('<div style="height:14px;"></div>')

                # Analyse button
                gr.HTML('<div style="display:flex;justify-content:flex-start;">')
                ana_btn = gr.Button("Analyse Footage", elem_id="ana-btn", elem_classes="gr-button")
                gr.HTML('</div>')

                # Spinner
                spinner_html = gr.HTML("")

                gr.HTML('<div style="height:28px;"></div>')

                # ── FIX-3: Risk Assessment & Report — bolder section label ──
                gr.HTML(_sec_label("Risk Assessment &amp; Report"))
                report_out = gr.HTML(
                    value=("<div style='text-align:center;padding:60px 0;"
                           "font-family:Charter,Georgia,serif;font-size:.9rem;"
                           "color:#2a2a2a;'>Upload footage and press Analyse.</div>"),
                    elem_id="report-html")

                gr.HTML('<div style="height:28px;"></div>')

                # ── Peak Anomaly Frame ───────────────────────────
                gr.HTML(_sec_label("Peak Anomaly Frame"))
                frame_out = gr.Image(label="", elem_id="kf-img", show_label=False)

                gr.HTML('<div style="height:28px;"></div>')

                # ── Download PDF ─────────────────────────────────
                gr.HTML(_sec_label("Download Forensic PDF Report"))
                pdf_out = gr.File(label="", elem_id="pdf-out",
                                   show_label=False, file_types=[".pdf"], interactive=False)
                gr.HTML("""
                <div style="font-family:'Roboto Mono',monospace;font-size:.6rem;
                            color:#444;line-height:1.7;padding:10px 0;
                            border-top:1px solid #0d0d0d;margin-top:6px;
                            font-weight:500;-webkit-font-smoothing:antialiased;">
                  11-page forensic dossier: AE temporal graph &middot; classification
                  charts &middot; weapon detections &middot; audio features &middot;
                  motion tracking &middot; fusion reasoning &middot; RL3033 output
                  &middot; anomalous frames &middot; LLaVA narrative
                </div>""")

                gr.HTML('<div style="height:28px;"></div>')

                # ── Structured JSON ──────────────────────────────
                gr.HTML(_sec_label("Structured JSON Output"))
                json_out = gr.Textbox(label="", elem_id="json-out",
                                       show_label=False, lines=26,
                                       placeholder="JSON output will appear here…")

                gr.HTML('</div>')

            # ── Wire up button ────────────────────────────────────
            def _show_spinner():
                return gr.update(value=_spinner_html())

            def _hide_spinner():
                return gr.update(value="")

            ana_btn.click(
                fn=_show_spinner, inputs=None, outputs=spinner_html, queue=False
            ).then(
                fn=process_video, inputs=[vid_in],
                outputs=[report_out, frame_out, json_out, pdf_out]
            ).then(
                fn=_hide_spinner, inputs=None, outputs=spinner_html, queue=False
            )

            gr.HTML(_FOOTER)   # FIX-5: disclaimer footer on Analyse page

        # ══════════════════════════════════════════════════════════
        # CONTACT
        # ══════════════════════════════════════════════════════════
        with gr.TabItem("Contact"):
            gr.HTML("""
            <div style="max-width:600px;margin:0 auto;padding:52px 28px 0;">
              <div style="font-family:'Roboto Mono',monospace;font-size:.62rem;
                          letter-spacing:.16em;color:#FF8C00;text-transform:uppercase;
                          margin-bottom:12px;font-weight:600;-webkit-font-smoothing:antialiased;">Get In Touch</div>
              <h2 style="font-family:Charter,Georgia,serif;font-size:1.9rem;
                         font-weight:400;color:#fff;margin-bottom:6px;">Contact Us</h2>
              <p style="font-family:Charter,Georgia,serif;font-size:.88rem;
                        color:#555;margin-bottom:28px;">
                Reach out with questions, partnership inquiries, or case-specific requests.
              </p>
            </div>
            """)

            with gr.Column():
                gr.HTML('<div style="max-width:600px;margin:0 auto;padding:0 28px 48px;">')
                gr.HTML('<div class="contact-card">')

                ct_n = gr.Textbox(label="Name",         placeholder="Your full name")
                gr.HTML('<div style="height:12px;"></div>')
                ct_e = gr.Textbox(label="Email",        placeholder="your@email.com")
                gr.HTML('<div style="height:12px;"></div>')
                ct_p = gr.Textbox(label="Phone Number", placeholder="+91 XXXXXXXXXX")
                gr.HTML('<div style="height:12px;"></div>')
                ct_m = gr.Textbox(label="Message",      placeholder="Your message…", lines=5)
                gr.HTML('<div style="height:18px;"></div>')

                gr.HTML('<div style="display:flex;justify-content:flex-start;">')
                ct_b = gr.Button("Send Message", elem_classes="gr-button",
                                  elem_id="send-btn")
                gr.HTML('</div>')

                ct_r = gr.HTML("")

                gr.HTML("""
                <div class="contact-info-block">
                  <div style="font-family:'Roboto Mono',monospace;font-size:.6rem;
                              letter-spacing:.14em;text-transform:uppercase;
                              color:#FF8C00;margin-bottom:14px;font-weight:600;
                              -webkit-font-smoothing:antialiased;">Direct Contact</div>
                  <div style="display:flex;flex-direction:column;gap:12px;">
                    <div style="display:flex;align-items:center;gap:12px;">
                      <svg width="14" height="14" viewBox="0 0 24 24" fill="none"
                           stroke="#FF8C00" stroke-width="1.5">
                        <path d="M4 4h16c1.1 0 2 .9 2 2v12c0 1.1-.9 2-2 2H4c-1.1 0-2-.9-2-2V6c0-1.1.9-2 2-2z"/>
                        <polyline points="22,6 12,13 2,6"/>
                      </svg>
                      <a href="mailto:anumulasamarth008@gmail.com"
                         style="font-family:Charter,Georgia,serif;font-size:.86rem;
                                color:#bbb;text-decoration:none;"
                         onmouseover="this.style.color='#FF8C00'"
                         onmouseout="this.style.color='#bbb'">
                        anumulasamarth008@gmail.com
                      </a>
                    </div>
                    <div style="display:flex;align-items:center;gap:12px;">
                      <svg width="14" height="14" viewBox="0 0 24 24" fill="none"
                           stroke="#FF8C00" stroke-width="1.5">
                        <path d="M22 16.92v3a2 2 0 01-2.18 2 19.79 19.79 0 01-8.63-3.07A19.5 19.5 0 013.07 9.8 19.79 19.79 0 01.1 1.18 2 2 0 012.08 0h3a2 2 0 012 1.72c.127.96.361 1.903.7 2.81a2 2 0 01-.45 2.11L6.09 7.91a16 16 0 006 6l1.27-1.27a2 2 0 012.11-.45c.907.339 1.85.573 2.81.7A2 2 0 0122 14.92z"/>
                      </svg>
                      <span style="font-family:Charter,Georgia,serif;font-size:.86rem;color:#bbb;">
                        +91 837982517
                      </span>
                    </div>
                  </div>
                </div>
                """)

                gr.HTML('</div>')  # close contact-card
                gr.HTML('</div>')  # close wrapper

            gr.HTML("""
            <style>
            #send-btn, #send-btn button {
              width: auto !important;
              max-width: 200px !important;
              min-height: 44px !important;
              height: 44px !important;
              padding: 0 28px !important;
              font-size: .72rem !important;
              letter-spacing: .12em !important;
            }
            </style>""")

            def handle_contact(name, email, phone, msg):
                if not name.strip() or not email.strip() or not msg.strip():
                    return ("<div style='font-family:\"Roboto Mono\",monospace;font-size:.72rem;"
                            "color:#FF4500;padding:10px 0;font-weight:600;"
                            "-webkit-font-smoothing:antialiased;'>"
                            "Name, email, and message are required.</div>")
                if not re.match(r"^[^@]+@[^@]+\.[^@]+$", email.strip()):
                    return ("<div style='font-family:\"Roboto Mono\",monospace;font-size:.72rem;"
                            "color:#FF4500;padding:10px 0;font-weight:600;"
                            "-webkit-font-smoothing:antialiased;'>"
                            "Please enter a valid email address.</div>")
                _send_contact_email(name.strip(), email.strip(), phone.strip(), msg.strip())
                return ("<div style='font-family:\"Roboto Mono\",monospace;font-size:.72rem;"
                        "color:#4CAF50;padding:10px 0;letter-spacing:.06em;font-weight:600;"
                        "-webkit-font-smoothing:antialiased;'>"
                        "&#10003; Message received. We will respond at the provided email.</div>")

            ct_b.click(handle_contact, [ct_n, ct_e, ct_p, ct_m], ct_r)

            gr.HTML(_FOOTER)   # FIX-5: disclaimer footer on Contact page

# ─────────────────────────────────────────────────────────────────────────────
# LAUNCH
# ─────────────────────────────────────────────────────────────────────────────
demo.launch(share=True)

/tmp/ipykernel_704/463674231.py:988: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="CrimeSceneSolver") as demo:
/usr/local/lib/python3.12/dist-packages/gradio/component_meta.py:189: DeprecationWarning: The default value of 'padding' in gr.HTML will be changed from True to False in Gradio 6.0. You will need to explicitly set padding=True if you want to keep the padding.
  return fn(self, **kwargs)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and sche

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26bf50f7fa8fba3a44.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
